# Sensor Match Dominates Pretraining Source in Geospatial Foundation Model Transfer

**Companion notebook.** 17 configurations x 6 label budgets x 5 seeds = 510 runs, on EuroSAT and NWPU-RESISC45.

Open in Colab, pick a **GPU** runtime, and run the cells top to bottom. **Cell 1 is the only one you need to edit.**

### What the study asks

Published comparisons of geospatial foundation models bundle two things together: pretraining on satellite imagery, and feeding the model extra spectral bands. When one arm wins, you cannot tell which factor did the work. This notebook separates them, and adds a third question the literature mostly skips — whether the pretraining *sensor* has to match the target sensor.

### The rule that governs everything

Every configuration gets the same protocol. No per-model or per-backbone tuning, nothing chosen on the test split, no augmentation in the main results. Any difference in the results is then attributable to the factors under study rather than to how much attention each model received.

### Order of operations

`config` -> `setup` -> write `src/*.py` -> stage the data -> **test suite (stops on failure)** -> GPU preflight -> smoke test -> **pilot, which pauses for your review** -> full grid -> analysis.

Everything durable is written to `ARTIFACT_ROOT` on Drive, so a dropped session costs at most one run.

## 1 — Config

The only cell you should edit. Takes a few seconds.

In [ ]:
# --- CELL 1 - CONFIG ---

# --- Where things live -----------------------------------------------
ARTIFACT_ROOT = "/content/drive/MyDrive/fm_transfer_study"   # durable, on Drive
SCRATCH_ROOT  = "/content/fmts"                              # temporary, on the VM

# --- Where the data comes from ---------------------------------------
EUROSAT_SOURCE  = "kaggle"
RESISC45_SOURCE = "drive"

KAGGLE_EUROSAT_SLUG  = "apollo2506/eurosat-dataset"
KAGGLE_RESISC45_SLUG = ""

EUROSAT_ARCHIVE  = "/content/drive/MyDrive/datasets/eurosat-dataset.zip"
RESISC45_ARCHIVE = "/content/drive/MyDrive/datasets/NWPU-RESISC45.zip"   # ← CHECK

KAGGLE_JSON_ON_DRIVE = "/content/drive/MyDrive/kaggle.json"              # ← CHECK
KAGGLE_USERNAME = ""
KAGGLE_KEY      = ""

# --- What to run  (DEBUG SETTINGS - see Step 7 to switch) ------------
RUN_DATASETS  = ["eurosat", "resisc45"]                                          # ← CHECK
TIERS         = ["core", "factorial", "matched"]
MAX_HOURS     = 15.0                                      # ← CHECK
PILOT_BUDGETS = [5,10]
PILOT_SEEDS   = [42,123]

# --- Protocol. DO NOT EDIT ANY LINE BELOW THIS COMMENT ---------------
MIN_STEPS_STAGE1 = 300
MIN_STEPS_STAGE2 = 200
USE_AMP          = True
WARMUP_FRAC      = 0.10
GRAD_CLIP_NORM   = 1.0
LLRD_GAMMA       = 0.75
HEAD_LR_MULT     = 10.0
CHANNELS_LAST    = True
USE_EMA          = False
EMA_DECAY        = 0.999
FREEZE_BN_STAGE2 = False
GRAD_CHECKPOINTING = False      # preflight will tell you if this must become True
SWIN_ARCH = "swin_v2_b"

# --- Practical -------------------------------------------------------
NUM_WORKERS        = 2
CACHE_RESISC45_NPY = True
SAVE_CURVES        = True
VERBOSE_EPOCHS     = True
print("Config loaded. ARTIFACT_ROOT =", ARTIFACT_ROOT)

Config loaded. ARTIFACT_ROOT = /content/drive/MyDrive/fm_transfer_study


In [ ]:
# --- CELL 1b - CONFIG CHECK.  Changes nothing. Run after every Cell 1 edit. ---
import os, json, shutil, zipfile

FAIL, WARN, GOOD = [], [], []
def bad(m, fix):  FAIL.append((m, fix))
def warn(m, fix): WARN.append((m, fix))
def good(m):      GOOD.append(m)

# --- Drive -----------------------------------------------------------
if not os.path.isdir("/content/drive/MyDrive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass
if os.path.isdir("/content/drive/MyDrive"):
    good("Google Drive is mounted")
else:
    bad("Google Drive is not mounted",
        "Run the Setup cell (Cell 2) first, or re-run this cell and approve the popup.")

# --- Paths -----------------------------------------------------------
if ARTIFACT_ROOT.startswith("/content/drive/"):
    good("ARTIFACT_ROOT is on Drive (results will survive a disconnect)")
else:
    bad("ARTIFACT_ROOT is NOT on Drive - everything would be lost on disconnect",
        'Set ARTIFACT_ROOT = "/content/drive/MyDrive/fm_transfer_study"')

if SCRATCH_ROOT.startswith("/content/drive/"):
    bad("SCRATCH_ROOT is on Drive - training would be pathologically slow",
        'Set SCRATCH_ROOT = "/content/fmts"')
else:
    good("SCRATCH_ROOT is on local disk (fast)")

try:
    os.makedirs(ARTIFACT_ROOT, exist_ok=True)
    _t = os.path.join(ARTIFACT_ROOT, ".writetest")
    open(_t, "w").write("ok"); os.remove(_t)
    good("ARTIFACT_ROOT is writable")
except Exception as e:
    bad(f"Cannot write to ARTIFACT_ROOT: {e}",
        "Check the path spelling and that Drive is mounted.")

# --- Runtime ---------------------------------------------------------
try:
    import torch
    if torch.cuda.is_available():
        good(f"GPU available: {torch.cuda.get_device_name(0)}")
    else:
        bad("No GPU - the runtime is on CPU",
            "Runtime -> Change runtime type -> Hardware accelerator -> GPU, then re-run.")
except Exception as e:
    bad(f"Could not query torch: {e}", "Run the Setup cell first.")

free_gb = shutil.disk_usage("/content").free / 1e9
if free_gb > 40:
    good(f"Local free disk: {free_gb:.0f} GB")
else:
    warn(f"Only {free_gb:.0f} GB free on /content",
         "Caches need roughly 20-40 GB. Set CACHE_RESISC45_NPY = False if it runs out.")

# --- Names -----------------------------------------------------------
for d in RUN_DATASETS:
    if d not in ("eurosat", "resisc45"):
        bad(f"Unknown dataset name: {d!r}", 'Use only "eurosat" and/or "resisc45".')
for t in TIERS:
    if t not in ("core", "factorial", "matched", "aug"):
        bad(f"Unknown tier name: {t!r}", 'Use "core", "factorial", "matched".')
if "aug" in TIERS:
    bad("The augmentation tier is in TIERS with the main tiers",
        'Remove "aug". It must be run separately via Optional cell B, never mixed in.')
if not FAIL:
    good(f"Will run datasets {RUN_DATASETS}, tiers {TIERS}")

# --- Data sources ----------------------------------------------------
def _check_archive(path, label):
    if not os.path.exists(path):
        bad(f"{label} archive not found: {path}",
            "Upload it to that exact path in Drive, or fix the path in Cell 1.")
    elif not zipfile.is_zipfile(path):
        bad(f"{label} archive is not a valid .zip: {path}",
            "It may be a .rar renamed to .zip. Re-zip it properly.")
    else:
        good(f"{label} archive found and is a valid zip")

need_kaggle = False
for ds, src, arch, slug in (
    ("eurosat",  EUROSAT_SOURCE,  EUROSAT_ARCHIVE,  KAGGLE_EUROSAT_SLUG),
    ("resisc45", RESISC45_SOURCE, RESISC45_ARCHIVE, KAGGLE_RESISC45_SLUG),
):
    if ds not in RUN_DATASETS:
        continue
    if src == "drive":
        _check_archive(arch, ds)
    elif src == "kaggle":
        need_kaggle = True
        if not slug:
            bad(f"{ds} source is 'kaggle' but its slug is empty",
                f'Either set the slug, or set {ds.upper()}_SOURCE = "drive".')
        else:
            good(f"{ds} will download from Kaggle: {slug}")
    else:
        bad(f"{ds} source is {src!r}", 'Use "kaggle" or "drive".')

if "resisc45" in RUN_DATASETS and RESISC45_SOURCE == "kaggle":
    warn("RESISC45 from Kaggle: many mirrors are incomplete subsets",
         "The loader asserts 45 classes x 700 images. If it fails, the mirror is a subset.")

# --- Kaggle credentials ----------------------------------------------
if need_kaggle:
    if KAGGLE_USERNAME and KAGGLE_KEY:
        good("Kaggle credentials set inline")
    elif os.path.exists(KAGGLE_JSON_ON_DRIVE):
        try:
            k = json.load(open(KAGGLE_JSON_ON_DRIVE))
            if "username" in k and "key" in k:
                good(f"kaggle.json found and valid (user: {k['username']})")
            else:
                bad("kaggle.json is missing 'username' or 'key'",
                    "Re-download it from kaggle.com -> Settings -> API.")
        except Exception:
            bad("kaggle.json is not valid JSON",
                "Re-download it from kaggle.com -> Settings -> API.")
    else:
        bad(f"kaggle.json not found at {KAGGLE_JSON_ON_DRIVE}",
            "Upload it to the top level of your Drive.")

# --- Frozen protocol -------------------------------------------------
FROZEN = {"MIN_STEPS_STAGE1":300, "MIN_STEPS_STAGE2":200, "USE_AMP":True,
          "WARMUP_FRAC":0.10, "GRAD_CLIP_NORM":1.0, "LLRD_GAMMA":0.75,
          "HEAD_LR_MULT":10.0, "CHANNELS_LAST":True, "USE_EMA":False,
          "EMA_DECAY":0.999, "FREEZE_BN_STAGE2":False, "SWIN_ARCH":"swin_v2_b"}
drift = {k:(globals()[k], v) for k,v in FROZEN.items() if globals().get(k) != v}
if drift:
    for k,(got,exp) in drift.items():
        bad(f"Protocol value changed: {k} = {got!r}, expected {exp!r}",
            f"Set {k} = {exp!r}. Changing it invalidates every completed run.")
else:
    good("All 12 frozen protocol values are correct")

# --- Reporting settings ----------------------------------------------
if not SAVE_CURVES:
    bad("SAVE_CURVES is False - Figure 7 of the paper would have no source",
        "Set SAVE_CURVES = True.")
else:
    good("SAVE_CURVES is on (Figure 7 will have data)")

if not (0.25 <= MAX_HOURS <= 24):
    warn(f"MAX_HOURS = {MAX_HOURS}", "Typical values are 1 while debugging, 11 for real runs.")

# --- Mode ------------------------------------------------------------
debug_mode = (len(RUN_DATASETS) == 1 and TIERS == ["core"]
              and len(PILOT_BUDGETS) == 1 and len(PILOT_SEEDS) == 1)

# --- Report ----------------------------------------------------------
print("=" * 68)
for m in GOOD: print("  OK    ", m)
for m, f in WARN: print("  NOTE  ", m, "\n          ->", f)
for m, f in FAIL: print("  ERROR ", m, "\n          ->", f)
print("=" * 68)
if FAIL:
    print(f"\n{len(FAIL)} problem(s). Fix them in Cell 1, re-run this cell.")
    print("DO NOT run any further cells until this says READY.\n")
else:
    print("\nREADY." , "Mode: DEBUG (small, fast)." if debug_mode
          else "Mode: FULL RUN.")
    print("Next: run Cell 2 (Setup), then continue in order.\n")

Mounted at /content/drive
  OK     Google Drive is mounted
  OK     ARTIFACT_ROOT is on Drive (results will survive a disconnect)
  OK     SCRATCH_ROOT is on local disk (fast)
  OK     ARTIFACT_ROOT is writable
  OK     GPU available: NVIDIA A100-SXM4-80GB
  OK     Local free disk: 203 GB
  OK     Will run datasets ['eurosat', 'resisc45'], tiers ['core', 'factorial', 'matched']
  OK     eurosat will download from Kaggle: apollo2506/eurosat-dataset
  OK     resisc45 archive found and is a valid zip
  OK     kaggle.json found and valid (user: aaintel)
  OK     All 12 frozen protocol values are correct
  OK     SAVE_CURVES is on (Figure 7 will have data)

READY. Mode: FULL RUN.
Next: run Cell 2 (Setup), then continue in order.



In [ ]:
print(EUROSAT_SOURCE, RUN_DATASETS, TIERS)

drive ['eurosat', 'resisc45'] ['core', 'factorial', 'matched']


## 2 — Setup

Mounts Drive, installs the few packages Colab lacks, and prints a version report. One to two minutes.

**Torch and torchvision are deliberately left alone.** Colab ships its own build against its own CUDA, and pip-installing over it is the usual way to break a session. The cell checks afterwards that the torch version is unchanged.

In [ ]:
import os, sys, subprocess, json

# ---- mount Drive ----
try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Not running in Colab - Drive not mounted.")

import torch
_torch_before = (torch.__version__, torch.version.cuda)

# ---- install ONLY what Colab lacks ----
# --no-deps on satlaspretrain_models: its dependency list includes torch, and
# resolving it can drag in a different torch build and break CUDA.
def _pip(args):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install"] + args,
                   check=False)

def _have(mod):
    try:
        __import__(mod); return True
    except ImportError:
        return False

if not _have("satlaspretrain_models"):
    _pip(["--no-deps", "satlaspretrain_models"])
missing = [m for m, p in (("rasterio", "rasterio"),
                          ("statsmodels", "statsmodels"),
                          ("tabulate", "tabulate"),
                          ("kagglehub", "kagglehub")) if not _have(m)]
if missing:
    _pip(missing)

import importlib
importlib.invalidate_caches()
import torch
assert (torch.__version__, torch.version.cuda) == _torch_before, (
    "torch was replaced by a pip install - restart the runtime and re-run "
    "without installing torch/torchvision")

os.makedirs("src", exist_ok=True)
if "src" not in sys.path:
    sys.path.insert(0, os.path.abspath("src"))

# ---- Kaggle credentials ----
if KAGGLE_USERNAME and KAGGLE_KEY:
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    os.environ["KAGGLE_KEY"] = KAGGLE_KEY
elif os.path.exists(KAGGLE_JSON_ON_DRIVE):
    with open(KAGGLE_JSON_ON_DRIVE) as f:
        _kj = json.load(f)
    os.environ["KAGGLE_USERNAME"] = _kj.get("username", "")
    os.environ["KAGGLE_KEY"] = _kj.get("key", "")
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    import shutil as _sh
    _sh.copy(KAGGLE_JSON_ON_DRIVE, os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

# ---- version report ----
import platform, torchvision
rep = {"python": platform.python_version(), "torch": torch.__version__,
       "torchvision": torchvision.__version__,
       "cuda_runtime": torch.version.cuda,
       "cuda_available": torch.cuda.is_available(),
       "cudnn": torch.backends.cudnn.version()}
if torch.cuda.is_available():
    rep["gpu"] = torch.cuda.get_device_name(0)
    rep["vram_gb"] = round(torch.cuda.mem_get_info()[1] / 1e9, 1)
    try:
        rep["bf16"] = torch.cuda.is_bf16_supported()
    except Exception:
        rep["bf16"] = False
for m in ("numpy", "scipy", "sklearn", "statsmodels", "pandas", "rasterio",
          "satlaspretrain_models", "PIL", "matplotlib"):
    try:
        rep[m] = getattr(__import__(m), "__version__", "installed")
    except Exception:
        rep[m] = "MISSING"
w = max(len(k) for k in rep)
print("=" * 58); print("ENVIRONMENT"); print("=" * 58)
for k, v in rep.items():
    print(f"  {k.ljust(w)} : {v}")
print("=" * 58)
print("\nNOTE: these are the versions this run actually used.")
print("these versions when reporting the software environment.")
if not torch.cuda.is_available():
    print("\n*** NO GPU. Runtime > Change runtime type > GPU, then re-run. ***")

ENVIRONMENT
  python                : 3.12.13
  torch                 : 2.11.0+cu128
  torchvision           : 0.26.0+cu128
  cuda_runtime          : 12.8
  cuda_available        : True
  cudnn                 : 91900
  gpu                   : NVIDIA A100-SXM4-80GB
  vram_gb               : 85.1
  bf16                  : True
  numpy                 : 2.0.2
  scipy                 : 1.16.3
  sklearn               : 1.6.1
  statsmodels           : 0.14.6
  pandas                : 2.2.3
  rasterio              : 1.5.1
  satlaspretrain_models : installed
  PIL                   : 11.3.0
  matplotlib            : 3.10.0

NOTE: Sec. 3.10's software-version list must be updated to exactly
these versions before submission. See METHODOLOGY_DELTA.md item 9.


## 3 — Write the source modules

Writes `src/*.py`. The study's code lives in real modules that get imported rather than in notebook cells, so it can be tested. Instant.

In [ ]:
%%writefile src/protocol.py
"""The frozen experimental protocol.

This module is the single place where the training protocol is defined. Every
one of the 17 configurations reads the SAME object from here. Nothing in this
file may be made a function of backbone, pretraining source, channel count,
dataset or GPU. The only quantity permitted to vary is the epoch count, and it
varies through the stated formula in `plan_stage` -- a uniform rule applied
identically everywhere, not a per-configuration choice.

Design-freeze invariants asserted at import and re-asserted in the test suite:

  * batch_size is 32 in every code path, on every GPU.
  * label_smoothing is 0.0. It is not a tunable here: smoothing directly
    manipulates expected calibration error, which is a *measured outcome* of
    this study.
  * The optimisation-step floor uses one (min_steps, base_epochs) pair per
    stage for the whole grid.

NOTE: epochs are floors, not fixed counts.
This protocol keeps those as *floors on epochs* and adds a floor on optimiser
steps.
"""
import math
from dataclasses import dataclass, asdict

# Frozen. Never a function of which GPU was assigned (Colab hands out T4/L4/
# A100/V100 unpredictably). Memory pressure is handled with gradient
# checkpointing, which is mathematically identical, never with a smaller batch
# and never with gradient accumulation -- accumulation would change ResNet-50's
# BatchNorm statistics while leaving Swin's LayerNorm untouched, altering one
# backbone and not the other and breaking the architecture control.
BATCH_SIZE = 32

@dataclass(frozen=True)
class Protocol:
    # ---- the protocol table, unchanged ----
    batch_size: int = BATCH_SIZE
    stage1_epochs: int = 15          # now a FLOOR on epochs, see plan_stage
    stage2_epochs: int = 10          # now a FLOOR on epochs, see plan_stage
    stage1_lr: float = 1e-3
    stage2_lr: float = 1e-4
    stage1_wd: float = 1e-4
    stage2_wd: float = 1e-5
    img_size: int = 224
    head_hidden: int = 512
    head_dropout: float = 0.3

    # ---- 2.1 optimisation-step floor ----
    min_steps_stage1: int = 300
    min_steps_stage2: int = 200

    # ---- 2.3 uniform optimisation upgrades (frozen default set) ----
    use_amp: bool = True             # bf16 where supported, else fp16, else off
    warmup_frac: float = 0.10        # linear warmup over first ~10% of steps
    grad_clip_norm: float = 1.0      # global-norm clipping
    llrd_gamma: float = 0.75         # layer-wise lr decay, Stage 2
    llrd_n_stages: int = 5           # SAME stage count for both backbones
    head_lr_mult: float = 10.0       # discriminative head lr, Stage 2
    channels_last: bool = True       # conv backbone only; pure throughput
    use_ema: bool = False            # opt-in flag
    ema_decay: float = 0.999
    freeze_bn_stage2: bool = False   # opt-in flag, all configs or none
    grad_checkpointing: bool = False # decided once grid-wide at preflight

    # ---- forbidden, pinned ----
    label_smoothing: float = 0.0     # off: plain cross-entropy everywhere
    use_tta: bool = False            # off: no test-time augmentation
    torch_compile: bool = False      # off: keeps runs bit-reproducible

    def __post_init__(self):
        assert self.batch_size == 32, "batch size is frozen at 32"
        assert self.label_smoothing == 0.0, (
            "label smoothing is forbidden: it manipulates ECE, a measured "
            "outcome of this study")
        assert self.use_tta is False, "test-time augmentation is forbidden"
        assert self.torch_compile is False, (
            "torch.compile is forbidden: it can break the bitwise determinism "
            "guaranteed by the paper")
        assert 0.0 <= self.warmup_frac < 0.5
        assert 0.0 < self.llrd_gamma <= 1.0
        assert self.llrd_n_stages >= 1

    def as_dict(self):
        return asdict(self)

    def fingerprint(self):
        """Short stable string identifying this protocol, logged on every row."""
        import hashlib
        blob = repr(sorted(self.as_dict().items())).encode()
        return hashlib.sha1(blob).hexdigest()[:12]

PROTOCOL = Protocol()

# --------
# The optimisation-step floor
# --------
def drop_last_for(n_train: int, batch_size: int = BATCH_SIZE) -> bool:
    """Uniform rule, stated as a formula, applied identically everywhere.

    A trailing batch of size 1 makes ResNet-50's BatchNorm raise in train()
    mode ("Expected more than 1 value per channel"). RESISC45 at k=5 is exactly
    this case: 45 classes x 5 = 225 = 7*32 + 1. Swin (LayerNorm) would survive
    it, so a rule that fired only for ResNet would break the architecture
    control. Dropping the singleton for every configuration keeps the rule
    uniform; it discards at most one training image, and only when
    n_train % batch_size == 1.
    """
    return (n_train % batch_size) == 1 and n_train > batch_size

def steps_per_epoch(n_train: int, batch_size: int = BATCH_SIZE) -> int:
    """Actual optimiser steps in one epoch, honouring the drop_last rule."""
    if n_train <= 0:
        return 0
    if drop_last_for(n_train, batch_size):
        return n_train // batch_size
    return math.ceil(n_train / batch_size)

def epochs_for(n_train: int, base_epochs: int, min_steps: int,
               batch_size: int = BATCH_SIZE) -> int:
    """epochs = max(base_epochs, ceil(min_steps / steps_per_epoch)).

    The protocol fixes epochs, not steps. At k=5 on EuroSAT that is 50 images
    -> 2 batches/epoch -> 30 Stage-1 steps and 20 Stage-2 steps, against ~9,000
    at full data. The low-budget end would then measure undertraining rather
    than data limitation, and the study's headline question is precisely about
    the low-budget end. This floor removes that artefact. It is one formula for
    all 17 configurations, both stages, both datasets.
    """
    spe = steps_per_epoch(n_train, batch_size)
    if spe == 0:
        return 0
    return max(int(base_epochs), math.ceil(min_steps / spe))

def plan_stage(n_train: int, stage: int, proto: Protocol = PROTOCOL) -> dict:
    """Everything the training loop needs to schedule one stage."""
    assert stage in (1, 2)
    base = proto.stage1_epochs if stage == 1 else proto.stage2_epochs
    floor = proto.min_steps_stage1 if stage == 1 else proto.min_steps_stage2
    bs = proto.batch_size
    spe = steps_per_epoch(n_train, bs)
    ep = epochs_for(n_train, base, floor, bs)
    total = spe * ep
    return {
        "n_train": int(n_train),
        "batch_size": bs,
        "drop_last": drop_last_for(n_train, bs),
        "steps_per_epoch": spe,
        "epochs": ep,
        "total_steps": total,
        "warmup_steps": int(round(total * proto.warmup_frac)),
        "base_epochs": int(base),
        "min_steps": int(floor),
        "floor_engaged": ep > base,
        "val_every": val_every(ep, base),
        "val_epochs": validation_epochs(ep, base),
    }

def val_every(epochs: int, base_epochs: int) -> int:
    """Validate every `val_every` epochs, so the NUMBER of model-selection
    opportunities stays at the manuscript's 15 (Stage 1) / 10 (Stage 2)
    regardless of how far the step floor stretches the epoch count.

    Two reasons, both uniform across all 17 configurations:

      * Selection procedure. the protocol table selects among 15 and 10 epoch-end
        checkpoints. Letting the floor turn that into 150 selection points at
        k=5 would give the low-budget arms far more opportunity to fit the
        validation split than the full-data arms get -- a budget-dependent
        change in the selection procedure, which is exactly what the design
        freeze exists to prevent.
      * Cost. The validation split is never subsampled (see the paper), so at
        k=5 on RESISC45 a per-epoch validation would run 4,725 forward passes
        29 times over, dwarfing the 6.5k training passes it is supervising.

    The final epoch is always a validation point, so a best checkpoint always
    exists.

    ceil, not round: with round, 19 epochs against a base of 15 gives an
    interval of 1 and therefore 19 selection points, while a neighbouring
    budget gets 10 -- reintroducing exactly the budget-dependent variation this
    rule exists to remove. ceil guarantees the count never exceeds the base.
    """
    return max(1, int(math.ceil(epochs / float(max(1, base_epochs)))))

def validation_epochs(epochs: int, base_epochs: int) -> set:
    step = val_every(epochs, base_epochs)
    pts = set(range(step - 1, epochs, step))
    pts.add(epochs - 1)          # the last epoch is always evaluated
    return {p for p in pts if 0 <= p < epochs}

def lr_scale(step: int, total_steps: int, warmup_steps: int) -> float:
    """Linear warmup over the first ~10% of steps, then cosine to zero.

    Scheduled per optimiser step over the ACTUAL total step count, which is
    what "set cosine T_max to the actual epoch count" is asking for: with the
    step floor engaged the epoch count is no longer 15/10, so a T_max left at
    15/10 would restart the cosine many times over. Identical for every
    configuration.
    """
    if total_steps <= 0:
        return 0.0
    step = min(step, total_steps)
    if warmup_steps > 0 and step < warmup_steps:
        return (step + 1) / float(warmup_steps)
    denom = max(1, total_steps - warmup_steps)
    prog = (step - warmup_steps) / denom
    prog = min(max(prog, 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * prog))

Overwriting src/protocol.py


In [ ]:
%%writefile src/configs.py
"""The 17-configuration grid (Tables I and II), plus the separately flagged
augmentation tier.

A configuration is (backbone, pretraining, channel_mode, satlas_variant, tier,
augment). `augment` defaults to False and is False for every configuration in
the three main tiers -- augmentation must not enter the main results, because
its benefit scales inversely with training-set size and would therefore
confound the label-budget axis directly (see the paper).

Main grid:   core 360 + factorial 120 + matched 30 = 510 runs.
Aug tier:    a mirror of the core tier only, 360 runs, augment=True, so every
             augmented result has a directly paired unaugmented twin.
"""
from typing import NamedTuple, List, Optional

class Cfg(NamedTuple):
    backbone: str            # resnet50 | swin_b
    pretraining: str         # satlas | imagenet | scratch
    channel_mode: str        # ms9 | rgb3
    satlas_variant: Optional[str]   # ms | rgb | aerial | None
    tier: str                # core | factorial | matched | aug
    augment: bool = False

# -------- EuroSAT --
# Table I. Ten configurations: the 2x2 factorial {satlas, imagenet} x
# {9-band, 3-band}, plus the scratch floor at 9 bands.
EUROSAT_CONFIGS: List[Cfg] = [
    # --- CORE ---
    Cfg("resnet50", "satlas",   "ms9",  "ms",  "core"),
    Cfg("swin_b",   "satlas",   "ms9",  "ms",  "core"),
    Cfg("resnet50", "imagenet", "rgb3", None,  "core"),
    Cfg("swin_b",   "imagenet", "rgb3", None,  "core"),
    Cfg("resnet50", "scratch",  "ms9",  None,  "core"),
    Cfg("swin_b",   "scratch",  "ms9",  None,  "core"),
    # --- FACTORIAL: completes {satlas, imagenet} x {9-band, 3-band} ---
    Cfg("resnet50", "satlas",   "rgb3", "rgb", "factorial"),
    Cfg("swin_b",   "satlas",   "rgb3", "rgb", "factorial"),
    Cfg("resnet50", "imagenet", "ms9",  None,  "factorial"),
    Cfg("swin_b",   "imagenet", "ms9",  None,  "factorial"),
]

# -------- RESISC45 --
# Table II. Seven configurations; all three-channel, so this dataset isolates
# pretraining source with no channel-count variation whatsoever.
RESISC45_CONFIGS: List[Cfg] = [
    # --- CORE ---
    Cfg("resnet50", "satlas",   "rgb3", "rgb",    "core"),
    Cfg("swin_b",   "satlas",   "rgb3", "rgb",    "core"),
    Cfg("resnet50", "imagenet", "rgb3", None,     "core"),
    Cfg("swin_b",   "imagenet", "rgb3", None,     "core"),
    Cfg("resnet50", "scratch",  "rgb3", None,     "core"),
    Cfg("swin_b",   "scratch",  "rgb3", None,     "core"),
    # --- MATCHED: aerial-pretrained Satlas, modality-matched to RESISC45.
    # Swin-only because SatlasPretrain publishes no aerial ResNet-50 (Sec.
    # 3.3.1); a property of the released weights, not a design choice.
    Cfg("swin_b",   "satlas",   "rgb3", "aerial", "matched"),
]

_BASE = {"eurosat": EUROSAT_CONFIGS, "resisc45": RESISC45_CONFIGS}

# The augmentation tier mirrors CORE only, so results are directly paired.
AUG_CONFIGS = {
    ds: [c._replace(tier="aug", augment=True) for c in cfgs if c.tier == "core"]
    for ds, cfgs in _BASE.items()
}

CONFIGS = {ds: _BASE[ds] + AUG_CONFIGS[ds] for ds in _BASE}

# Main-results tiers, in execution order. "aug" is deliberately absent: it is
# opt-in and never mixes with the main results.
TIER_ORDER = ["core", "factorial", "matched"]
ALL_TIERS = TIER_ORDER + ["aug"]

DATASETS = ["eurosat", "resisc45"]
NUM_CLASSES = {"eurosat": 10, "resisc45": 45}
SEEDS = [42, 123, 2024, 7, 999]
K_SHOTS = [5, 10, 25, 50, 100, "full"]

def configs_for(dataset: str, tiers=None) -> List[Cfg]:
    tiers = list(tiers) if tiers else list(TIER_ORDER)
    for t in tiers:
        assert t in ALL_TIERS, f"unknown tier {t!r}"
    return [c for c in CONFIGS[dataset] if c.tier in tiers]

def config_id(backbone, pretraining, channel_mode, satlas_variant,
              augment=False) -> str:
    """Stable id used in result rows, prediction filenames and checkpoints.

    `augment` is part of the id so an augmented run can never overwrite or be
    mistaken for its unaugmented twin.
    """
    src = pretraining if pretraining != "satlas" else f"satlas-{satlas_variant}"
    cid = f"{backbone}_{src}_{channel_mode}"
    return cid + "_aug" if augment else cid

def cfg_id(c: Cfg) -> str:
    return config_id(c.backbone, c.pretraining, c.channel_mode,
                     c.satlas_variant, c.augment)

def grid_size(dataset=None, tiers=None) -> int:
    dss = [dataset] if dataset else DATASETS
    return sum(len(configs_for(d, tiers)) * len(K_SHOTS) * len(SEEDS)
               for d in dss)

Overwriting src/configs.py


In [ ]:
%%writefile src/utils.py
"""Seeding, device, hashing and small shared helpers.

the paper guarantees that a run is reproducible bit-for-bit on identical
hardware. `set_seed` seeds Python, NumPy, torch and CUDA, puts cuDNN in
deterministic mode and disables autotuning -- exactly the guarantee the
paper makes, and no more (on
torch.use_deterministic_algorithms). `loader_generator` and `worker_init_fn`
close the remaining hole: without them the shuffle order depends on how much
global torch RNG the model build happened to consume.
"""
import os
import random
import hashlib
import platform

import numpy as np
import torch

def set_seed(seed: int) -> None:
    """Full determinism for one run. Call before building model + sampling."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # the paper: cuDNN deterministic, autotuning disabled. Kept even though it
    # costs throughput -- reproducibility is a stated guarantee, so the cost is
    # measured and reported rather than traded away.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def loader_generator(seed: int) -> torch.Generator:
    """Explicit generator for shuffling, independent of global RNG state."""
    g = torch.Generator()
    g.manual_seed(int(seed))
    return g

def worker_init_fn(worker_id: int) -> None:
    """Deterministic per-worker seeding."""
    base = torch.initial_seed() % (2 ** 31 - 1)
    s = (base + worker_id) % (2 ** 31 - 1)
    np.random.seed(s)
    random.seed(s)

def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def gpu_name() -> str:
    """Recorded on every result row (see the paper). analyze.py flags any
    comparison whose two arms ran on different GPU types."""
    if torch.cuda.is_available():
        try:
            return torch.cuda.get_device_name(0)
        except Exception:
            return "cuda:unknown"
    return "CPU"

def amp_dtype_for(device: torch.device, enabled: bool):
    """bf16 where supported, else fp16 on CUDA, else disabled.

    Returned dtype is used identically by every configuration.
    """
    if not enabled or device.type != "cuda":
        return None
    try:
        if torch.cuda.is_bf16_supported():
            return torch.bfloat16
    except Exception:
        pass
    return torch.float16

def file_sha256(path: str, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def short_hash(*parts) -> str:
    h = hashlib.sha1()
    for p in parts:
        h.update(str(p).encode())
    return h.hexdigest()[:16]

@torch.no_grad()
def backbone_fingerprint(module: torch.nn.Module, per_tensor: int = 64) -> str:
    """Cheap, collision-resistant fingerprint of a module's parameters.

    Used to key the Stage-1 feature cache. For every pretrained arm the Stage-1
    backbone weights are identical across all 6 budgets and 5 seeds, so 30 runs
    can share one forward pass; for scratch arms the weights depend on the
    seed. Rather than reasoning about which case applies, the cache is keyed on
    what the weights actually are, so a stale or mismatched cache is impossible
    by construction.

    Hashes shapes plus an evenly spaced subsample of each tensor, which is
    microseconds rather than the ~1s a full 350 MB hash would cost per run.
    """
    h = hashlib.sha1()
    for name, p in sorted(module.state_dict().items()):
        h.update(name.encode())
        if not torch.is_tensor(p):
            h.update(repr(p).encode())
            continue
        h.update(str(tuple(p.shape)).encode())
        h.update(str(p.dtype).encode())
        flat = p.detach().reshape(-1).float()
        n = flat.numel()
        if n == 0:
            continue
        step = max(1, n // per_tensor)
        sub = flat[::step][:per_tensor].cpu().numpy()
        h.update(np.ascontiguousarray(sub, dtype=np.float64).tobytes())
    return h.hexdigest()[:16]

def version_report() -> dict:
    """Printed by the setup cell. This is the authoritative record of
    whatever Colab actually provides."""
    import torchvision
    rep = {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(),
        "numpy": np.__version__,
    }
    for mod in ("sklearn", "scipy", "statsmodels", "pandas", "rasterio",
                "satlaspretrain_models", "PIL"):
        try:
            m = __import__(mod)
            rep[mod] = getattr(m, "__version__", "installed")
        except Exception:
            rep[mod] = "MISSING"
    if torch.cuda.is_available():
        rep["gpu"] = gpu_name()
        try:
            free, total = torch.cuda.mem_get_info()
            rep["vram_gb"] = round(total / 1e9, 1)
        except Exception:
            pass
        rep["capability"] = ".".join(str(x) for x in
                                     torch.cuda.get_device_capability(0))
        try:
            rep["bf16"] = bool(torch.cuda.is_bf16_supported())
        except Exception:
            rep["bf16"] = False
    return rep

def print_version_report(rep=None) -> None:
    rep = rep or version_report()
    width = max(len(k) for k in rep)
    print("=" * 58)
    print("ENVIRONMENT")
    print("=" * 58)
    for k, v in rep.items():
        print(f"  {k.ljust(width)} : {v}")
    print("=" * 58)

def fmt_hms(seconds: float) -> str:
    seconds = int(max(0, seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:d}h{m:02d}m{s:02d}s"

Overwriting src/utils.py


In [ ]:
%%writefile src/store.py
"""Durable artefact store, resume logic and the results CSV.

THE BUG THIS FIXES. /content is wiped when a Colab session ends, and the
original code wrote results, predictions, splits and checkpoints to relative
paths. An interrupted session therefore lost everything -- and resume, which
reads completed runs back from the results CSV, could not work because the CSV
was gone with it. the paper claims "an interrupted grid loses at most one run";
that claim was false on Colab.

Layout:
  ARTIFACT_ROOT (Drive, durable)      SCRATCH (/content, transient)
    results/*.csv                       data/            extracted datasets
    preds/*.npz                         cache/           decoded .npy memmaps
    splits/*.csv                        feats/           Stage-1 feature cache
    checkpoints/*.pt
    logs/progress.md
    _partial/*.json                   <- interrupted-run sentinels

Never train from a Drive-mounted path: many small reads over Drive are
pathologically slow. Archives are copied to /content and extracted there.
"""
import os
import csv
import json
import time
import shutil

RESULT_FIELDS = [
    # ---- identity ----
    "run_key", "dataset", "config_id", "backbone", "pretraining",
    "channel_mode", "satlas_variant", "swin_arch", "tier", "augment",
    "k_shot", "seed",
    # ---- Stage 2 (fine-tuned): the headline numbers, all six metrics ----
    "test_acc", "macro_f1", "weighted_f1", "kappa", "roc_auc_ovr", "ece",
    # ---- Stage 1 (frozen backbone): the second results axis, all six ----
    "frozen_acc", "frozen_macro_f1", "frozen_weighted_f1", "frozen_kappa",
    "frozen_roc_auc_ovr", "frozen_ece",
    # ---- realised schedule (the optimisation-step floor) ----
    "epochs_stage1", "epochs_stage2", "steps_stage1", "steps_stage2",
    "steps_per_epoch_s1", "steps_per_epoch_s2",
    "min_steps_stage1", "min_steps_stage2", "floor_engaged_s1",
    "floor_engaged_s2", "drop_last",
    # ---- model selection (validation only, never test) ----
    "best_val_acc_stage1", "best_val_acc_stage2",
    "best_epoch_stage1", "best_epoch_stage2",
    # ---- timing ----
    "train_seconds", "eval_seconds", "stage1_train_seconds",
    "stage2_train_seconds", "total_seconds",
    # ---- data provenance ----
    "n_train_pool", "n_train_used", "n_val", "n_test",
    "split_hash_train", "split_hash_val", "split_hash_test",
    # ---- environment / freeze provenance ----
    "gpu_name", "protocol_fp", "backbone_fp", "feat_cache_hit",
    "amp_dtype", "torch_version", "timestamp",
]

_SUBDIRS = ("results", "preds", "splits", "checkpoints", "logs", "_partial")

class Paths:
    """All durable output goes under ARTIFACT_ROOT; nothing durable elsewhere."""

    def __init__(self, artifact_root: str, scratch: str = "/content/fmts"):
        self.root = os.path.abspath(artifact_root)
        self.scratch = os.path.abspath(scratch)
        for d in _SUBDIRS:
            os.makedirs(os.path.join(self.root, d), exist_ok=True)
        for d in ("data", "cache", "feats", "archives"):
            os.makedirs(os.path.join(self.scratch, d), exist_ok=True)

    # -- durable --
    @property
    def results(self): return os.path.join(self.root, "results")
    @property
    def preds(self): return os.path.join(self.root, "preds")
    @property
    def splits(self): return os.path.join(self.root, "splits")
    @property
    def checkpoints(self): return os.path.join(self.root, "checkpoints")
    @property
    def logs(self): return os.path.join(self.root, "logs")
    @property
    def partial(self): return os.path.join(self.root, "_partial")

    # -- transient --
    @property
    def data(self): return os.path.join(self.scratch, "data")
    @property
    def cache(self): return os.path.join(self.scratch, "cache")
    @property
    def feats(self): return os.path.join(self.scratch, "feats")
    @property
    def archives(self): return os.path.join(self.scratch, "archives")

    def results_csv(self, dataset: str) -> str:
        return os.path.join(self.results, f"{dataset}_results.csv")

    def pred_npz(self, dataset, cid, k, seed, stage) -> str:
        return os.path.join(self.preds,
                            f"{dataset}_{cid}_{k}_{seed}_{stage}.npz")

    def verify_writable(self) -> None:
        """Fail early and loudly if Drive is not actually writable."""
        probe = os.path.join(self.root, ".write_probe")
        try:
            with open(probe, "w") as f:
                f.write("ok")
                f.flush()
                os.fsync(f.fileno())
            with open(probe) as f:
                assert f.read() == "ok"
            os.remove(probe)
        except Exception as e:
            raise RuntimeError(
                f"ARTIFACT_ROOT is not writable: {self.root}\n"
                f"  ({type(e).__name__}: {e})\n"
                "Mount Drive first, and check the path in the Config cell."
            ) from e

    def free_gb(self) -> float:
        try:
            return shutil.disk_usage(self.root).free / 1e9
        except Exception:
            return float("nan")

def run_key(dataset, config_id, k_shot, seed) -> str:
    return f"{dataset}|{config_id}|{k_shot}|{seed}"

# -------- sentinel --
def sentinel_path(paths: Paths, key: str) -> str:
    safe = key.replace("|", "__").replace("/", "_")
    return os.path.join(paths.partial, f"{safe}.json")

def begin_run(paths: Paths, key: str, meta: dict) -> None:
    """Mark a run in progress, so an interruption is redone rather than read
    as finished."""
    p = sentinel_path(paths, key)
    with open(p, "w") as f:
        json.dump({"run_key": key, "started": time.time(), **meta}, f)
        f.flush()
        os.fsync(f.fileno())

def end_run(paths: Paths, key: str) -> None:
    p = sentinel_path(paths, key)
    if os.path.exists(p):
        os.remove(p)

def open_sentinels(paths: Paths) -> set:
    out = set()
    if not os.path.isdir(paths.partial):
        return out
    for fn in os.listdir(paths.partial):
        if fn.endswith(".json"):
            try:
                with open(os.path.join(paths.partial, fn)) as f:
                    out.add(json.load(f)["run_key"])
            except Exception:
                out.add(fn[:-5].replace("__", "|"))
    return out

# -------- results CSV --
def append_result(csv_path: str, row: dict) -> None:
    """Append one run's result and force it to stable storage immediately.

    the paper requires results are never held only in memory. A plain append is
    not enough on Colab: the row sits in the OS page cache (and, on Drive, in
    the FUSE layer) until something flushes it, so a killed session can lose
    rows that were already 'written'. flush + fsync on every row.
    """
    exists = os.path.exists(csv_path) and os.path.getsize(csv_path) > 0
    with open(csv_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=RESULT_FIELDS, extrasaction="ignore")
        if not exists:
            w.writeheader()
        w.writerow({k: row.get(k, "") for k in RESULT_FIELDS})
        f.flush()
        os.fsync(f.fileno())

def read_result_keys(csv_path: str) -> set:
    if not os.path.exists(csv_path):
        return set()
    keys = set()
    with open(csv_path, newline="") as f:
        for r in csv.DictReader(f):
            k = r.get("run_key")
            if not k:
                k = run_key(r.get("dataset"), r.get("config_id"),
                            r.get("k_shot"), r.get("seed"))
            keys.add(k)
    return keys

class Resume:
    """Which runs are already complete, and which were interrupted.

    A run counts as complete only if it has a CSV row AND no open sentinel.
    Because the row is written before the sentinel is cleared, a crash in that
    narrow window causes one run to be repeated; the duplicate row is removed
    on read by `load_results(dedupe=True)`.
    """

    def __init__(self, paths: Paths, datasets=("eurosat", "resisc45")):
        self.paths = paths
        self.done = set()
        for ds in datasets:
            self.done |= read_result_keys(paths.results_csv(ds))
        self.interrupted = open_sentinels(paths)
        self.done -= self.interrupted

    def is_done(self, dataset, cid, k, seed) -> bool:
        return run_key(dataset, cid, k, seed) in self.done

    def mark(self, key: str) -> None:
        self.done.add(key)
        self.interrupted.discard(key)

    def report(self) -> str:
        s = f"{len(self.done)} run(s) already complete"
        if self.interrupted:
            s += (f"; {len(self.interrupted)} interrupted run(s) will be "
                  f"redone")
        return s

def load_results(csv_path: str, dedupe: bool = True):
    """Read a results CSV as a DataFrame, dropping duplicate run_keys."""
    import pandas as pd
    if not os.path.exists(csv_path):
        return None
    df = pd.read_csv(csv_path)
    if not len(df):
        return None
    if dedupe and "run_key" in df.columns:
        df = df.drop_duplicates(subset=["run_key"], keep="last")
    return df.reset_index(drop=True)

def write_text(path: str, text: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
        f.flush()
        os.fsync(f.fileno())

Overwriting src/store.py


In [ ]:
%%writefile src/kshot.py
"""k-shot per-class subsampling (see the paper).

Uniform random within each class, without replacement, from a seed-controlled
random state. Sampling per class rather than globally guarantees that no class
is left unrepresented at the smallest budgets, which would otherwise make
low-budget results a function of sampling luck rather than of the method.

Subsampling is applied to the TRAINING split only. Validation and test remain
at full size at every budget, so every model is selected and evaluated on
identical data regardless of how much supervision it received. `assert_splits_
intact` enforces that and is called on every run.
"""
import numpy as np
from torch.utils.data import Dataset, Subset

class EmptyClassError(AssertionError):
    pass

class SplitSubsampledError(AssertionError):
    pass

def kshot_indices(labels, k: int, seed: int) -> np.ndarray:
    """Indices selecting exactly min(k, n_c) samples per class.

    Deterministic in (labels, k, seed): a fresh RandomState is used, so the
    draw does not depend on how much global RNG the model build consumed.
    """
    rng = np.random.RandomState(seed)
    labels = np.asarray(labels)
    chosen = []
    for cls in np.unique(labels):
        cls_idx = np.where(labels == cls)[0]
        if len(cls_idx) == 0:
            raise EmptyClassError(f"class {cls} has no examples in the pool")
        take = min(int(k), len(cls_idx))
        chosen.extend(rng.choice(cls_idx, size=take, replace=False).tolist())
    out = np.asarray(sorted(chosen), dtype=np.int64)
    assert len(out) == len(set(out.tolist())), "k-shot drew a duplicate index"
    return out

def assert_kshot_valid(labels_all, idx, k, n_classes):
    """Fail loudly: every class present, correct per-class counts."""
    lab = np.asarray(labels_all)[np.asarray(idx)]
    present, counts = np.unique(lab, return_counts=True)
    if len(present) != n_classes:
        missing = sorted(set(range(n_classes)) - set(present.tolist()))
        raise EmptyClassError(
            f"k-shot subset at k={k} is missing class(es) {missing}; "
            "low-budget results would be a function of sampling luck")
    pool = np.asarray(labels_all)
    for c, n in zip(present, counts):
        expect = min(int(k), int((pool == c).sum()))
        if int(n) != expect:
            raise AssertionError(
                f"class {c}: expected {expect} examples at k={k}, got {n}")

def kshot_subset(dataset: Dataset, labels, k, seed: int, n_classes: int):
    """k == 'full' returns the dataset unchanged; otherwise a per-class subset.

    Returns (subset, indices) where indices is None for the full split.
    """
    if k == "full":
        return dataset, None
    idx = kshot_indices(labels, int(k), seed)
    assert_kshot_valid(labels, idx, int(k), n_classes)
    return Subset(dataset, idx.tolist()), idx

def assert_splits_intact(val_ds, test_ds, expect_val: int, expect_test: int):
    """Validation and test must never be subsampled, at any budget."""
    if len(val_ds) != expect_val:
        raise SplitSubsampledError(
            f"validation split has {len(val_ds)} items, expected {expect_val}")
    if len(test_ds) != expect_test:
        raise SplitSubsampledError(
            f"test split has {len(test_ds)} items, expected {expect_test}")

Overwriting src/kshot.py


In [ ]:
%%writefile src/datasets.py
"""Datasets, the the protocol table normalization contract, and GPU-side preprocessing.

WHAT MOVED AND WHY (throughput; results unchanged):

  * EuroSAT decoded arrays are cached as a memory-mapped .npy at NATIVE 64x64
    uint16, in model band order. Reading a 13-band GeoTIFF per item through
    rasterio dominated dataloading cost; the cache is byte-identical to the
    direct path and the test suite asserts it.
  * Resizing to 224x224 happens ON GPU, per batch, instead of per item on CPU.

WHAT DID NOT MOVE, AND MUST NOT. The operation ORDER is part of the contract:

        scale -> clip -> standardize      (at native resolution)
        resize to 224                     (afterwards)

  Standardization is affine and therefore commutes with bilinear interpolation,
  so it may sit on either side of the resize. **Clipping does not commute.**
  clip(DN/4000) applied at 64x64 and then resized is not the same image as a
  resize followed by a clip, so the clip stays at native resolution exactly
  where the protocol table puts it.

the protocol table, implemented literally:

  EuroSAT, 9-band   B4,B3,B2          clip(DN/4000)   Satlas: none
                    B5,B6,B7,B8,B11,B12  clip(DN/8160)   ImageNet/scratch:
                                                       ImageNet mu,sigma on the
                                                       visible three and
                                                       mu=0.449, sigma=0.226 on
                                                       the six non-visible
  EuroSAT, 3-band   B4,B3,B2          clip(DN/4000)   as above (visible only)
  RESISC45          R,G,B             value/255       as above (visible only)
"""
import os
import csv

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# No ImageNet-equivalent statistics exist for red-edge or SWIR bands, and
# estimating them from the target dataset would leak evaluation-domain
# information into preprocessing. The scalar mean of the ImageNet statistics
# places every channel on a comparable scale without inventing values.
NONVIS_MEAN = float(np.mean(IMAGENET_MEAN))     # 0.449
NONVIS_STD = float(np.mean(IMAGENET_STD))       # 0.226

IMG_SIZE = 224
EUROSAT_NATIVE = 64
RESISC_NATIVE = 256

# 0-based band indices inside the 13-band EuroSATallBands GeoTIFF
# (B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,B10,B11,B12) selecting the SatlasPretrain
# Sentinel-2 layout B4,B3,B2,B5,B6,B7,B8,B11,B12. Excluded: B1, B8A, B9, B10.
EUROSAT_SATLAS_BANDS = [3, 2, 1, 4, 5, 6, 7, 11, 12]
N_BANDS_CACHED = len(EUROSAT_SATLAS_BANDS)

# EuroSATallBands ships raw 16-bit L1C DNs and no TCI product, so the TCI/255
# input the Satlas contract specifies is approximated with the standard ESA
# true-colour rendering: a 2.5x reflectance gain == clip(DN/4000, 0, 1). This
# is the single point at which any input deviates from a published
# preprocessing specification (see the paper).
VISIBLE_GAIN = 4000.0
NONVIS_GAIN = 8160.0            # official SatlasPretrain non-TCI contract

EUROSAT_CLASSES = [
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
    "Pasture", "PermanentCrop", "Residential", "River", "SeaLake",
]

# --------
# Datasets: return RAW arrays. All normalization happens in GPUPreprocessor.
# --------
class EuroSATSplit(Dataset):
    """One immutable split of EuroSAT, served from the uint16 memmap cache.

    __getitem__ returns the raw 9 retained bands at native 64x64 as uint16, in
    model band order. Channel selection for the 3-band conditions is a slice of
    the first three (B4,B3,B2), so both conditions are derived from the same
    source product exactly as the paper requires.
    """

    def __init__(self, array_path: str, labels_path: str):
        self.array_path = array_path
        self.labels = np.load(labels_path).astype(np.int64)
        self._arr = None
        arr = self._array()
        assert arr.shape[0] == len(self.labels), (
            f"{array_path}: {arr.shape[0]} images vs {len(self.labels)} labels")
        assert arr.shape[1] == N_BANDS_CACHED, f"expected 9 bands, got {arr.shape[1]}"
        assert arr.shape[2] == arr.shape[3] == EUROSAT_NATIVE, (
            f"EuroSAT patches must be {EUROSAT_NATIVE}x{EUROSAT_NATIVE}, "
            f"got {arr.shape[2]}x{arr.shape[3]}")
        self.shape = arr.shape

    def _array(self):
        # Opened lazily and per worker process; np.load(mmap_mode) is not
        # fork-safe to share across DataLoader workers.
        if self._arr is None:
            self._arr = np.load(self.array_path, mmap_mode="r")
        return self._arr

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = np.asarray(self._array()[idx])           # (9, 64, 64) uint16
        return {"image": torch.from_numpy(x.astype(np.int32)),
                "label": int(self.labels[idx])}

class RESISC45Split(Dataset):
    """One immutable split of RESISC45. Returns raw uint8 (3, 256, 256)."""

    def __init__(self, split_csv: str, data_root: str, array_path=None,
                 labels_path=None):
        self.data_root = data_root
        self.array_path = array_path
        self._arr = None
        if array_path and os.path.exists(array_path):
            self.labels = np.load(labels_path).astype(np.int64)
            self.files = None
        else:
            self.files, labels, self.class_names = [], [], {}
            with open(split_csv, newline="") as fh:
                for r in csv.DictReader(fh):
                    self.files.append(r["filepath"])
                    labels.append(int(r["label"]))
                    self.class_names[int(r["label"])] = r["class"]
            self.labels = np.asarray(labels, dtype=np.int64)

    def _array(self):
        if self._arr is None:
            self._arr = np.load(self.array_path, mmap_mode="r")
        return self._arr

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        if self.array_path:
            x = np.asarray(self._array()[idx])            # (3, 256, 256) uint8
        else:
            from PIL import Image
            p = self.files[idx]
            if not os.path.isabs(p):
                p = os.path.join(self.data_root, p)
            img = Image.open(p).convert("RGB")
            x = np.asarray(img, dtype=np.uint8).transpose(2, 0, 1)
        return {"image": torch.from_numpy(x.astype(np.int32)),
                "label": int(self.labels[idx])}

# --------
# the protocol table, on GPU, per batch
# --------
class GPUPreprocessor:
    """Normalization + optional geometric augmentation + resize, on device.

    One object per (dataset, channel_mode, pretraining) condition. `standardize`
    follows the CHECKPOINT contract, not the dataset: Satlas arms receive none,
    ImageNet and scratch arms receive ImageNet statistics. The normalizations
    differ only by a deterministic affine transform of the same bands, so no
    additional spectral information enters and the 3-vs-9 channel contrast that
    defines the factorial is untouched.
    """

    def __init__(self, dataset: str, channel_mode: str, standardize: bool,
                 device, img_size: int = IMG_SIZE, augment: bool = False,
                 aug_scale=(0.7, 1.0)):
        assert dataset in ("eurosat", "resisc45")
        assert channel_mode in ("ms9", "rgb3")
        if dataset == "resisc45":
            assert channel_mode == "rgb3", "RESISC45 is three-channel only"
        self.dataset = dataset
        self.channel_mode = channel_mode
        self.standardize = standardize
        self.device = device
        self.img_size = img_size
        self.augment = augment
        self.aug_scale = aug_scale
        self.n_ch = 9 if channel_mode == "ms9" else 3

        if dataset == "eurosat":
            gains = [VISIBLE_GAIN] * 3 + [NONVIS_GAIN] * 6
            mean = list(IMAGENET_MEAN) + [NONVIS_MEAN] * 6
            std = list(IMAGENET_STD) + [NONVIS_STD] * 6
        else:
            gains = [255.0] * 3
            mean = list(IMAGENET_MEAN)
            std = list(IMAGENET_STD)
        gains, mean, std = gains[:self.n_ch], mean[:self.n_ch], std[:self.n_ch]

        def col(v):
            return torch.tensor(v, dtype=torch.float32,
                                device=device).view(1, -1, 1, 1)
        self._gain = col(gains)
        self._mean = col(mean)
        self._std = col(std)

    # -- the contract ------------------------------------------------------
    def normalize(self, x_raw: torch.Tensor) -> torch.Tensor:
        """x_raw: (B, C_src, H, W) integer DN / uint8 -> normalized float32.

        Applied at NATIVE resolution, before any resize, because clip does not
        commute with interpolation.
        """
        x = x_raw[:, :self.n_ch].to(self.device, non_blocking=True).float()
        x = torch.clamp(x / self._gain, 0.0, 1.0)
        if self.standardize:
            x = (x - self._mean) / self._std
        return x

    def resize(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[-1] == self.img_size and x.shape[-2] == self.img_size:
            return x
        return F.interpolate(x, size=(self.img_size, self.img_size),
                             mode="bilinear", align_corners=False,
                             antialias=False)

    def __call__(self, x_raw: torch.Tensor, generator=None) -> torch.Tensor:
        x = self.normalize(x_raw)
        if self.augment:
            x = self._augment(x, generator)
        return self.resize(x)

    # -- augmentation tier only (Sec. 2.5) ---------------------------------
    def _augment(self, x: torch.Tensor, generator) -> torch.Tensor:
        """Geometric only, training split only.

        Random resized crop (conservative scale range), horizontal and vertical
        flip, 90-degree rotations. NO colour jitter: photometric jitter is
        defined for RGB and has no meaning on red-edge or SWIR bands, so
        applying it would perturb the multispectral arm differently from the
        RGB arm and silently re-confound the factorial.
        """
        B, C, H, W = x.shape
        lo, hi = self.aug_scale
        r = torch.rand(B, 3, generator=generator).tolist()
        ks = torch.randint(0, 4, (B,), generator=generator).tolist()
        fl = torch.rand(B, 2, generator=generator).tolist()
        out = []
        for i in range(B):
            s = lo + (hi - lo) * r[i][0]
            side = max(8, int(round(H * (s ** 0.5))))
            side = min(side, H)
            top = int(r[i][1] * (H - side)) if H > side else 0
            left = int(r[i][2] * (W - side)) if W > side else 0
            v = x[i:i + 1, :, top:top + side, left:left + side]
            if fl[i][0] < 0.5:
                v = torch.flip(v, dims=[3])          # horizontal
            if fl[i][1] < 0.5:
                v = torch.flip(v, dims=[2])          # vertical
            if ks[i]:
                v = torch.rot90(v, k=int(ks[i]), dims=[2, 3])
            out.append(F.interpolate(v, size=(H, W), mode="bilinear",
                                     align_corners=False, antialias=False))
        return torch.cat(out, 0)

def preprocessor_for(dataset, channel_mode, pretraining, device,
                     augment=False, img_size=IMG_SIZE):
    """the protocol table routing. Satlas -> no standardization; ImageNet and scratch ->
    ImageNet standardization. Identical raw scaling for every arm."""
    return GPUPreprocessor(dataset, channel_mode,
                           standardize=(pretraining != "satlas"),
                           device=device, img_size=img_size, augment=augment)

# --------
# RESISC45 stratified split (see the paper)
# --------
def build_resisc45_splits(data_root: str, out_dir: str, seed: int = 42):
    """Build once and persist the stratified 70/15/15 split; reuse verbatim.

    Paths are written RELATIVE to data_root. The original code stored absolute
    paths under a relative `splits/` directory, so a split generated in one
    Colab session could not be reused in the next -- which defeats the purpose
    of persisting it at all (the paper requires it be "reused verbatim by
    every configuration and every seed").
    """
    from sklearn.model_selection import train_test_split
    os.makedirs(out_dir, exist_ok=True)
    paths = {s: os.path.join(out_dir, f"resisc45_{s}.csv")
             for s in ("train", "val", "test")}
    if all(os.path.exists(p) for p in paths.values()):
        return paths

    classes = sorted(d for d in os.listdir(data_root)
                     if os.path.isdir(os.path.join(data_root, d)))
    assert len(classes) == 45, (
        f"Expected 45 class folders, found {len(classes)} in {data_root}.\n"
        "Reduced mirrors of RESISC45 are in circulation and are NOT "
        "interchangeable with the full release. Use the complete "
        "45 x 700 NWPU-RESISC45.")
    class_to_idx = {c: i for i, c in enumerate(classes)}

    files, labels = [], []
    for c in classes:
        cdir = os.path.join(data_root, c)
        cfiles = sorted(f for f in os.listdir(cdir)
                        if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif")))
        assert len(cfiles) == 700, (
            f"{c}: expected 700 images, found {len(cfiles)}. This is not the "
            "full NWPU-RESISC45 release.")
        files += [f"{c}/{f}" for f in cfiles]
        labels += [class_to_idx[c]] * len(cfiles)

    tr_f, tmp_f, tr_l, tmp_l = train_test_split(
        files, labels, test_size=0.30, stratify=labels, random_state=seed)
    va_f, te_f, va_l, te_l = train_test_split(
        tmp_f, tmp_l, test_size=0.50, stratify=tmp_l, random_state=seed)

    for name, (fs, ls) in {"train": (tr_f, tr_l), "val": (va_f, va_l),
                           "test": (te_f, te_l)}.items():
        rows = sorted(zip(fs, ls))               # deterministic file order
        with open(paths[name], "w", newline="") as fh:
            w = csv.writer(fh)
            w.writerow(["filepath", "label", "class"])
            for fp, lb in rows:
                w.writerow([fp, lb, classes[lb]])
            fh.flush()
            os.fsync(fh.fileno())    # the split lives on Drive; force it down
    return paths

Overwriting src/datasets.py


In [ ]:
%%writefile src/models.py
"""Model factory: backbone x pretraining source x input channel count.

Head is ONE identical family for every configuration (the paper, design
freeze):
    LayerNorm(d) -> Linear(d, 512) -> GELU -> Dropout(0.3) -> Linear(512, K)

ARCHITECTURE CONTROL. SatlasPretrain's
`SwinBackbone` is built on torchvision.models.swin_v2_b -- Swin **V2**. The
original code paired those checkpoints against torchvision swin_b, Swin **V1**,
so every Satlas-vs-ImageNet Swin comparison carried a V1/V2 architecture
difference (cosine attention, post-norm residuals, log-CPB relative position
bias) inside it. the paper states architecture is a controlled factor, so the
arms are aligned on swin_v2_b, for which torchvision publishes
Swin_V2_B_Weights.IMAGENET1K_V1. This is applied identically to every Swin
configuration -- it removes a confound rather than introducing a per-arm
choice. Set SWIN_ARCH='swin_b' in the Config cell to revert; `swin_arch` is
logged on every result row either way.
"""
import re
import inspect

import torch
import torch.nn as nn
import torchvision.models as tv_models

FEAT_DIM = {"resnet50": 2048, "swin_b": 1024}
CHANNELS = {"rgb3": 3, "ms9": 9}

SATLAS_IDS = {
    ("resnet50", "ms"):     "Sentinel2_Resnet50_SI_MS",
    ("swin_b",   "ms"):     "Sentinel2_SwinB_SI_MS",
    ("resnet50", "rgb"):    "Sentinel2_Resnet50_SI_RGB",
    ("swin_b",   "rgb"):    "Sentinel2_SwinB_SI_RGB",
    ("swin_b",   "aerial"): "Aerial_SwinB_SI",
}
SATLAS_CHANNELS = {"ms": 9, "rgb": 3, "aerial": 3}

# Aligned with SatlasPretrain's SwinBackbone. Overridable from the Config cell.
SWIN_ARCH = "swin_v2_b"

# --------
# First-layer channel inflation (see the paper)
# --------
def adapt_first_layer(conv: nn.Conv2d, in_ch: int) -> nn.Conv2d:
    """Inflate a 3-channel first conv to `in_ch` channels.

    Pretrained RGB kernels are copied into the first three input channels and
    the remaining channels cycle those same kernels (input channel i gets the
    pretrained kernel of channel i mod 3). The complete tensor is then rescaled
    by 3/in_ch.

    The rescaling is not cosmetic: an inflated layer sums over in_ch/3 times as
    many input channels as it was calibrated for and would otherwise produce
    pre-activations that much larger, so the frozen-backbone stage would begin
    from a feature distribution no downstream weight has ever seen and the
    resulting deficit would be misread as ImageNet features transferring poorly
    to multispectral input. Repeat-and-rescale originates with Carreira &
    Zisserman (arXiv:1705.07750); cycling rather than mean-of-RGB is used so no
    channel is initialised from a colour-agnostic average.
    """
    assert conv.in_channels == 3, f"expected 3-channel conv, got {conv.in_channels}"
    if in_ch == 3:
        return conv
    new = nn.Conv2d(in_ch, conv.out_channels, conv.kernel_size,
                    stride=conv.stride, padding=conv.padding,
                    dilation=conv.dilation, groups=conv.groups,
                    bias=conv.bias is not None)
    with torch.no_grad():
        w = conv.weight.data                        # (out, 3, kh, kw)
        idx = [i % 3 for i in range(in_ch)]         # R,G,B,R,G,B,...
        new.weight.data = w[:, idx].clone() * (3.0 / in_ch)
        if conv.bias is not None:
            new.bias.data = conv.bias.data.clone()
    return new

# --------
# Classifier
# --------
class FMClassifier(nn.Module):
    """Backbone -> pooled feature -> shared classification head."""

    def __init__(self, backbone, feat_dim, num_classes, returns_fmap_list,
                 family, dropout=0.3, hidden=512):
        super().__init__()
        self.backbone = backbone
        self.returns_list = returns_fmap_list
        self.family = family                 # resnet50 | swin_b
        self.feat_dim = feat_dim
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )

    def forward_features(self, x):
        """Pooled backbone feature. This is what the Stage-1 cache stores."""
        feats = self.backbone(x)
        if self.returns_list:
            # the paper: backbones returning a feature-map list are reduced by
            # adaptive average pooling before the head.
            return torch.flatten(self.pool(feats[-1]), 1)
        return feats

    def forward_head(self, f):
        return self.classifier(f)

    def forward(self, x):
        return self.classifier(self.forward_features(x))

    def set_backbone_trainable(self, trainable: bool):
        for p in self.backbone.parameters():
            p.requires_grad = trainable

# --------
# Swin variant detection / alignment
# --------
def swin_variant_of(module: nn.Module) -> str:
    """'v2' if any block is a SwinTransformerBlockV2, else 'v1'."""
    for m in module.modules():
        if type(m).__name__ == "SwinTransformerBlockV2":
            return "v2"
    for m in module.modules():
        if type(m).__name__ == "SwinTransformerBlock":
            return "v1"
    return "unknown"

def _build_tv_swin(arch: str, pretrained: bool):
    if arch == "swin_v2_b":
        w = tv_models.Swin_V2_B_Weights.IMAGENET1K_V1 if pretrained else None
        return tv_models.swin_v2_b(weights=w)
    if arch == "swin_b":
        w = tv_models.Swin_B_Weights.IMAGENET1K_V1 if pretrained else None
        return tv_models.swin_b(weights=w)
    raise ValueError(f"unknown swin arch {arch!r}")

def load_satlas_backbone(identifier: str, device=None):
    """Load a SatlasPretrain backbone, tolerating signature differences.

    get_pretrained_model's `device` default is not documented and has been
    'cuda' in some releases, which breaks on a CPU-only box (the test suite).
    The signature is introspected and only supported kwargs are passed.
    """
    from satlaspretrain_models import Weights
    fn = Weights().get_pretrained_model
    kwargs = {}
    try:
        params = inspect.signature(fn).parameters
    except (TypeError, ValueError):
        params = {}
    if "fpn" in params:
        kwargs["fpn"] = False
    if "head" in params:
        kwargs["head"] = None
    if "device" in params and device is not None:
        kwargs["device"] = str(device)
    return fn(identifier, **kwargs)

# --------
# Factory
# --------
def build_model(backbone_name, pretraining, num_classes, channel_mode="rgb3",
                satlas_variant="ms", swin_arch=None, dropout=0.3, hidden=512,
                device=None):
    """backbone_name in {resnet50, swin_b}
       pretraining   in {imagenet, satlas, scratch}
       channel_mode  in {rgb3, ms9}
       satlas_variant in {ms, rgb, aerial} (ignored unless pretraining=satlas)
    """
    assert backbone_name in FEAT_DIM and channel_mode in CHANNELS
    assert pretraining in ("imagenet", "satlas", "scratch")
    swin_arch = swin_arch or SWIN_ARCH
    feat_dim, in_ch = FEAT_DIM[backbone_name], CHANNELS[channel_mode]

    if pretraining == "satlas":
        key = (backbone_name, satlas_variant)
        assert key in SATLAS_IDS, f"no Satlas checkpoint for {key}"
        expected = SATLAS_CHANNELS[satlas_variant]
        assert in_ch == expected, (
            f"Satlas '{satlas_variant}' expects {expected} channels, got "
            f"{in_ch}. Use satlas_variant='rgb' for 3-band configs.")
        backbone = load_satlas_backbone(SATLAS_IDS[key], device=device)
        return FMClassifier(backbone, feat_dim, num_classes, True,
                            backbone_name, dropout, hidden)

    if backbone_name == "resnet50":
        w = tv_models.ResNet50_Weights.IMAGENET1K_V2 if pretraining == "imagenet" else None
        backbone = tv_models.resnet50(weights=w)
        if pretraining == "imagenet":
            backbone.conv1 = adapt_first_layer(backbone.conv1, in_ch)
        elif in_ch != 3:
            # No inflation is required or possible without pretrained kernels:
            # the first layer is simply a random conv of the required width.
            backbone.conv1 = nn.Conv2d(in_ch, 64, 7, 2, 3, bias=False)
        backbone.fc = nn.Identity()
        return FMClassifier(backbone, feat_dim, num_classes, False,
                            backbone_name, dropout, hidden)

    backbone = _build_tv_swin(swin_arch, pretrained=(pretraining == "imagenet"))
    proj = backbone.features[0][0]                  # patch-embedding Conv2d
    if pretraining == "imagenet":
        backbone.features[0][0] = adapt_first_layer(proj, in_ch)
    elif in_ch != 3:
        backbone.features[0][0] = nn.Conv2d(
            in_ch, proj.out_channels, proj.kernel_size, stride=proj.stride)
    backbone.head = nn.Identity()
    return FMClassifier(backbone, feat_dim, num_classes, False, backbone_name,
                        dropout, hidden)

# --------
# Layer-wise learning-rate decay (Stage 2)
# --------
# Both backbones are divided into exactly the SAME number of stages (5), so the
# multiplier SET is identical for ResNet-50 and Swin: {g^4, g^3, g^2, g^1, 1}.
# Mapping by a shared rule rather than by a shared number is what keeps this
# uniform -- had the groups been "one per parameter tensor", ResNet-50 and
# Swin-B would have received different effective decay ranges from the same
# gamma, which would be a per-backbone hyperparameter in disguise.
_SWIN_FEATURE_TO_STAGE = {0: 0, 1: 1, 2: 2, 3: 2, 4: 3, 5: 3, 6: 4, 7: 4}

def resnet_stage_of(name: str) -> int:
    n = "." + name
    for i in (1, 2, 3, 4):
        if f".layer{i}." in n:
            return i
    return 0            # stem: conv1 / bn1 (possibly under a 'resnet.' prefix)

def swin_stage_of(name: str) -> int:
    m = re.search(r"features\.(\d+)", name)
    if m:
        return _SWIN_FEATURE_TO_STAGE.get(int(m.group(1)), 4)
    return 4            # trailing norm / anything after the last stage

def stage_of(name: str, family: str) -> int:
    return resnet_stage_of(name) if family == "resnet50" else swin_stage_of(name)

def llrd_param_groups(model: FMClassifier, base_lr: float, gamma: float,
                      n_stages: int, head_mult: float, weight_decay: float):
    """Stage-2 parameter groups: layer-wise decay on the backbone, a single
    discriminative multiplier on the shared head.

    Stage n_stages-1 (deepest) keeps base_lr; each earlier stage is multiplied
    by an extra factor of gamma. The head gets base_lr * head_mult.
    """
    buckets = {i: [] for i in range(n_stages)}
    head = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("classifier."):
            head.append(p)
        else:
            s = stage_of(name, model.family)
            buckets[min(max(s, 0), n_stages - 1)].append(p)
    groups = []
    for s in range(n_stages):
        if not buckets[s]:
            continue
        groups.append({
            "params": buckets[s],
            "lr": base_lr * (gamma ** (n_stages - 1 - s)),
            "weight_decay": weight_decay,
            "name": f"backbone_stage{s}",
        })
    if head:
        groups.append({"params": head, "lr": base_lr * head_mult,
                       "weight_decay": weight_decay, "name": "head"})
    assert groups, "no trainable parameters"
    return groups

def head_param_groups(model: FMClassifier, lr: float, weight_decay: float):
    """Stage 1: only the classification head is trainable."""
    params = [p for n, p in model.named_parameters()
              if p.requires_grad and n.startswith("classifier.")]
    assert params, "Stage 1 has no trainable head parameters"
    return [{"params": params, "lr": lr, "weight_decay": weight_decay,
             "name": "head"}]

# --------
# Uniform throughput / memory options
# --------
def set_bn_eval(model: nn.Module) -> int:
    """Put every BatchNorm in eval mode (running stats frozen). Opt-in for
    Stage 2, applied to all configurations or none."""
    n = 0
    for m in model.modules():
        if isinstance(m, nn.modules.batchnorm._BatchNorm):
            m.eval()
            n += 1
    return n

class _Ckpt(nn.Module):
    def __init__(self, mod):
        super().__init__()
        self.mod = mod

    def forward(self, *a, **kw):
        from torch.utils.checkpoint import checkpoint
        if self.training and torch.is_grad_enabled():
            return checkpoint(self.mod, *a, use_reentrant=False, **kw)
        return self.mod(*a, **kw)

def enable_grad_checkpointing(model: FMClassifier) -> int:
    """Trade compute for memory without changing the computed function.

    This is how memory pressure is handled -- never by shrinking the batch
    (frozen at 32) and never by gradient accumulation, which would change
    ResNet-50's BatchNorm statistics while leaving Swin's LayerNorm untouched.
    Decided once, grid-wide, at preflight.
    """
    n = 0
    for m in model.backbone.modules():
        seq = getattr(m, "features", None)
        if isinstance(seq, nn.Sequential):
            for i in range(1, len(seq)):
                seq[i] = _Ckpt(seq[i])
                n += 1
    for m in model.backbone.modules():
        for attr in ("layer1", "layer2", "layer3", "layer4"):
            sub = getattr(m, attr, None)
            if isinstance(sub, nn.Sequential):
                setattr(m, attr, _Ckpt(sub))
                n += 1
    return n

def count_params(model: nn.Module):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr

Overwriting src/models.py


In [ ]:
%%writefile src/featcache.py
"""Stage-1 feature cache.

In Stage 1 the backbone is frozen and augmentation is off, so every epoch
pushes identical images through identical weights and recomputes identical
features. Stage 1 runs at least 15 such epochs (more once the step floor
engages). It needs one.

The cache is keyed on a fingerprint of the backbone's actual weights, not on
assumptions about which runs share them. For every pretrained arm the Stage-1
weights are identical across all 6 budgets and 5 seeds, so 30 runs share one
computation; scratch arms differ per seed, so their caches are per seed. Both
fall out of the key automatically, and a stale cache is impossible by
construction.

Train features are computed over the ENTIRE training split once and then
row-indexed for each k-shot subset, so all six budgets of a configuration
share one pass.

CORRECTNESS, NOT ONLY SPEED. The backbone is put in eval() mode, not merely
requires_grad=False. In train() mode BatchNorm running statistics still update
and stochastic depth still fires, so a "frozen" backbone is not actually frozen
and its features are not reproducible epoch to epoch.
"""
import os
import contextlib

import numpy as np
import torch
from torch.utils.data import DataLoader

from utils import worker_init_fn

def _autocast(device, amp_dtype):
    if amp_dtype is None or device.type != "cuda":
        return contextlib.nullcontext()
    return torch.autocast(device_type="cuda", dtype=amp_dtype)

@torch.no_grad()
def extract_features(model, dataset, prep, device, amp_dtype, batch_size,
                     num_workers=2):
    """One forward pass over `dataset`, returning pooled features (float32).

    The backbone is in eval() mode and no augmentation is applied.
    """
    was_training = model.training
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=(device.type == "cuda"),
                        worker_init_fn=worker_init_fn, drop_last=False)
    feats, labels = [], []
    for batch in loader:
        x = prep(batch["image"])
        if model.family == "resnet50" and x.is_cuda:
            x = x.contiguous(memory_format=torch.channels_last)
        with _autocast(device, amp_dtype):
            f = model.forward_features(x)
        feats.append(f.float().cpu())
        labels.append(batch["label"])
    if was_training:
        model.train()
    return torch.cat(feats).contiguous(), torch.cat(labels).contiguous()

class FeatureCache:
    """On-disk float32 feature cache under the transient scratch directory."""

    def __init__(self, cache_dir: str, enabled: bool = True):
        self.dir = cache_dir
        self.enabled = enabled
        os.makedirs(cache_dir, exist_ok=True)
        self.hits = 0
        self.misses = 0

    def _paths(self, key):
        return (os.path.join(self.dir, f"{key}_f.npy"),
                os.path.join(self.dir, f"{key}_y.npy"))

    def get_or_compute(self, key, compute_fn):
        """compute_fn() -> (features, labels) tensors."""
        fp, yp = self._paths(key)
        if self.enabled and os.path.exists(fp) and os.path.exists(yp):
            self.hits += 1
            return (torch.from_numpy(np.load(fp)),
                    torch.from_numpy(np.load(yp)))
        self.misses += 1
        f, y = compute_fn()
        if self.enabled:
            tmp_f, tmp_y = fp + ".tmp", yp + ".tmp"
            with open(tmp_f, "wb") as fh:
                np.save(fh, f.numpy())
            with open(tmp_y, "wb") as fh:
                np.save(fh, y.numpy())
            os.replace(tmp_f, fp)      # atomic: a killed session cannot leave
            os.replace(tmp_y, yp)      # a half-written cache behind
        return f, y

    def key(self, dataset, config_id, backbone_fp, split):
        return f"{dataset}__{config_id}__{backbone_fp}__{split}"

    def sweep(self, keep_prefix=None, max_gb=None):
        """Drop old cache files if the scratch disk gets tight."""
        if max_gb is None:
            return 0
        files = [os.path.join(self.dir, f) for f in os.listdir(self.dir)]
        files = [f for f in files if os.path.isfile(f)]
        total = sum(os.path.getsize(f) for f in files)
        if total <= max_gb * 1e9:
            return 0
        files.sort(key=lambda f: os.path.getmtime(f))
        freed = 0
        for f in files:
            if keep_prefix and os.path.basename(f).startswith(keep_prefix):
                continue
            sz = os.path.getsize(f)
            os.remove(f)
            freed += sz
            total -= sz
            if total <= 0.7 * max_gb * 1e9:
                break
        return freed

Overwriting src/featcache.py


In [ ]:
%%writefile src/train.py
"""The two-stage transfer protocol (see the paper), identical for every config.

  Stage 1 -- frozen transfer.  Backbone frozen AND in eval() mode; only the
             classification head is trained. Runs on cached features.
  Stage 2 -- full fine-tuning. All parameters unfrozen, reduced learning rate.

Model selection is on best validation accuracy within each stage, restored at
the end of that stage. The test partition is used for reporting only, at
Stage 1 exactly as at Stage 2 -- nothing (checkpoint, epoch count or
hyperparameter) is ever selected using test data.

Uniform optimisation upgrades, applied identically to all 17 configurations:
mixed precision, linear warmup then cosine over the actual step count, global
grad-norm clipping, layer-wise lr decay and a discriminative head lr in
Stage 2, optional weight EMA, optional Stage-2 BatchNorm freeze, channels_last
for the convolutional backbone. See protocol.py for the frozen default set.
"""
import copy
import time
import contextlib

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from protocol import PROTOCOL, plan_stage, lr_scale
from models import llrd_param_groups, head_param_groups, set_bn_eval
from utils import loader_generator, worker_init_fn

def _autocast(device, amp_dtype):
    if amp_dtype is None or device.type != "cuda":
        return contextlib.nullcontext()
    return torch.autocast(device_type="cuda", dtype=amp_dtype)

def _make_scaler(amp_dtype):
    enabled = amp_dtype == torch.float16
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)

class EMA:
    """Exponential moving average of trainable weights (opt-in).

    Selected on validation like everything else: each epoch both the raw and
    the EMA weights are scored on the validation split and the better of the
    two is what the stage's best-checkpoint tracker sees.
    """

    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float()
                       for k, v in model.state_dict().items()
                       if v.dtype.is_floating_point}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(),
                                                     alpha=1.0 - self.decay)

    def state_dict(self, model):
        sd = copy.deepcopy(model.state_dict())
        for k, v in self.shadow.items():
            sd[k] = v.to(sd[k].dtype)
        return sd

# --------
# Stage 1: head-only, on frozen features
# --------
def run_stage1(model, train_feats, train_labels, val_feats, val_labels,
               device, proto=PROTOCOL, seed=0, verbose=False):
    """Train the shared head on pre-extracted features.

    Mathematically identical to running the frozen backbone every epoch: the
    backbone is in eval() mode, so its output is a deterministic function of
    the input, and augmentation is off in Stage 1 for the main tiers.
    """
    n = len(train_labels)
    plan = plan_stage(n, 1, proto)
    model.classifier.to(device)
    train_feats = train_feats.to(device)
    train_labels = train_labels.to(device)
    val_feats = val_feats.to(device)
    val_labels = val_labels.to(device)

    ds = TensorDataset(train_feats, train_labels)
    g = loader_generator(seed)
    loader = DataLoader(ds, batch_size=plan["batch_size"], shuffle=True,
                        generator=g, drop_last=plan["drop_last"],
                        num_workers=0)

    crit = nn.CrossEntropyLoss(label_smoothing=proto.label_smoothing)
    opt = torch.optim.AdamW(
        head_param_groups(model, proto.stage1_lr, proto.stage1_wd))
    base_lrs = [pg["lr"] for pg in opt.param_groups]

    hist = {"accuracy": [], "train_loss": [], "val_accuracy": [], "val_loss": []}
    best_val, best_state, best_epoch, step = -1.0, None, -1, 0
    for ep in range(plan["epochs"]):
        model.classifier.train()
        tl, tc, tn = 0.0, 0, 0
        for f, y in loader:
            s = lr_scale(step, plan["total_steps"], plan["warmup_steps"])
            for pg, b in zip(opt.param_groups, base_lrs):
                pg["lr"] = b * s
            opt.zero_grad(set_to_none=True)
            out = model.forward_head(f)
            loss = crit(out, y)
            loss.backward()
            if proto.grad_clip_norm:
                nn.utils.clip_grad_norm_(model.classifier.parameters(),
                                         proto.grad_clip_norm)
            opt.step()
            step += 1
            tl += loss.item() * y.size(0)
            tc += (out.argmax(1) == y).sum().item()
            tn += y.size(0)

        hist["accuracy"].append(tc / max(tn, 1))
        hist["train_loss"].append(tl / max(tn, 1))
        if ep not in plan["val_epochs"]:
            hist["val_accuracy"].append(float("nan"))
            hist["val_loss"].append(float("nan"))
            continue
        vl, vc = _eval_features(model, val_feats, val_labels, crit,
                                plan["batch_size"])
        hist["val_accuracy"].append(vc)
        hist["val_loss"].append(vl)
        if vc > best_val:
            best_val, best_epoch = vc, ep
            best_state = copy.deepcopy(model.classifier.state_dict())
        if verbose:
            print(f"    s1 ep{ep+1}/{plan['epochs']} val={vc:.4f}", flush=True)

    if best_state is not None:
        model.classifier.load_state_dict(best_state)
    return hist, plan, best_val, best_epoch

@torch.no_grad()
def _eval_features(model, feats, labels, crit, bs):
    model.classifier.eval()
    tot_loss, correct = 0.0, 0
    for i in range(0, len(labels), bs):
        f, y = feats[i:i + bs], labels[i:i + bs]
        out = model.forward_head(f)
        tot_loss += crit(out, y).item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
    n = max(len(labels), 1)
    return tot_loss / n, correct / n

@torch.no_grad()
def predict_from_features(model, feats, labels, bs, device):
    model.classifier.eval()
    probs = []
    feats = feats.to(device)
    for i in range(0, len(labels), bs):
        out = torch.softmax(model.forward_head(feats[i:i + bs]).float(), dim=1)
        probs.append(out.cpu().numpy())
    p = np.concatenate(probs)
    return np.asarray(labels), p.argmax(1), p

# --------
# Stage 2: full fine-tuning on images
# --------
def run_stage2(model, train_ds, val_ds, prep_train, prep_eval, device,
               proto=PROTOCOL, seed=0, amp_dtype=None, num_workers=2,
               verbose=False):
    n = len(train_ds)
    plan = plan_stage(n, 2, proto)
    model.to(device)
    model.set_backbone_trainable(True)
    if proto.channels_last and model.family == "resnet50" and device.type == "cuda":
        model.to(memory_format=torch.channels_last)

    g = loader_generator(seed + 1)
    train_loader = DataLoader(
        train_ds, batch_size=plan["batch_size"], shuffle=True, generator=g,
        drop_last=plan["drop_last"], num_workers=num_workers,
        pin_memory=(device.type == "cuda"), worker_init_fn=worker_init_fn,
        persistent_workers=(num_workers > 0))
    val_loader = DataLoader(
        val_ds, batch_size=plan["batch_size"], shuffle=False,
        num_workers=num_workers, pin_memory=(device.type == "cuda"),
        worker_init_fn=worker_init_fn, persistent_workers=(num_workers > 0))

    crit = nn.CrossEntropyLoss(label_smoothing=proto.label_smoothing)
    opt = torch.optim.AdamW(llrd_param_groups(
        model, proto.stage2_lr, proto.llrd_gamma, proto.llrd_n_stages,
        proto.head_lr_mult, proto.stage2_wd))
    base_lrs = [pg["lr"] for pg in opt.param_groups]
    scaler = _make_scaler(amp_dtype)
    ema = EMA(model, proto.ema_decay) if proto.use_ema else None
    aug_gen = torch.Generator().manual_seed(seed + 7)

    hist = {"accuracy": [], "train_loss": [], "val_accuracy": [], "val_loss": []}
    best_val, best_state, best_epoch, step = -1.0, None, -1, 0
    for ep in range(plan["epochs"]):
        model.train()
        if proto.freeze_bn_stage2:
            set_bn_eval(model)
        tl, tc, tn = 0.0, 0, 0
        for batch in train_loader:
            s = lr_scale(step, plan["total_steps"], plan["warmup_steps"])
            for pg, b in zip(opt.param_groups, base_lrs):
                pg["lr"] = b * s
            x = prep_train(batch["image"], generator=aug_gen)
            if proto.channels_last and model.family == "resnet50" and x.is_cuda:
                x = x.contiguous(memory_format=torch.channels_last)
            y = batch["label"].to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with _autocast(device, amp_dtype):
                out = model(x)
                loss = crit(out, y)
            if scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), proto.grad_clip_norm)
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), proto.grad_clip_norm)
                opt.step()
            if ema is not None:
                ema.update(model)
            step += 1
            tl += loss.item() * y.size(0)
            tc += (out.argmax(1) == y).sum().item()
            tn += y.size(0)

        hist["accuracy"].append(tc / max(tn, 1))
        hist["train_loss"].append(tl / max(tn, 1))
        if ep not in plan["val_epochs"]:
            hist["val_accuracy"].append(float("nan"))
            hist["val_loss"].append(float("nan"))
            continue
        vl, vc = evaluate_images(model, val_loader, prep_eval, crit, device,
                                 amp_dtype)
        cand = [(vc, None)]
        if ema is not None:
            raw = copy.deepcopy(model.state_dict())
            model.load_state_dict(ema.state_dict(model))
            _, vc_ema = evaluate_images(model, val_loader, prep_eval, crit,
                                        device, amp_dtype)
            model.load_state_dict(raw)
            cand.append((vc_ema, "ema"))
        pick_val, pick_kind = max(cand, key=lambda t: t[0])

        hist["val_accuracy"].append(pick_val)
        hist["val_loss"].append(vl)
        if pick_val > best_val:
            best_val, best_epoch = pick_val, ep
            best_state = copy.deepcopy(
                ema.state_dict(model) if pick_kind == "ema"
                else model.state_dict())
        if verbose:
            print(f"    s2 ep{ep+1}/{plan['epochs']} val={pick_val:.4f}",
                  flush=True)

    if best_state is not None:
        model.load_state_dict(best_state)
    return hist, plan, best_val, best_epoch

def run_stage1_uncached(model, train_ds, val_ds, prep_train, prep_eval, device,
                        proto=PROTOCOL, seed=0, amp_dtype=None, num_workers=0,
                        aug_generator=None):
    """Stage 1 without the feature cache: the backbone is re-run every epoch.

    Kept for two reasons. (1) The augmentation tier needs it -- with
    augmentation on, the images differ every epoch, so Stage-1 features are not
    cacheable. (2) It is the reference path the cache-equivalence test checks
    against.

    RNG streams are constructed to match the cached path exactly: the same
    generator seed over the same number of items gives the same permutation,
    and an eval()-mode backbone under no_grad draws no RNG, so the head's
    dropout stream is identical in both paths.
    """
    n = len(train_ds)
    plan = plan_stage(n, 1, proto)
    model.to(device)
    model.set_backbone_trainable(False)

    g = loader_generator(seed)
    train_loader = DataLoader(
        train_ds, batch_size=plan["batch_size"], shuffle=True, generator=g,
        drop_last=plan["drop_last"], num_workers=num_workers,
        worker_init_fn=worker_init_fn)
    val_loader = DataLoader(val_ds, batch_size=plan["batch_size"],
                            shuffle=False, num_workers=num_workers,
                            worker_init_fn=worker_init_fn)

    crit = nn.CrossEntropyLoss(label_smoothing=proto.label_smoothing)
    opt = torch.optim.AdamW(
        head_param_groups(model, proto.stage1_lr, proto.stage1_wd))
    base_lrs = [pg["lr"] for pg in opt.param_groups]

    hist = {"accuracy": [], "train_loss": [], "val_accuracy": [], "val_loss": []}
    best_val, best_state, best_epoch, step = -1.0, None, -1, 0
    for ep in range(plan["epochs"]):
        # eval() on the whole model: the backbone must be genuinely frozen.
        # requires_grad=False alone leaves BatchNorm running statistics
        # updating and stochastic depth firing.
        model.eval()
        model.classifier.train()
        tl, tc, tn = 0.0, 0, 0
        for batch in train_loader:
            s = lr_scale(step, plan["total_steps"], plan["warmup_steps"])
            for pg, b in zip(opt.param_groups, base_lrs):
                pg["lr"] = b * s
            x = prep_train(batch["image"], generator=aug_generator)
            y = batch["label"].to(device, non_blocking=True)
            with torch.no_grad(), _autocast(device, amp_dtype):
                f = model.forward_features(x)
            opt.zero_grad(set_to_none=True)
            out = model.forward_head(f.float())
            loss = crit(out, y)
            loss.backward()
            if proto.grad_clip_norm:
                nn.utils.clip_grad_norm_(model.classifier.parameters(),
                                         proto.grad_clip_norm)
            opt.step()
            step += 1
            tl += loss.item() * y.size(0)
            tc += (out.argmax(1) == y).sum().item()
            tn += y.size(0)

        hist["accuracy"].append(tc / max(tn, 1))
        hist["train_loss"].append(tl / max(tn, 1))
        if ep not in plan["val_epochs"]:
            hist["val_accuracy"].append(float("nan"))
            hist["val_loss"].append(float("nan"))
            continue
        vl, vc = evaluate_images(model, val_loader, prep_eval, crit, device,
                                 amp_dtype)
        hist["val_accuracy"].append(vc)
        hist["val_loss"].append(vl)
        if vc > best_val:
            best_val, best_epoch = vc, ep
            best_state = copy.deepcopy(model.classifier.state_dict())

    if best_state is not None:
        model.classifier.load_state_dict(best_state)
    return hist, plan, best_val, best_epoch

@torch.no_grad()
def evaluate_images(model, loader, prep, crit, device, amp_dtype):
    model.eval()
    tot, correct, n = 0.0, 0, 0
    for batch in loader:
        x = prep(batch["image"])
        if model.family == "resnet50" and x.is_cuda:
            x = x.contiguous(memory_format=torch.channels_last)
        y = batch["label"].to(device, non_blocking=True)
        with _autocast(device, amp_dtype):
            out = model(x)
        tot += crit(out.float(), y).item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        n += y.size(0)
    return tot / max(n, 1), correct / max(n, 1)

@torch.no_grad()
def predict_images(model, dataset, prep, device, bs, amp_dtype, num_workers=2):
    model.to(device).eval()
    loader = DataLoader(dataset, batch_size=bs, shuffle=False,
                        num_workers=num_workers,
                        pin_memory=(device.type == "cuda"),
                        worker_init_fn=worker_init_fn)
    probs, ys = [], []
    for batch in loader:
        x = prep(batch["image"])
        if model.family == "resnet50" and x.is_cuda:
            x = x.contiguous(memory_format=torch.channels_last)
        with _autocast(device, amp_dtype):
            out = model(x)
        probs.append(torch.softmax(out.float(), dim=1).cpu().numpy())
        ys.append(batch["label"].numpy())
    p = np.concatenate(probs)
    return np.concatenate(ys), p.argmax(1), p

def assert_probs_valid(p, tol=1e-3):
    s = p.sum(axis=1)
    bad = np.abs(s - 1.0) > tol
    if bad.any():
        raise AssertionError(
            f"{int(bad.sum())} predicted probability rows do not sum to 1 "
            f"(max deviation {np.abs(s-1.0).max():.2e})")
    if not np.isfinite(p).all():
        raise AssertionError("non-finite predicted probabilities")

def plot_history(h1, h2, save_path=None):
    """Fig. 7: training and validation curves across both stages, with the
    frozen-to-fine-tuning transition marked."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    acc = h1["accuracy"] + h2["accuracy"]
    loss = h1["train_loss"] + h2["train_loss"]
    vacc = h1["val_accuracy"] + h2["val_accuracy"]
    vloss = h1["val_loss"] + h2["val_loss"]
    fig, ax = plt.subplots(1, 2, figsize=(16, 5))
    for a, (tr, va, name) in zip(ax, [(acc, vacc, "Accuracy"),
                                      (loss, vloss, "Loss")]):
        a.plot(tr, label="Training")
        a.plot(va, label="Validation")
        a.axvline(len(h1["accuracy"]) - 0.5, ls="--", c="grey", lw=1)
        a.set_xlabel("Epoch"); a.set_ylabel(name); a.set_title(name); a.legend()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)          # 510 runs; unclosed figures leak
    return save_path

Overwriting src/train.py


In [ ]:
%%writefile src/metrics.py
"""Evaluation metrics (see the paper) and statistical tests (see the paper).

Six metrics are computed for every run at BOTH stages: overall accuracy,
macro-F1, weighted-F1, Cohen's kappa, one-vs-rest ROC-AUC and expected
calibration error. Per-class F1 and confusion matrices are derived from the
archived per-image predictions, never by retraining.
"""
import numpy as np
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score,
                             roc_auc_score, confusion_matrix)

# With five paired observations the smallest two-sided p-value the exact
# signed-rank test can return is 2/2**5 = 0.0625, so a determination at
# alpha = 0.05 is unreachable by construction regardless of effect size. The
# floor is emitted as a column wherever seed-level results appear, and the
# paired t-test on the same five differences is what carries any significance
# claim at that level.
WILCOXON_MIN_P_5_SEEDS = 0.0625

def expected_calibration_error(y_true, y_prob, n_bins: int = 15) -> float:
    """Fifteen equal-width bins on maximum-probability confidence; the
    population-weighted mean absolute gap between confidence and accuracy
    within each bin (see the paper)."""
    y_true = np.asarray(y_true)
    conf = y_prob.max(axis=1)
    pred = y_prob.argmax(axis=1)
    correct = (pred == y_true).astype(np.float64)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece, n = 0.0, len(y_true)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() == 0:
            continue
        ece += (m.sum() / n) * abs(correct[m].mean() - conf[m].mean())
    return float(ece)

def compute_metrics(y_true, y_pred, y_prob=None, n_classes=None) -> dict:
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    out = {
        "test_acc": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro",
                                   zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted",
                                      zero_division=0)),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
    }
    if y_prob is not None:
        k = n_classes or y_prob.shape[1]
        try:
            out["roc_auc_ovr"] = float(roc_auc_score(
                y_true, y_prob, multi_class="ovr", average="macro",
                labels=np.arange(k)))
        except ValueError:
            out["roc_auc_ovr"] = ""       # a class absent from y_true
        out["ece"] = expected_calibration_error(y_true, y_prob)
    else:
        out["roc_auc_ovr"], out["ece"] = "", ""
    return out

def per_class_f1(y_true, y_pred, n_classes):
    return f1_score(y_true, y_pred, average=None, zero_division=0,
                    labels=np.arange(n_classes))

def confusion(y_true, y_pred, n_classes):
    return confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))

# -------- the paper ----
def mcnemar_test(y_true, preds_a, preds_b):
    """Exact binomial form when the number of DISCORDANT PAIRS is below 25,
    chi-square approximation otherwise.

    The original code switched on min(b, c) rather than on b + c; the paper
    specifies "the number of discordant pairs", which is their sum.
    """
    from statsmodels.stats.contingency_tables import mcnemar
    y_true = np.asarray(y_true)
    ca = np.asarray(preds_a) == y_true
    cb = np.asarray(preds_b) == y_true
    b = int(np.sum(ca & ~cb))
    c = int(np.sum(~ca & cb))
    table = [[int(np.sum(ca & cb)), b], [c, int(np.sum(~ca & ~cb))]]
    n_disc = b + c
    if n_disc == 0:
        return float("nan"), 0, b, c
    p = float(mcnemar(table, exact=(n_disc < 25)).pvalue)
    return p, n_disc, b, c

def holm_bonferroni(pvalues):
    """Holm step-down adjusted p-values, input order preserved."""
    p = np.asarray(pvalues, dtype=float)
    n = len(p)
    if n == 0:
        return np.asarray([])
    order = np.argsort(p)
    adj = np.empty(n, dtype=float)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (n - rank) * p[i])
        adj[i] = min(running, 1.0)
    return adj

def wilcoxon_across_seeds(acc_a, acc_b):
    from scipy.stats import wilcoxon
    a, b = np.asarray(acc_a, float), np.asarray(acc_b, float)
    if len(a) != len(b) or len(a) < 5 or np.allclose(a, b):
        return float("nan"), float("nan")
    try:
        s, p = wilcoxon(a, b)
        return float(s), float(p)
    except ValueError:
        return float("nan"), float("nan")

def paired_ttest_across_seeds(acc_a, acc_b):
    """Reported alongside the signed-rank result, and not optional -- see
    WILCOXON_MIN_P_5_SEEDS."""
    from scipy.stats import ttest_rel
    a, b = np.asarray(acc_a, float), np.asarray(acc_b, float)
    if len(a) != len(b) or len(a) < 3 or np.allclose(a, b):
        return float("nan"), float("nan")
    try:
        t, p = ttest_rel(a, b)
        return float(t), float(p)
    except ValueError:
        return float("nan"), float("nan")

def error_overlap(y_true, preds_a, preds_b) -> dict:
    """Jaccard index between two configurations' error sets. Two models at
    equal accuracy that fail on disjoint images have learned materially
    different things."""
    y_true = np.asarray(y_true)
    ea = np.asarray(preds_a) != y_true
    eb = np.asarray(preds_b) != y_true
    inter, union = int(np.sum(ea & eb)), int(np.sum(ea | eb))
    return {"n_err_a": int(ea.sum()), "n_err_b": int(eb.sum()),
            "n_err_both": inter,
            "jaccard": (inter / union) if union else float("nan")}

def is_non_monotonic(values, tol=1e-9) -> bool:
    """True if a label-efficiency curve ever decreases as the budget grows."""
    v = [x for x in values if x is not None and not np.isnan(x)]
    return any(v[i + 1] < v[i] - tol for i in range(len(v) - 1))

Overwriting src/metrics.py


In [ ]:
%%writefile src/staging.py
"""Data staging: acquire, extract, verify, cache.

Rules this module enforces:
  * Archives are copied from Drive to /content and extracted THERE. Training
    never reads from a Drive-mounted path -- many small reads over the Drive
    FUSE layer are pathologically slow (often 50-100x slower than local disk).
  * Integrity is asserted before anything else, and failures are loud and
    specific. Reduced mirrors of RESISC45 are in circulation; they are not
    interchangeable with the full 45 x 700 release.
  * Every step is idempotent, so a restarted session skips completed work.
"""
import os
import csv
import glob
import json
import shutil
import zipfile
import tarfile
import subprocess

import numpy as np

from datasets import (EUROSAT_SATLAS_BANDS, N_BANDS_CACHED, EUROSAT_NATIVE,
                      RESISC_NATIVE, EUROSAT_CLASSES, build_resisc45_splits)

EUROSAT_KAGGLE = "apollo2506/eurosat-dataset"
_SPLIT_CSVS = {"train": ["train.csv"],
               "val": ["validation.csv", "val.csv"],
               "test": ["test.csv"]}

def _log(msg):
    print(f"[stage] {msg}", flush=True)

# --------
# Archive handling
# --------
def extract_archive(archive: str, dest: str, marker: str = None) -> str:
    """Extract to dest, idempotently. `marker` is a path that, if present,
    means extraction already happened."""
    os.makedirs(dest, exist_ok=True)
    if marker and os.path.exists(marker):
        _log(f"already extracted: {marker}")
        return dest
    low = archive.lower()
    _log(f"extracting {os.path.basename(archive)} -> {dest}")
    if low.endswith(".zip"):
        with zipfile.ZipFile(archive) as z:
            z.extractall(dest)
    elif low.endswith((".tar", ".tar.gz", ".tgz", ".tar.bz2", ".tar.xz")):
        with tarfile.open(archive) as t:
            t.extractall(dest)
    elif low.endswith(".rar"):
        if shutil.which("unrar") is None:
            subprocess.run(["apt-get", "-qq", "install", "-y", "unrar"],
                           check=False)
        if shutil.which("unrar") is None:
            raise RuntimeError(
                f"{archive} is a .rar and `unrar` is unavailable. Re-upload "
                "the dataset as a .zip, or run "
                "`!apt-get install -y unrar` in the Setup cell.")
        subprocess.run(["unrar", "x", "-o+", "-idq", archive, dest], check=True)
    elif low.endswith(".7z"):
        try:
            import py7zr
        except ImportError as e:
            raise RuntimeError(
                f"{archive} is a .7z; `pip install py7zr` first.") from e
        with py7zr.SevenZipFile(archive) as z:
            z.extractall(dest)
    else:
        raise RuntimeError(f"unsupported archive type: {archive}")
    return dest

def stage_archive(src: str, scratch_archives: str) -> str:
    """Copy an archive off Drive onto local disk before extracting."""
    assert os.path.exists(src), f"archive not found: {src}"
    dst = os.path.join(scratch_archives, os.path.basename(src))
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        _log(f"archive already staged: {dst}")
        return dst
    _log(f"copying {src} -> {dst} ({os.path.getsize(src)/1e9:.2f} GB)")
    shutil.copy2(src, dst)
    return dst

def kaggle_download(slug: str, scratch_archives: str) -> str:
    """Download a Kaggle dataset. Requires kaggle credentials to be present."""
    try:
        import kagglehub
        _log(f"kagglehub download: {slug}")
        return kagglehub.dataset_download(slug)
    except Exception as e:
        _log(f"kagglehub failed ({e}); trying the kaggle CLI")
    dest = os.path.join(scratch_archives, slug.split("/")[-1])
    os.makedirs(dest, exist_ok=True)
    r = subprocess.run(["kaggle", "datasets", "download", "-d", slug,
                        "-p", dest, "--unzip"], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(
            f"Kaggle download of {slug} failed.\n{r.stderr}\n"
            "Set KAGGLE_USERNAME/KAGGLE_KEY, or upload kaggle.json, or switch "
            "the Config cell to the 'drive' dataset source.")
    return dest

# --------
# EuroSAT: locate the released partition and the 13-band GeoTIFFs
# --------
def _find_tif_root(root: str):
    """Directory holding the class subfolders of 13-band .tif patches."""
    best, best_n = None, 0
    for dirpath, dirnames, filenames in os.walk(root):
        tifs = [f for f in filenames if f.lower().endswith((".tif", ".tiff"))]
        if len(tifs) > best_n:
            best, best_n = dirpath, len(tifs)
    if best is None:
        raise RuntimeError(
            f"No .tif files found under {root}. The multispectral variant "
            "(EuroSATallBands) is required -- the RGB-only release cannot "
            "supply the nine-band condition.")
    # climb to the parent holding the class directories
    parent = os.path.dirname(best)
    if os.path.basename(best) in EUROSAT_CLASSES:
        return parent, best_n
    return best, best_n

def _find_split_csvs(root: str):
    found = {}
    for split, names in _SPLIT_CSVS.items():
        for name in names:
            hits = glob.glob(os.path.join(root, "**", name), recursive=True)
            if hits:
                found[split] = sorted(hits, key=len)[0]
                break
    missing = [s for s in ("train", "val", "test") if s not in found]
    if missing:
        raise RuntimeError(
            f"EuroSAT split CSVs missing: {missing}. the paper requires the "
            "dataset's RELEASED train/validation/test partition, preserved "
            "unchanged; this pipeline will not invent one. Expected "
            "train.csv / validation.csv / test.csv somewhere under "
            f"{root}.")
    return found

def _index_tifs(tif_root: str):
    idx = {}
    for dirpath, _, filenames in os.walk(tif_root):
        for f in filenames:
            if f.lower().endswith((".tif", ".tiff")):
                idx.setdefault(os.path.splitext(f)[0], os.path.join(dirpath, f))
    return idx

def _resolve(fn: str, csv_dir: str, tif_root: str, index: dict):
    cands = [os.path.join(csv_dir, fn), os.path.join(tif_root, fn)]
    stem = os.path.splitext(fn)[0]
    cands += [os.path.join(csv_dir, stem + ".tif"),
              os.path.join(tif_root, stem + ".tif")]
    for c in cands:
        if os.path.exists(c) and c.lower().endswith((".tif", ".tiff")):
            return c
    return index.get(os.path.basename(stem))

def resolve_eurosat(root: str) -> dict:
    """Locate the multispectral patches and the released split CSVs.

    The Kaggle redistribution's exact layout is not fixed across mirrors, so
    the layout is discovered and then asserted rather than hardcoded.
    """
    tif_root, n_tif = _find_tif_root(root)
    csvs = _find_split_csvs(root)
    index = _index_tifs(tif_root)
    _log(f"EuroSAT: {n_tif} GeoTIFFs under {tif_root}")

    out = {"tif_root": tif_root, "csvs": csvs, "splits": {}}
    seen = set()
    for split, path in csvs.items():
        csv_dir = os.path.dirname(path)
        files, labels, names = [], [], []
        with open(path, newline="") as fh:
            rd = csv.DictReader(fh)
            cols = {c.lower(): c for c in (rd.fieldnames or [])}
            fcol = cols.get("filename") or cols.get("file") or cols.get("path")
            lcol = cols.get("label")
            ccol = cols.get("classname") or cols.get("class")
            if not fcol or lcol is None:
                raise RuntimeError(
                    f"{path}: expected 'Filename' and 'Label' columns, found "
                    f"{rd.fieldnames}")
            for r in rd:
                fn = r[fcol]
                p = _resolve(fn, csv_dir, tif_root, index)
                if p is None:
                    raise RuntimeError(
                        f"{path}: could not locate '{fn}' under {tif_root}. "
                        "The split CSVs and the image tree do not match; "
                        "check that EuroSATallBands was extracted completely.")
                files.append(p)
                labels.append(int(r[lcol]))
                names.append(r[ccol] if ccol else "")
                seen.add(os.path.abspath(p))
        out["splits"][split] = {"files": files,
                                "labels": np.asarray(labels, dtype=np.int64),
                                "class_names": names}
    tot = sum(len(v["files"]) for v in out["splits"].values())
    assert len(seen) == tot, (
        f"EuroSAT splits overlap: {tot} rows resolve to {len(seen)} distinct "
        "files. No configuration may be evaluated on images another trained "
        "on (see the paper).")
    _log(f"EuroSAT partition: " + ", ".join(
        f"{k}={len(v['files'])}" for k, v in out["splits"].items()) +
        f" (total {tot})")
    return out

# --------
# Caches
# --------
def build_eurosat_cache(resolved: dict, cache_dir: str) -> dict:
    """Decode the 9 retained bands once into a uint16 memmap at native 64x64.

    Reading a 13-band GeoTIFF per item through rasterio dominates dataloading
    cost. The cache stores exactly the bands the model consumes, in model
    order, with no scaling applied -- so it is byte-identical to the direct
    path and the test suite asserts that.
    """
    import rasterio
    os.makedirs(cache_dir, exist_ok=True)
    out = {}
    for split, d in resolved["splits"].items():
        apath = os.path.join(cache_dir, f"eurosat_{split}_b9_u16.npy")
        lpath = os.path.join(cache_dir, f"eurosat_{split}_labels.npy")
        out[split] = (apath, lpath)
        n = len(d["files"])
        if os.path.exists(apath) and os.path.exists(lpath):
            if np.load(apath, mmap_mode="r").shape[0] == n:
                _log(f"eurosat/{split}: cache present ({n} patches)")
                continue
        np.save(lpath, d["labels"])
        arr = np.lib.format.open_memmap(
            apath, mode="w+", dtype=np.uint16,
            shape=(n, N_BANDS_CACHED, EUROSAT_NATIVE, EUROSAT_NATIVE))
        for i, fp in enumerate(d["files"]):
            with rasterio.open(fp) as src:
                if src.count < 13:
                    raise RuntimeError(
                        f"{fp} has {src.count} bands; the 13-band "
                        "EuroSATallBands product is required.")
                if src.height != EUROSAT_NATIVE or src.width != EUROSAT_NATIVE:
                    raise RuntimeError(
                        f"{fp} is {src.height}x{src.width}; EuroSAT patches "
                        f"must be {EUROSAT_NATIVE}x{EUROSAT_NATIVE}.")
                img = src.read()
            arr[i] = img[EUROSAT_SATLAS_BANDS].astype(np.uint16)
            if (i + 1) % 2500 == 0:
                _log(f"  eurosat/{split}: {i+1}/{n}")
        arr.flush()
        del arr
        _log(f"eurosat/{split}: cached {n} patches")
    return out

def build_resisc45_cache(split_csvs: dict, data_root: str, cache_dir: str,
                         enabled: bool = True) -> dict:
    """Decode RESISC45 JPEGs once into a uint8 memmap at native 256x256."""
    from PIL import Image
    os.makedirs(cache_dir, exist_ok=True)
    out = {}
    for split, csv_path in split_csvs.items():
        rows = list(csv.DictReader(open(csv_path, newline="")))
        labels = np.asarray([int(r["label"]) for r in rows], dtype=np.int64)
        lpath = os.path.join(cache_dir, f"resisc45_{split}_labels.npy")
        np.save(lpath, labels)
        if not enabled:
            out[split] = (None, lpath)
            continue
        apath = os.path.join(cache_dir, f"resisc45_{split}_u8.npy")
        out[split] = (apath, lpath)
        n = len(rows)
        if os.path.exists(apath) and np.load(apath, mmap_mode="r").shape[0] == n:
            _log(f"resisc45/{split}: cache present ({n} images)")
            continue
        arr = np.lib.format.open_memmap(
            apath, mode="w+", dtype=np.uint8,
            shape=(n, 3, RESISC_NATIVE, RESISC_NATIVE))
        for i, r in enumerate(rows):
            img = Image.open(os.path.join(data_root, r["filepath"])).convert("RGB")
            if img.size != (RESISC_NATIVE, RESISC_NATIVE):
                raise RuntimeError(
                    f"{r['filepath']} is {img.size}; RESISC45 images must be "
                    f"{RESISC_NATIVE}x{RESISC_NATIVE}.")
            arr[i] = np.asarray(img, dtype=np.uint8).transpose(2, 0, 1)
            if (i + 1) % 5000 == 0:
                _log(f"  resisc45/{split}: {i+1}/{n}")
        arr.flush()
        del arr
        _log(f"resisc45/{split}: cached {n} images")
    return out

# --------
# dataset_summary.csv -- resolves the the paper editorial note
# --------
def write_dataset_summary(rows, out_csv: str, out_md: str = None):
    """Exact per-class and per-split counts for both datasets.

    the paper carries an open editorial note: the stated class populations
    total ~27,600 whereas the canonical EuroSAT release is reported as 27,000,
    and asks that this be reconciled against the split CSVs.
    This file is that reconciliation, computed from the actual partition.
    """
    import pandas as pd
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    if out_md:
        lines = ["# Dataset summary (generated from the actual splits)", ""]
        for ds, g in df.groupby("dataset"):
            tot = int(g.n_images.sum())
            lines.append(f"## {ds} — total {tot} images")
            piv = g.pivot_table(index="class_name", columns="split",
                                values="n_images", aggfunc="sum",
                                fill_value=0)
            for c in ("train", "val", "test"):
                if c not in piv.columns:
                    piv[c] = 0
            piv = piv[["train", "val", "test"]]
            piv["total"] = piv.sum(axis=1)
            lines += ["", piv.to_markdown(), ""]
            lines.append(f"Split totals: " + ", ".join(
                f"{k}={int(v)}" for k, v in
                g.groupby('split').n_images.sum().items()))
            lines.append("")
        with open(out_md, "w", encoding="utf-8") as f:
            f.write("\n".join(lines))
    return df

def summarize_split(dataset, split, labels, class_names):
    labels = np.asarray(labels)
    rows = []
    for ci in sorted(np.unique(labels).tolist()):
        rows.append({"dataset": dataset, "split": split, "class_index": int(ci),
                     "class_name": class_names.get(int(ci), str(ci)),
                     "n_images": int((labels == ci).sum())})
    return rows

Overwriting src/staging.py


In [ ]:
%%writefile src/run_experiment.py
"""Grid driver: run ordering, resume, time budget, progress reporting.

ORDERING (the paper of the build spec). Tier order core -> factorial ->
matched. Within a tier, budget-major and ascending: every configuration and
every seed at k=5, then k=10, and so on, with `full` last. Small budgets are
far cheaper even with the step floor, so this yields most of the
label-efficiency curve early -- a grid that stops after four budgets still
answers the primary question, whereas one that stops halfway through a
configuration-major ordering answers nothing. All five seeds of a
(configuration, budget) block stay together, so paired seed tests are never
left incomplete.
"""
import os
import gc
import time

import numpy as np
import torch

from protocol import PROTOCOL
from configs import (configs_for, cfg_id, TIER_ORDER, SEEDS, K_SHOTS,
                     NUM_CLASSES, DATASETS)
from datasets import EuroSATSplit, RESISC45Split, preprocessor_for
from kshot import kshot_subset, assert_splits_intact
import models as MO
from models import build_model, enable_grad_checkpointing
from featcache import FeatureCache, extract_features
from metrics import compute_metrics
from store import (run_key, begin_run, end_run, append_result, write_text)
from utils import (set_seed, backbone_fingerprint, gpu_name, amp_dtype_for,
                   file_sha256, fmt_hms)
import train as T

class DataHandles:
    """Immutable splits for one dataset, built once and shared by every run."""

    def __init__(self, dataset, train_ds, val_ds, test_ds, train_labels,
                 split_files, class_names):
        self.dataset = dataset
        self.train = train_ds
        self.val = val_ds
        self.test = test_ds
        self.train_labels = np.asarray(train_labels)
        self.n_classes = NUM_CLASSES[dataset]
        self.class_names = class_names
        self.n_val, self.n_test = len(val_ds), len(test_ds)
        self.hashes = {k: (file_sha256(v)[:16] if v and os.path.exists(v) else "")
                       for k, v in split_files.items()}

def build_handles(dataset, cache, split_files, class_names, data_root=None):
    if dataset == "eurosat":
        ds = {s: EuroSATSplit(*cache[s]) for s in ("train", "val", "test")}
    else:
        ds = {s: RESISC45Split(split_files.get(s), data_root, *cache[s])
              for s in ("train", "val", "test")}
    return DataHandles(dataset, ds["train"], ds["val"], ds["test"],
                       ds["train"].labels, split_files, class_names)

# --------
def run_one(dataset, cfg, k, seed, handles, paths, fcache, device,
            proto=PROTOCOL, num_workers=2, verbose=False, save_curves=False):
    cid = cfg_id(cfg)
    key = run_key(dataset, cid, k, seed)
    begin_run(paths, key, {"dataset": dataset, "config_id": cid,
                           "k_shot": str(k), "seed": seed})
    t_run = time.time()

    set_seed(seed)
    model = build_model(cfg.backbone, cfg.pretraining, handles.n_classes,
                        channel_mode=cfg.channel_mode,
                        satlas_variant=cfg.satlas_variant or "ms",
                        dropout=proto.head_dropout, hidden=proto.head_hidden,
                        device=device).to(device)
    if proto.grad_checkpointing:
        enable_grad_checkpointing(model)
    bfp = backbone_fingerprint(model.backbone)
    amp_dtype = amp_dtype_for(device, proto.use_amp)

    prep_eval = preprocessor_for(dataset, cfg.channel_mode, cfg.pretraining,
                                 device, augment=False)
    prep_train = preprocessor_for(dataset, cfg.channel_mode, cfg.pretraining,
                                  device, augment=cfg.augment)

    subset, idx = kshot_subset(handles.train, handles.train_labels, k, seed,
                               handles.n_classes)
    assert_splits_intact(handles.val, handles.test, handles.n_val, handles.n_test)

    # -------- Stage 1: frozen transfer ----------------
    model.set_backbone_trainable(False)
    t0 = time.time()
    cache_hit = False
    if cfg.augment:
        # Augmented inputs differ every epoch, so Stage-1 features are not
        # cacheable. Only the aug tier takes this path.
        h1, plan1, best_val1, best_ep1 = T.run_stage1_uncached(
            model, subset, handles.val, prep_train, prep_eval, device, proto,
            seed, amp_dtype, num_workers,
            aug_generator=torch.Generator().manual_seed(seed + 11))
        t_s1 = time.time() - t0
        t0e = time.time()
        y_true1, y_pred1, y_prob1 = T.predict_images(
            model, handles.test, prep_eval, device, proto.batch_size,
            amp_dtype, num_workers)
        t_eval = time.time() - t0e
    else:
        def _feats(split_ds, split_name):
            kk = fcache.key(dataset, cid, bfp, split_name)
            return fcache.get_or_compute(kk, lambda: extract_features(
                model, split_ds, prep_eval, device, amp_dtype,
                proto.batch_size, num_workers))
        before = fcache.hits
        ftr, ytr = _feats(handles.train, "train")     # whole split, once
        fva, yva = _feats(handles.val, "val")
        fte, yte = _feats(handles.test, "test")
        cache_hit = (fcache.hits - before) == 3
        if idx is not None:
            ftr, ytr = ftr[idx], ytr[idx]
        h1, plan1, best_val1, best_ep1 = T.run_stage1(
            model, ftr, ytr, fva, yva, device, proto, seed, verbose)
        t_s1 = time.time() - t0
        t0e = time.time()
        y_true1, y_pred1, y_prob1 = T.predict_from_features(
            model, fte, yte.numpy(), proto.batch_size, device)
        t_eval = time.time() - t0e
        del ftr, fva, fte
    T.assert_probs_valid(y_prob1)
    m1 = compute_metrics(y_true1, y_pred1, y_prob1, handles.n_classes)

    # -------- Stage 2: full fine-tuning ----------------
    t0 = time.time()
    h2, plan2, best_val2, best_ep2 = T.run_stage2(
        model, subset, handles.val, prep_train, prep_eval, device, proto, seed,
        amp_dtype, num_workers, verbose)
    t_s2 = time.time() - t0
    t0e = time.time()
    y_true2, y_pred2, y_prob2 = T.predict_images(
        model, handles.test, prep_eval, device, proto.batch_size, amp_dtype,
        num_workers)
    t_eval += time.time() - t0e
    T.assert_probs_valid(y_prob2)
    m2 = compute_metrics(y_true2, y_pred2, y_prob2, handles.n_classes)

    # -------- archive ----------------
    for stage, (yt, yp, pr) in (("stage1", (y_true1, y_pred1, y_prob1)),
                                ("stage2", (y_true2, y_pred2, y_prob2))):
        np.savez_compressed(paths.pred_npz(dataset, cid, k, seed, stage),
                            y_true=yt.astype(np.int16),
                            y_pred=yp.astype(np.int16),
                            y_prob=pr.astype(np.float16),
                            n_classes=handles.n_classes)
    if save_curves:
        T.plot_history(h1, h2, os.path.join(
            paths.logs, f"curves_{dataset}_{cid}_{k}_{seed}.png"))
    if k == "full" and seed == 42:
        torch.save({"state_dict": model.state_dict(), "config": cfg._asdict()},
                   os.path.join(paths.checkpoints, f"{dataset}_{cid}_full_s42.pt"))

    row = {
        "run_key": key, "dataset": dataset, "config_id": cid,
        "backbone": cfg.backbone, "pretraining": cfg.pretraining,
        "channel_mode": cfg.channel_mode,
        "satlas_variant": cfg.satlas_variant or "", "tier": cfg.tier,
        "augment": bool(cfg.augment), "k_shot": k, "seed": seed,
        "swin_arch": MO.SWIN_ARCH if cfg.backbone == "swin_b" else "",
        **m2,
        "frozen_acc": m1["test_acc"], "frozen_macro_f1": m1["macro_f1"],
        "frozen_weighted_f1": m1["weighted_f1"], "frozen_kappa": m1["kappa"],
        "frozen_roc_auc_ovr": m1["roc_auc_ovr"], "frozen_ece": m1["ece"],
        "epochs_stage1": plan1["epochs"], "epochs_stage2": plan2["epochs"],
        "steps_stage1": plan1["total_steps"], "steps_stage2": plan2["total_steps"],
        "steps_per_epoch_s1": plan1["steps_per_epoch"],
        "steps_per_epoch_s2": plan2["steps_per_epoch"],
        "min_steps_stage1": plan1["min_steps"],
        "min_steps_stage2": plan2["min_steps"],
        "floor_engaged_s1": plan1["floor_engaged"],
        "floor_engaged_s2": plan2["floor_engaged"],
        "drop_last": plan2["drop_last"],
        "best_val_acc_stage1": best_val1, "best_val_acc_stage2": best_val2,
        "best_epoch_stage1": best_ep1, "best_epoch_stage2": best_ep2,
        "stage1_train_seconds": round(t_s1, 2),
        "stage2_train_seconds": round(t_s2, 2),
        "train_seconds": round(t_s1 + t_s2, 2),
        "eval_seconds": round(t_eval, 2),
        "total_seconds": round(time.time() - t_run, 2),
        "n_train_pool": len(handles.train), "n_train_used": len(subset),
        "n_val": handles.n_val, "n_test": handles.n_test,
        "split_hash_train": handles.hashes.get("train", ""),
        "split_hash_val": handles.hashes.get("val", ""),
        "split_hash_test": handles.hashes.get("test", ""),
        "gpu_name": gpu_name(), "protocol_fp": proto.fingerprint(),
        "backbone_fp": bfp, "feat_cache_hit": cache_hit,
        "amp_dtype": str(amp_dtype).replace("torch.", "") if amp_dtype else "off",
        "torch_version": torch.__version__,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    }
    append_result(paths.results_csv(dataset), row)
    end_run(paths, key)

    del model
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

    print(f"[done] {dataset} {cid} k={k} seed={seed} "
          f"acc={m2['test_acc']:.4f} frozen={m1['test_acc']:.4f} "
          f"e1={plan1['epochs']} e2={plan2['epochs']} "
          f"({row['total_seconds']/60:.1f} min)", flush=True)
    return row

# --------
def iter_grid(tiers, datasets, budgets=None, seeds=None):
    """Tier-major, then budget-major ascending, then dataset, config, seed."""
    budgets = budgets or K_SHOTS
    seeds = seeds or SEEDS
    for tier in [t for t in TIER_ORDER + ["aug"] if t in tiers]:
        for k in budgets:
            for ds in datasets:
                for cfg in configs_for(ds, [tier]):
                    for seed in seeds:
                        yield tier, ds, cfg, k, seed

def write_progress(paths, done, total, times, started, stopped_reason=""):
    med = float(np.median(times)) if times else float("nan")
    remaining = total - done
    eta = med * remaining if times else float("nan")
    lines = [
        "# Grid progress", "",
        f"- Runs complete: **{done} / {total}**",
        f"- Runs remaining: **{remaining}**",
        f"- Measured time per run (median of {len(times)}): "
        f"**{med/60:.2f} min**" if times else "- Measured time per run: n/a",
        f"- Projected time to finish: **{fmt_hms(eta)}**" if times
        else "- Projected time to finish: n/a",
        f"- Elapsed this session: {fmt_hms(time.time() - started)}",
        f"- GPU: {gpu_name()}",
        f"- Updated: {time.strftime('%Y-%m-%d %H:%M:%S')}",
    ]
    if stopped_reason:
        lines += ["", f"> **Stopped:** {stopped_reason}"]
    write_text(os.path.join(paths.logs, "progress.md"), "\n".join(lines) + "\n")

def run_grid(handles_by_ds, paths, resume, device, tiers=None, datasets=None,
             budgets=None, seeds=None, max_hours=None, proto=PROTOCOL,
             num_workers=2, feat_cache=None, verbose=False, save_curves=False):
    tiers = tiers or TIER_ORDER
    datasets = datasets or [d for d in DATASETS if d in handles_by_ds]
    fcache = feat_cache or FeatureCache(paths.feats)
    plan = list(iter_grid(tiers, datasets, budgets, seeds))
    total = len(plan)
    started = time.time()
    times, done, ran = [], 0, 0
    reason = ""

    for tier, ds, cfg, k, seed in plan:
        cid = cfg_id(cfg)
        if resume.is_done(ds, cid, k, seed):
            done += 1
            continue
        if max_hours is not None:
            elapsed = (time.time() - started) / 3600.0
            est = (np.median(times) / 3600.0) if times else 0.0
            if elapsed + est >= max_hours:
                reason = (f"time budget reached ({max_hours} h); "
                          f"{total - done} run(s) left. Re-run this cell to "
                          "continue -- completed runs are skipped.")
                print(f"\n[stop] {reason}", flush=True)
                break
        row = run_one(ds, cfg, k, seed, handles_by_ds[ds], paths, fcache,
                      device, proto, num_workers, verbose, save_curves)
        resume.mark(row["run_key"])
        times.append(row["total_seconds"])
        done += 1
        ran += 1
        if ran % 5 == 0 or seed == (seeds or SEEDS)[-1]:
            write_progress(paths, done, total, times, started)

    write_progress(paths, done, total, times, started, reason)
    print(f"\n{done}/{total} runs complete. "
          f"progress.md written to {paths.logs}", flush=True)
    return {"done": done, "total": total, "ran": ran, "times": times,
            "stopped": reason}

Overwriting src/run_experiment.py


In [ ]:
%%writefile src/analyze.py
"""Derived analyses (see the paper), statistical tests (see the paper) and figures.

Everything here is computed from the results CSV and the archived per-image
prediction .npz files. Nothing is ever retrained.

Two guards that fail loudly:
  * `assert_single_augment` -- rows with mixed `augment` values are never
    aggregated together. Augmentation's benefit scales inversely with training
    set size, so pooling augmented and unaugmented runs would confound the
    label-budget axis directly.
  * GPU heterogeneity -- any comparison whose two arms ran on different GPU
    types is flagged in a `gpu_mismatch` column.
"""
import os
import itertools

import numpy as np
import pandas as pd

from metrics import (mcnemar_test, holm_bonferroni, wilcoxon_across_seeds,
                     paired_ttest_across_seeds, error_overlap, per_class_f1,
                     confusion, compute_metrics, WILCOXON_MIN_P_5_SEEDS,
                     is_non_monotonic)
from store import load_results

K_ORDER = ["5", "10", "25", "50", "100", "full"]

class MixedAugmentError(AssertionError):
    pass

def assert_single_augment(df, where="aggregate"):
    if "augment" not in df.columns:
        return
    vals = set(df["augment"].astype(str).str.lower().unique())
    vals -= {"", "nan"}
    if len(vals) > 1:
        raise MixedAugmentError(
            f"refusing to {where} rows with mixed augment values {sorted(vals)}. "
            "The augmentation tier must be analysed separately from the main "
            "results.")

def _cell(df, cid, k, col="test_acc"):
    s = pd.to_numeric(
        df[(df.config_id == cid) & (df.k_shot.astype(str) == str(k))][col],
        errors="coerce").dropna()
    return (s.mean(), s.std(), len(s)) if len(s) else (np.nan, np.nan, 0)

def _gpus(df, cid, k=None):
    sel = df[df.config_id == cid]
    if k is not None:
        sel = sel[sel.k_shot.astype(str) == str(k)]
    return set(sel.get("gpu_name", pd.Series(dtype=str)).dropna().unique())

# -------- 1. Satlas - ImageNet deltas (Fig 8)
def compute_deltas(df, ds):
    assert_single_augment(df, "compute deltas over")
    rows = []
    for bb in sorted(df.backbone.unique()):
        sat = df[(df.backbone == bb) & (df.pretraining == "satlas")]
        imn = df[(df.backbone == bb) & (df.pretraining == "imagenet")]
        for scid in sorted(sat.config_id.unique()):
            ch = sat[sat.config_id == scid].channel_mode.iloc[0]
            match = imn[imn.channel_mode == ch]      # input held constant
            if not len(match):
                continue
            icid = match.config_id.iloc[0]
            for k in K_ORDER:
                sm, ss, ns = _cell(df, scid, k)
                im, isd, ni = _cell(df, icid, k)
                if ns == 0 or ni == 0:
                    continue
                ga, gb = _gpus(df, scid, k), _gpus(df, icid, k)
                rows.append({
                    "dataset": ds, "backbone": bb, "channel_mode": ch,
                    "satlas_config": scid, "imagenet_config": icid,
                    "k_shot": k, "n_seeds_satlas": ns, "n_seeds_imagenet": ni,
                    "satlas_acc": sm, "satlas_std": ss,
                    "imagenet_acc": im, "imagenet_std": isd,
                    "delta": sm - im,
                    "gpu_mismatch": bool(ga and gb and ga != gb),
                    "gpus": "|".join(sorted(ga | gb)),
                })
    return pd.DataFrame(rows)

# -------- 2. EuroSAT 2x2 decomposition (Fig 9)
def factorial_decomposition(df):
    """E_source, E_spectral and the interaction, per backbone and budget."""
    assert_single_augment(df, "decompose")
    rows = []
    for bb in sorted(df.backbone.unique()):
        for k in K_ORDER:
            cells, ok = {}, True
            for src, ch in itertools.product(["satlas", "imagenet"],
                                             ["ms9", "rgb3"]):
                sel = df[(df.backbone == bb) & (df.pretraining == src)
                         & (df.channel_mode == ch)]
                if not len(sel):
                    ok = False
                    break
                cells[(src, ch)] = _cell(df, sel.config_id.iloc[0], k)[0]
            if not ok or any(np.isnan(v) for v in cells.values()):
                continue
            s9, s3 = cells[("satlas", "ms9")], cells[("satlas", "rgb3")]
            i9, i3 = cells[("imagenet", "ms9")], cells[("imagenet", "rgb3")]
            rows.append({
                "backbone": bb, "k_shot": k,
                "satlas_ms9": s9, "satlas_rgb3": s3,
                "imagenet_ms9": i9, "imagenet_rgb3": i3,
                "effect_source": ((s9 + s3) - (i9 + i3)) / 2,
                "effect_spectral": ((s9 + i9) - (s3 + i3)) / 2,
                "interaction": (s9 - s3) - (i9 - i3),
            })
    return pd.DataFrame(rows)

# -------- 3. Label efficiency (3.9.3)
def label_efficiency(df, ds, target=0.95):
    """Labels per class at which a configuration first reaches `target` of its
    OWN full-data accuracy, by linear interpolation on log10(k).

    Non-monotonic curves are flagged rather than silently interpolated: with
    five seeds at k=5 the curve can dip, and an interpolation across a dip is
    not a label-efficiency estimate.
    """
    assert_single_augment(df, "compute label efficiency over")
    ks = [5, 10, 25, 50, 100]
    rows = []
    for cid in sorted(df.config_id.unique()):
        full = _cell(df, cid, "full")[0]
        if np.isnan(full):
            continue
        thr = target * full
        curve = [(k, _cell(df, cid, k)[0]) for k in ks]
        curve = [(k, a) for k, a in curve if not np.isnan(a)]
        if not curve:
            continue
        accs = [a for _, a in curve]
        need, how = np.nan, "not_reached"
        if curve[0][1] >= thr:
            need, how = float(curve[0][0]), "reached_at_first_budget"
        else:
            for (k0, a0), (k1, a1) in zip(curve, curve[1:]):
                if a0 < thr <= a1:
                    f = (thr - a0) / (a1 - a0)
                    need = 10 ** (np.log10(k0) + f * (np.log10(k1) - np.log10(k0)))
                    how = "interpolated"
                    break
        rows.append({
            "dataset": ds, "config_id": cid, "full_acc": full,
            "target_acc": thr, "labels_per_class_to_95pct": need,
            "reached_within_ladder": bool(not np.isnan(need)),
            "method": how,
            "non_monotonic": bool(is_non_monotonic(accs + [full])),
            "curve": ";".join(f"{k}:{a:.4f}" for k, a in curve),
        })
    return pd.DataFrame(rows)

# -------- 4. Frozen vs fine-tuned (3.9.4)
def frozen_vs_finetuned(df, ds):
    assert_single_augment(df, "compare stages over")
    if "frozen_acc" not in df.columns or df.frozen_acc.isna().all():
        return pd.DataFrame()
    rows = []
    for cid in sorted(df.config_id.unique()):
        for k in K_ORDER:
            fz = _cell(df, cid, k, "frozen_acc")
            ft = _cell(df, cid, k, "test_acc")
            if fz[2] == 0:
                continue
            rows.append({"dataset": ds, "config_id": cid, "k_shot": k,
                         "n_seeds": fz[2],
                         "frozen_acc": fz[0], "frozen_std": fz[1],
                         "finetuned_acc": ft[0], "finetuned_std": ft[1],
                         "finetuning_gain": ft[0] - fz[0]})
    return pd.DataFrame(rows)

# -------- 5. McNemar + Holm (3.8)
def run_mcnemar(paths, datasets=("eurosat", "resisc45"), seed=42,
                stage="stage2", budgets=None):
    """Pairwise McNemar on identical test images, per budget.

    Holm is applied WITHIN each (dataset, budget) family rather than across the
    entire grid at once: the budgets address separate questions, so correcting
    over every pair at every budget simultaneously would be needlessly
    conservative (see the paper).
    """
    budgets = budgets or K_ORDER
    rows = []
    for ds in datasets:
        df = load_results(paths.results_csv(ds))
        if df is None:
            continue
        for k in budgets:
            cids = sorted(df[df.k_shot.astype(str) == str(k)].config_id.unique())
            for a, b in itertools.combinations(cids, 2):
                fa = paths.pred_npz(ds, a, k, seed, stage)
                fb = paths.pred_npz(ds, b, k, seed, stage)
                if not (os.path.exists(fa) and os.path.exists(fb)):
                    continue
                da, db = np.load(fa), np.load(fb)
                if len(da["y_true"]) != len(db["y_true"]):
                    raise AssertionError(
                        f"{ds}: {a} and {b} produced different numbers of test "
                        f"predictions ({len(da['y_true'])} vs "
                        f"{len(db['y_true'])}) -- the test split is not "
                        "identical across configurations")
                if not np.array_equal(da["y_true"], db["y_true"]):
                    raise AssertionError(
                        f"{ds}: {a} and {b} disagree on test labels/order")
                p, ndisc, nb, nc = mcnemar_test(da["y_true"], da["y_pred"],
                                                db["y_pred"])
                ov = error_overlap(da["y_true"], da["y_pred"], db["y_pred"])
                ga, gb = _gpus(df, a, k), _gpus(df, b, k)
                rows.append({"dataset": ds, "k_shot": k, "stage": stage,
                             "seed": seed, "config_a": a, "config_b": b,
                             "p_raw": p, "n_discordant": ndisc,
                             "b_only": nb, "c_only": nc,
                             "exact_used": bool(ndisc < 25), **ov,
                             "gpu_mismatch": bool(ga and gb and ga != gb)})
    out = pd.DataFrame(rows)
    if len(out):
        out["p_holm"] = np.nan
        for (ds, k), grp in out.groupby(["dataset", "k_shot"]):
            m = grp.p_raw.notna()
            if m.any():
                out.loc[grp.index[m], "p_holm"] = holm_bonferroni(
                    grp.loc[m, "p_raw"].values)
        out["significant_holm_05"] = out.p_holm < 0.05
    return out

# -------- 6. Seed-level tests (3.8)
def seed_tests(df, ds):
    """Wilcoxon signed-rank AND the paired t-test across the five seeds.

    The exact signed-rank floor at n=5 is emitted as a column so no reader can
    mistake p = 0.0625 for a null result.
    """
    assert_single_augment(df, "run seed tests over")
    rows = []
    cids = sorted(df.config_id.unique())
    for a, b in itertools.combinations(cids, 2):
        for k in K_ORDER:
            sa = df[(df.config_id == a) & (df.k_shot.astype(str) == k)] \
                .sort_values("seed").test_acc.values
            sb = df[(df.config_id == b) & (df.k_shot.astype(str) == k)] \
                .sort_values("seed").test_acc.values
            if len(sa) != len(sb) or len(sa) < 5:
                continue
            s, p = wilcoxon_across_seeds(sa, sb)
            t, pt = paired_ttest_across_seeds(sa, sb)
            ga, gb = _gpus(df, a, k), _gpus(df, b, k)
            rows.append({
                "dataset": ds, "config_a": a, "config_b": b, "k_shot": k,
                "n_seeds": len(sa),
                "mean_a": float(np.mean(sa)), "mean_b": float(np.mean(sb)),
                "mean_diff": float(np.mean(sa) - np.mean(sb)),
                "wilcoxon_stat": s, "p_wilcoxon": p,
                "wilcoxon_p_floor": WILCOXON_MIN_P_5_SEEDS,
                "wilcoxon_at_floor": bool(
                    not np.isnan(p) and abs(p - WILCOXON_MIN_P_5_SEEDS) < 1e-9),
                "ttest_stat": t, "p_ttest": pt,
                "gpu_mismatch": bool(ga and gb and ga != gb),
            })
    out = pd.DataFrame(rows)
    # Holm WITHIN each budget, not across the whole grid.
    for col, adj in (("p_wilcoxon", "p_wilcoxon_holm"),
                     ("p_ttest", "p_ttest_holm")):
        if len(out) and out[col].notna().any():
            out[adj] = np.nan
            for k, grp in out.groupby("k_shot"):
                m = grp[col].notna()
                if m.any():
                    out.loc[grp.index[m], adj] = holm_bonferroni(
                        grp.loc[m, col].values)
    return out

# -------- 7. Per-class F1 and confusion, from the archive --
def per_class_and_confusion(paths, datasets=("eurosat", "resisc45"),
                            class_names=None, stages=("stage1", "stage2")):
    """Derived from the archived predictions; nothing is retrained."""
    pc_rows, cms = [], {}
    for ds in datasets:
        df = load_results(paths.results_csv(ds))
        if df is None:
            continue
        names = (class_names or {}).get(ds, {})
        for _, r in df.iterrows():
            for stage in stages:
                f = paths.pred_npz(ds, r.config_id, r.k_shot, r.seed, stage)
                if not os.path.exists(f):
                    continue
                d = np.load(f)
                nc = int(d["n_classes"]) if "n_classes" in d else \
                    int(max(d["y_true"].max(), d["y_pred"].max()) + 1)
                f1s = per_class_f1(d["y_true"], d["y_pred"], nc)
                for ci, v in enumerate(f1s):
                    pc_rows.append({
                        "dataset": ds, "config_id": r.config_id,
                        "k_shot": r.k_shot, "seed": r.seed, "stage": stage,
                        "class_index": ci,
                        "class_name": names.get(ci, str(ci)), "f1": float(v)})
                key = (ds, r.config_id, str(r.k_shot), stage)
                cm = confusion(d["y_true"], d["y_pred"], nc)
                cms[key] = cms.get(key, np.zeros_like(cm)) + cm
    return pd.DataFrame(pc_rows), cms

# -------- figures
def figure_deltas(deltas, out_png):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    if not len(deltas):
        return None
    dss = sorted(deltas.dataset.unique())
    fig, axes = plt.subplots(1, len(dss), figsize=(7 * len(dss), 4.5),
                             squeeze=False)
    for ax, ds in zip(axes[0], dss):
        sub = deltas[deltas.dataset == ds]
        for (bb, ch, sc), g in sub.groupby(["backbone", "channel_mode",
                                            "satlas_config"]):
            g = g.set_index("k_shot").reindex(K_ORDER).dropna(subset=["delta"])
            variant = sc.split("_")[1] if "_" in sc else sc
            ax.plot(range(len(g)), g.delta.values, marker="o",
                    label=f"{bb} / {ch} / {variant}")
            ax.set_xticks(range(len(g)))
            ax.set_xticklabels(g.index)
        ax.axhline(0, c="grey", lw=1, ls="--")
        ax.set_title(f"{ds}: SatlasPretrain - ImageNet")
        ax.set_xlabel("labels per class")
        ax.set_ylabel("accuracy delta")
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out_png

def figure_factorial(fac, out_png):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    if not len(fac):
        return None
    bbs = sorted(fac.backbone.unique())
    fig, axes = plt.subplots(1, len(bbs), figsize=(7 * len(bbs), 4.5),
                             squeeze=False)
    for ax, bb in zip(axes[0], bbs):
        g = fac[fac.backbone == bb].set_index("k_shot").reindex(K_ORDER).dropna(
            subset=["effect_source"])
        x = range(len(g))
        for col, lab in (("effect_source", "source"),
                         ("effect_spectral", "spectral"),
                         ("interaction", "interaction")):
            ax.plot(x, g[col].values, marker="o", label=lab)
        ax.axhline(0, c="grey", lw=1, ls="--")
        ax.set_xticks(list(x)); ax.set_xticklabels(g.index)
        ax.set_title(f"EuroSAT 2x2 — {bb}")
        ax.set_xlabel("labels per class"); ax.set_ylabel("effect")
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out_png

# -------- main
def main(paths, class_names=None, augment=False, quiet=False):
    """Run every analysis the available results support and write to disk."""
    outdir = paths.results
    made = []

    def dump(frames, name, title):
        frames = [f for f in frames if f is not None and len(f)]
        if not frames:
            return None
        out = pd.concat(frames, ignore_index=True)
        p = os.path.join(outdir, f"{name}.csv")
        out.to_csv(p, index=False)
        made.append(p)
        if not quiet:
            print(f"\n=== {title} ===")
            print(out.to_string(index=False)[:4000])
        return out

    deltas, effs, fvf, seeds = [], [], [], []
    for ds in ("eurosat", "resisc45"):
        df = load_results(paths.results_csv(ds))
        if df is None:
            print(f"[skip] no results for {ds}")
            continue
        df = df[df.get("augment", pd.Series(False, index=df.index))
                .astype(str).str.lower().isin(["true" if augment else "false"])]
        if not len(df):
            continue
        deltas.append(compute_deltas(df, ds))
        effs.append(label_efficiency(df, ds))
        fvf.append(frozen_vs_finetuned(df, ds))
        seeds.append(seed_tests(df, ds))
        if ds == "eurosat":
            fac = factorial_decomposition(df)
            if len(fac):
                p = os.path.join(outdir, "factorial_eurosat.csv")
                fac.to_csv(p, index=False); made.append(p)
                figure_factorial(fac, os.path.join(outdir, "fig9_factorial.png"))
                if not quiet:
                    print("\n=== EuroSAT 2x2 factorial ===")
                    print(fac.to_string(index=False))

    d = dump(deltas, "synthesis_deltas", "Satlas - ImageNet deltas (Fig. 8)")
    if d is not None:
        figure_deltas(d, os.path.join(outdir, "fig8_deltas.png"))
    dump(effs, "label_efficiency", "Labels per class to reach 95% of full-data")
    dump(fvf, "frozen_vs_finetuned", "Frozen vs fine-tuned")
    dump(seeds, "seed_tests", "Seed-level Wilcoxon + paired t (Holm per budget)")

    mc = run_mcnemar(paths)
    if len(mc):
        p = os.path.join(outdir, "mcnemar_pvalues.csv")
        mc.to_csv(p, index=False); made.append(p)
        if not quiet:
            print("\n=== McNemar (raw + Holm within budget) + error overlap ===")
            print(mc.to_string(index=False)[:4000])

    pc, cms = per_class_and_confusion(paths, class_names=class_names)
    if len(pc):
        p = os.path.join(outdir, "per_class_f1.csv")
        pc.to_csv(p, index=False); made.append(p)
        cmdir = os.path.join(outdir, "confusion")
        os.makedirs(cmdir, exist_ok=True)
        for (ds, cid, k, stage), cm in cms.items():
            np.savetxt(os.path.join(cmdir, f"{ds}_{cid}_{k}_{stage}.csv"),
                       cm, fmt="%d", delimiter=",")
        made.append(cmdir)

    if any(len(f) and f.get("gpu_mismatch", pd.Series(False)).any()
           for f in [mc] if len(f)):
        print("\n[warn] some comparisons have arms that ran on different GPU "
              "types; see the gpu_mismatch column.")
    print(f"\nWrote {len(made)} artefact(s) to {outdir}")
    return made

Overwriting src/analyze.py


In [ ]:
%%writefile src/preflight.py
"""GPU preflight: capability report, memory proof, throughput, determinism.

Run before any real work. It answers four questions:
  1. What GPU did Colab actually hand out, and does it support bf16?
  2. Does the heaviest configuration (Swin, nine channels, batch 32, 224x224)
     fit -- forward AND backward?
  3. How fast is each backbone, and how long will the selected grid take?
  4. Does the same tiny job run twice give identical predictions?

Batch size is NOT adjusted here. It is frozen at 32 and must not become a
function of which GPU was assigned. If memory is tight the answer is gradient
checkpointing, which leaves the computation mathematically identical and is
decided once, grid-wide.
"""
import time

import numpy as np
import torch
import torch.nn as nn

from protocol import PROTOCOL, plan_stage, BATCH_SIZE
from configs import configs_for, K_SHOTS, SEEDS, NUM_CLASSES
# NB: models.SWIN_ARCH is read through the module, never imported by value --
# the Config cell may rebind it after this module is imported.
import models as MO
from models import build_model, enable_grad_checkpointing, swin_variant_of
from utils import set_seed, gpu_name, amp_dtype_for, get_device

def gpu_report():
    dev = get_device()
    rep = {"device": str(dev), "gpu": gpu_name()}
    if dev.type == "cuda":
        free, total = torch.cuda.mem_get_info()
        rep["vram_total_gb"] = round(total / 1e9, 2)
        rep["vram_free_gb"] = round(free / 1e9, 2)
        cap = torch.cuda.get_device_capability(0)
        rep["cuda_capability"] = f"{cap[0]}.{cap[1]}"
        try:
            rep["bf16_supported"] = bool(torch.cuda.is_bf16_supported())
        except Exception:
            rep["bf16_supported"] = False
    else:
        rep["warning"] = ("NO GPU DETECTED. Select Runtime > Change runtime "
                          "type > GPU before running the grid.")
    return rep

def _one_train_step(model, x, y, opt, crit, amp_dtype, device):
    opt.zero_grad(set_to_none=True)
    ctx = (torch.autocast(device_type="cuda", dtype=amp_dtype)
           if amp_dtype is not None and device.type == "cuda"
           else torch.autocast(device_type="cpu", enabled=False))
    with ctx:
        loss = crit(model(x), y)
    loss.backward()
    opt.step()
    return float(loss.item())

def fit_check(backbone="swin_b", channel_mode="ms9", n_classes=10,
              proto=PROTOCOL, grad_checkpointing=False):
    """One forward and backward for the heaviest configuration.

    Uses a scratch initialisation, which is architecturally identical to the
    pretrained arm of the same backbone, so no checkpoint download is needed to
    prove the memory footprint.
    """
    device = get_device()
    set_seed(0)
    model = build_model(backbone, "scratch", n_classes,
                        channel_mode=channel_mode, device=device).to(device)
    if grad_checkpointing:
        enable_grad_checkpointing(model)
    if proto.channels_last and backbone == "resnet50" and device.type == "cuda":
        model.to(memory_format=torch.channels_last)
    ch = 9 if channel_mode == "ms9" else 3
    x = torch.randn(BATCH_SIZE, ch, proto.img_size, proto.img_size,
                    device=device)
    if proto.channels_last and backbone == "resnet50" and device.type == "cuda":
        x = x.contiguous(memory_format=torch.channels_last)
    y = torch.randint(0, n_classes, (BATCH_SIZE,), device=device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    crit = nn.CrossEntropyLoss()
    amp = amp_dtype_for(device, proto.use_amp)
    try:
        _one_train_step(model, x, y, opt, crit, amp, device)
    except torch.cuda.OutOfMemoryError as e:
        raise RuntimeError(
            f"{backbone}/{channel_mode} at batch {BATCH_SIZE} and "
            f"{proto.img_size}x{proto.img_size} does not fit on "
            f"{gpu_name()}.\nBatch size is frozen at 32 and must not be "
            "reduced. Set GRAD_CHECKPOINTING = True in the Config cell "
            "(mathematically identical, ~30% slower) and re-run preflight."
        ) from e
    peak = (torch.cuda.max_memory_allocated() / 1e9
            if device.type == "cuda" else float("nan"))
    del model, opt, x, y
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    return {"backbone": backbone, "channel_mode": channel_mode,
            "peak_vram_gb": round(peak, 2), "ok": True}

def time_backbone(backbone, channel_mode, n_classes=10, n_steps=20,
                  proto=PROTOCOL, grad_checkpointing=False):
    """Time n_steps of training and one inference pass; return images/second."""
    device = get_device()
    set_seed(0)
    model = build_model(backbone, "scratch", n_classes,
                        channel_mode=channel_mode, device=device).to(device)
    if grad_checkpointing:
        enable_grad_checkpointing(model)
    if proto.channels_last and backbone == "resnet50" and device.type == "cuda":
        model.to(memory_format=torch.channels_last)
    ch = 9 if channel_mode == "ms9" else 3
    x = torch.randn(BATCH_SIZE, ch, proto.img_size, proto.img_size, device=device)
    if proto.channels_last and backbone == "resnet50" and device.type == "cuda":
        x = x.contiguous(memory_format=torch.channels_last)
    y = torch.randint(0, n_classes, (BATCH_SIZE,), device=device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    crit = nn.CrossEntropyLoss()
    amp = amp_dtype_for(device, proto.use_amp)

    model.train()
    for _ in range(3):                                   # warm up
        _one_train_step(model, x, y, opt, crit, amp, device)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_steps):
        _one_train_step(model, x, y, opt, crit, amp, device)
    if device.type == "cuda":
        torch.cuda.synchronize()
    train_s = (time.time() - t0) / n_steps

    model.eval()
    with torch.no_grad():
        for _ in range(3):
            model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n_steps):
            model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
    infer_s = (time.time() - t0) / n_steps

    del model, opt, x, y
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return {"backbone": backbone, "channel_mode": channel_mode,
            "train_s_per_step": train_s, "infer_s_per_batch": infer_s,
            "train_img_per_s": BATCH_SIZE / train_s,
            "infer_img_per_s": BATCH_SIZE / infer_s}

def determinism_check(n_classes=4):
    """The same tiny job twice must give identical predictions."""
    device = get_device()
    outs = []
    for _ in range(2):
        set_seed(1234)
        model = build_model("resnet50", "scratch", n_classes,
                            channel_mode="rgb3", device=device).to(device)
        x = torch.randn(8, 3, 64, 64, device=device)
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
        crit = nn.CrossEntropyLoss()
        y = torch.randint(0, n_classes, (8,), device=device)
        model.train()
        for _ in range(3):
            opt.zero_grad(set_to_none=True)
            loss = crit(model(x), y)
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            outs.append(model(x).float().cpu().numpy())
        del model, opt
    same_pred = bool(np.array_equal(outs[0].argmax(1), outs[1].argmax(1)))
    max_abs = float(np.abs(outs[0] - outs[1]).max())
    if not same_pred:
        raise AssertionError(
            "Determinism check FAILED: two identical runs gave different "
            "predictions. the paper guarantees bit-for-bit reproducibility.")
    return {"identical_predictions": same_pred, "max_abs_logit_diff": max_abs}

def swin_alignment_check(device=None):
    """Assert the ImageNet/scratch Swin matches SatlasPretrain's Swin.

    SatlasPretrain's SwinBackbone is built on torchvision swin_v2_b. Pairing it
    against swin_b (V1) would put a V1/V2 architecture difference inside every
    Satlas-vs-ImageNet Swin comparison, which the paper says is controlled.
    """
    from models import load_satlas_backbone, _build_tv_swin
    arch = MO.SWIN_ARCH
    ours = swin_variant_of(_build_tv_swin(arch, pretrained=False))
    try:
        sat = load_satlas_backbone("Sentinel2_SwinB_SI_RGB", device=device)
    except Exception as e:
        return {"ours": ours, "satlas": "unavailable", "aligned": None,
                "note": f"could not load a Satlas checkpoint to compare: {e}"}
    theirs = swin_variant_of(sat)
    del sat
    aligned = (ours == theirs)
    if not aligned:
        raise AssertionError(
            f"Swin architecture mismatch: this run builds {arch} "
            f"({ours}) for the ImageNet/scratch arms, but SatlasPretrain's "
            f"checkpoints are {theirs}. Every Satlas-vs-ImageNet Swin "
            "comparison would be confounded by architecture. Set "
            "SWIN_ARCH accordingly in the Config cell.")
    return {"ours": ours, "satlas": theirs, "aligned": aligned, "arch": arch}

def project_grid_time(timings, sizes, tiers, datasets, proto=PROTOCOL,
                      budgets=None, seeds=None):
    """Project total grid wall-clock from measured per-step timings.

    Accounts for: Stage-2 training steps, Stage-2 validation passes (the
    validation split is never subsampled), test inference at both stages, and
    one Stage-1 feature-extraction pass per configuration -- which is amortised
    across all 30 runs that share a backbone fingerprint.
    """
    budgets = budgets or K_SHOTS
    seeds = seeds or SEEDS
    tt = {(t["backbone"], t["channel_mode"]): t for t in timings}

    def pick(bb, ch):
        return tt.get((bb, ch)) or tt.get((bb, "ms9")) or tt.get((bb, "rgb3"))

    total, per_ds = 0.0, {}
    for ds in datasets:
        n_tr, n_va, n_te = sizes[ds]["train"], sizes[ds]["val"], sizes[ds]["test"]
        sub = 0.0
        for cfg in configs_for(ds, tiers):
            t = pick(cfg.backbone, cfg.channel_mode)
            if t is None:
                continue
            ipb = t["infer_s_per_batch"] / BATCH_SIZE
            # one feature-extraction pass per configuration (pretrained) or
            # per seed (scratch)
            n_fx = len(seeds) if cfg.pretraining == "scratch" else 1
            sub += n_fx * (n_tr + n_va + n_te) * ipb
            for k in budgets:
                n = n_tr if k == "full" else min(
                    n_tr, int(k) * NUM_CLASSES[ds])
                p2 = plan_stage(n, 2, proto)
                per_run = (p2["total_steps"] * t["train_s_per_step"]
                           + len(p2["val_epochs"]) * n_va * ipb
                           + n_te * ipb)
                sub += per_run * len(seeds)
        per_ds[ds] = sub
        total += sub
    return {"seconds": total, "hours": total / 3600.0, "per_dataset": per_ds}

def preflight(sizes=None, tiers=("core", "factorial", "matched"),
              datasets=("eurosat", "resisc45"), proto=PROTOCOL,
              check_satlas=True, n_steps=20):
    print("=" * 62)
    print("GPU PREFLIGHT")
    print("=" * 62)
    rep = gpu_report()
    for k, v in rep.items():
        print(f"  {k:20s}: {v}")
    if get_device().type != "cuda":
        print("\n  Preflight continues on CPU, but the grid needs a GPU.")

    print("\n-- memory: heaviest configuration --")
    heavy = fit_check("swin_b", "ms9", proto=proto,
                      grad_checkpointing=proto.grad_checkpointing)
    print(f"  Swin/9-band, batch {BATCH_SIZE} @ {proto.img_size}: OK "
          f"(peak {heavy['peak_vram_gb']} GB"
          f"{', grad checkpointing ON' if proto.grad_checkpointing else ''})")
    fit_check("resnet50", "ms9", proto=proto,
              grad_checkpointing=proto.grad_checkpointing)
    print(f"  ResNet-50/9-band, batch {BATCH_SIZE}: OK")

    print(f"\n-- throughput ({n_steps} steps per backbone) --")
    timings = []
    for bb, ch in (("resnet50", "ms9"), ("swin_b", "ms9")):
        t = time_backbone(bb, ch, n_steps=n_steps, proto=proto,
                          grad_checkpointing=proto.grad_checkpointing)
        timings.append(t)
        print(f"  {bb:9s}: train {t['train_img_per_s']:7.1f} img/s   "
              f"infer {t['infer_img_per_s']:7.1f} img/s")

    print("\n-- determinism --")
    d = determinism_check()
    print(f"  identical predictions across two identical runs: "
          f"{d['identical_predictions']} (max |logit diff| "
          f"{d['max_abs_logit_diff']:.2e})")

    align = None
    if check_satlas:
        print("\n-- architecture control --")
        align = swin_alignment_check(get_device())
        if align.get("aligned"):
            print(f"  Swin arms aligned: ours={align['ours']}, "
                  f"Satlas={align['satlas']} ({align['arch']})")
        else:
            print(f"  {align.get('note', align)}")

    proj = None
    if sizes:
        proj = project_grid_time(timings, sizes, tiers, datasets, proto)
        print("\n-- projected grid time --")
        for ds, s in proj["per_dataset"].items():
            print(f"  {ds:9s}: {s/3600:6.2f} h")
        print(f"  TOTAL    : {proj['hours']:6.2f} h "
              f"(tiers={list(tiers)})")
        print("  Note: a projection from synthetic-input timings. The pilot "
              "cell measures the real thing.")
    print("=" * 62)
    return {"gpu": rep, "fit": heavy, "timings": timings, "determinism": d,
            "swin_alignment": align, "projection": proj}

Overwriting src/preflight.py


In [ ]:
%%writefile src/tests_fast.py
"""Fast correctness suite on tiny synthetic data. CPU-only, well under 2 min.

If anything here fails, the notebook stops before spending GPU time. Each test
checks a commitment the methodology makes, not merely that the code runs.
"""
import os
import csv
import shutil
import tempfile
import dataclasses

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, Subset

import protocol as P
import metrics as M
import kshot as KS
import datasets as D
import models as MO
import train as T
import store as ST
import analyze as AN
from featcache import extract_features

_RESULTS = []

def _check(name, fn):
    try:
        fn()
        _RESULTS.append((name, True, ""))
        print(f"  PASS  {name}")
    except Exception as e:
        _RESULTS.append((name, False, f"{type(e).__name__}: {e}"))
        print(f"  FAIL  {name}\n          {type(e).__name__}: {e}")

# --------
# 1. Normalization contracts against the protocol table
# --------
def test_normalization_table_iii():
    dev = torch.device("cpu")
    # visible DN 2000 -> 0.5 ; visible DN 8000 -> clipped to 1.0
    # non-visible DN 4080 -> 0.5 ; non-visible DN 9000 -> clipped to 1.0
    raw = torch.zeros(1, 9, 4, 4, dtype=torch.int32)
    raw[:, :3] = 2000
    raw[:, 3:] = 4080
    sat9 = D.GPUPreprocessor("eurosat", "ms9", False, dev, img_size=4)
    x = sat9.normalize(raw)
    assert torch.allclose(x[:, :3], torch.full_like(x[:, :3], 0.5)), "visible /4000"
    assert torch.allclose(x[:, 3:], torch.full_like(x[:, 3:], 0.5)), "non-visible /8160"

    raw2 = torch.zeros(1, 9, 4, 4, dtype=torch.int32)
    raw2[:, :3] = 8000
    raw2[:, 3:] = 90000
    x2 = sat9.normalize(raw2)
    assert torch.allclose(x2, torch.ones_like(x2)), "clip to unit interval"

    # ImageNet/scratch 9-band: ImageNet mu,sigma on the visible three and the
    # scalar mean (0.449, 0.226) on the six non-visible.
    imn9 = D.GPUPreprocessor("eurosat", "ms9", True, dev, img_size=4)
    y = imn9.normalize(raw)
    for c, (mu, sd) in enumerate(zip(D.IMAGENET_MEAN, D.IMAGENET_STD)):
        assert abs(float(y[0, c, 0, 0]) - (0.5 - mu) / sd) < 1e-5, f"ch{c}"
    exp = (0.5 - D.NONVIS_MEAN) / D.NONVIS_STD
    assert abs(float(y[0, 5, 0, 0]) - exp) < 1e-5, "non-visible standardization"
    assert abs(D.NONVIS_MEAN - 0.449) < 1e-3 and abs(D.NONVIS_STD - 0.226) < 1e-3

    # 3-band uses the SAME visible scaling as the 9-band visible channels.
    sat3 = D.GPUPreprocessor("eurosat", "rgb3", False, dev, img_size=4)
    z = sat3.normalize(raw)
    assert torch.allclose(z, x[:, :3]), "3-band visible scaling must match 9-band"

    # RESISC45: value/255, Satlas unstandardized, ImageNet/scratch standardized.
    r = torch.full((1, 3, 4, 4), 255, dtype=torch.int32)
    rs = D.GPUPreprocessor("resisc45", "rgb3", False, dev, img_size=4)
    assert torch.allclose(rs.normalize(r), torch.ones(1, 3, 4, 4))
    ri = D.GPUPreprocessor("resisc45", "rgb3", True, dev, img_size=4)
    v = ri.normalize(r)
    for c, (mu, sd) in enumerate(zip(D.IMAGENET_MEAN, D.IMAGENET_STD)):
        assert abs(float(v[0, c, 0, 0]) - (1.0 - mu) / sd) < 1e-5

    # the protocol table routing: Satlas -> no standardization; imagenet AND scratch -> yes.
    assert D.preprocessor_for("eurosat", "ms9", "satlas", dev).standardize is False
    assert D.preprocessor_for("eurosat", "ms9", "imagenet", dev).standardize is True
    assert D.preprocessor_for("eurosat", "ms9", "scratch", dev).standardize is True

def test_band_order():
    assert D.EUROSAT_SATLAS_BANDS == [3, 2, 1, 4, 5, 6, 7, 11, 12]
    names = ["B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B9",
             "B10", "B11", "B12"]
    got = [names[i] for i in D.EUROSAT_SATLAS_BANDS]
    assert got == ["B4", "B3", "B2", "B5", "B6", "B7", "B8", "B11", "B12"], got
    excluded = sorted(set(range(13)) - set(D.EUROSAT_SATLAS_BANDS))
    assert [names[i] for i in excluded] == ["B1", "B8A", "B9", "B10"]

def test_clip_before_resize():
    """Clip does not commute with interpolation, so it must happen first."""
    dev = torch.device("cpu")
    raw = torch.zeros(1, 3, 8, 8, dtype=torch.int32)
    raw[:, :, :4] = 40000          # far above the clip point
    raw[:, :, 4:] = 0
    prep = D.GPUPreprocessor("eurosat", "rgb3", False, dev, img_size=16)
    correct = prep.resize(prep.normalize(raw))
    wrong = torch.clamp(
        prep.resize(raw.float()) / D.VISIBLE_GAIN, 0.0, 1.0)
    assert correct.max() <= 1.0 + 1e-6
    assert not torch.allclose(correct, wrong), (
        "clip-then-resize must differ from resize-then-clip; if these agree "
        "the test input is degenerate")

# --------
# 2. Channel inflation
# --------
def test_channel_inflation_preserves_magnitude():
    torch.manual_seed(0)
    conv = nn.Conv2d(3, 8, 3, padding=1, bias=False)
    inflated = MO.adapt_first_layer(conv, 9)
    assert inflated.in_channels == 9
    # weights cycle R,G,B and are rescaled by 3/9
    for i in range(9):
        assert torch.allclose(inflated.weight[:, i],
                              conv.weight[:, i % 3] * (3.0 / 9.0)), i
    # An input whose 9 channels cycle the same RGB content must give exactly
    # the 3-channel pre-activation: that is what "preserves the expected
    # pre-activation magnitude" means.
    rgb = torch.randn(2, 3, 16, 16)
    x9 = torch.cat([rgb, rgb, rgb], dim=1)
    assert torch.allclose(inflated(x9), conv(rgb), atol=1e-5)
    # And the ratio is 3/in_ch, not 1.
    naive = torch.cat([conv.weight[:, i % 3] for i in range(9)], 0)
    assert abs(float(inflated.weight.abs().sum() /
                     (conv.weight.abs().sum() * 3)) - (3.0 / 9.0)) < 1e-5
    assert MO.adapt_first_layer(conv, 3) is conv, "3->3 must be a no-op"

# --------
# 3. Feature-map geometry at 224
# --------
def test_final_feature_maps_are_7x7():
    import torchvision.models as tv
    x = torch.randn(1, 3, 224, 224)
    r = tv.resnet50(weights=None).eval()
    with torch.no_grad():
        h = r.conv1(x); h = r.bn1(h); h = r.relu(h); h = r.maxpool(h)
        h = r.layer1(h); h = r.layer2(h); h = r.layer3(h); h = r.layer4(h)
    assert tuple(h.shape[-2:]) == (7, 7), h.shape
    assert h.shape[1] == MO.FEAT_DIM["resnet50"] == 2048

    s = MO._build_tv_swin(MO.SWIN_ARCH, pretrained=False).eval()
    with torch.no_grad():
        f = s.features(x).permute(0, 3, 1, 2)
    assert tuple(f.shape[-2:]) == (7, 7), f.shape
    assert f.shape[1] == MO.FEAT_DIM["swin_b"] == 1024

def test_shared_head_identical_everywhere():
    """The head must not vary with backbone, source, channels or classes."""
    shapes = []
    for bb, ch in (("resnet50", "ms9"), ("swin_b", "rgb3")):
        m = MO.build_model(bb, "scratch", 10, channel_mode=ch)
        kinds = [type(l).__name__ for l in m.classifier]
        shapes.append(kinds)
        assert kinds == ["LayerNorm", "Linear", "GELU", "Dropout", "Linear"], kinds
        assert m.classifier[3].p == 0.3
        assert m.classifier[1].out_features == 512
        assert m.classifier[0].normalized_shape[0] == MO.FEAT_DIM[bb]
    assert shapes[0] == shapes[1]

# --------
# 4. k-shot sampling
# --------
def test_kshot_sampling():
    labels = np.repeat(np.arange(10), 50)
    idx = KS.kshot_indices(labels, 5, 42)
    assert len(idx) == 50
    KS.assert_kshot_valid(labels, idx, 5, 10)
    vals, counts = np.unique(labels[idx], return_counts=True)
    assert len(vals) == 10 and set(counts.tolist()) == {5}
    # deterministic in (labels, k, seed), independent of global RNG
    np.random.seed(7); torch.manual_seed(7)
    assert np.array_equal(idx, KS.kshot_indices(labels, 5, 42))
    assert not np.array_equal(idx, KS.kshot_indices(labels, 5, 123))
    # a class smaller than k contributes all of its examples
    small = np.array([0] * 3 + [1] * 50)
    i2 = KS.kshot_indices(small, 5, 42)
    assert (small[i2] == 0).sum() == 3 and (small[i2] == 1).sum() == 5
    # an empty class must fail loudly
    try:
        KS.assert_kshot_valid(labels, idx[:10], 5, 10)
        raise AssertionError("expected EmptyClassError")
    except KS.EmptyClassError:
        pass

def test_val_test_never_subsampled():
    val = list(range(100)); test = list(range(200))
    KS.assert_splits_intact(val, test, 100, 200)
    try:
        KS.assert_splits_intact(Subset(val, [0, 1]), test, 100, 200)
        raise AssertionError("expected SplitSubsampledError")
    except KS.SplitSubsampledError:
        pass
    # k='full' must return the training split untouched
    ds = list(range(37))
    out, idx = KS.kshot_subset(ds, np.zeros(37, int), "full", 42, 1)
    assert out is ds and idx is None

# --------
# 5. The optimisation-step floor and the schedule
# --------
def test_min_step_formula_and_tmax():
    # Asserted against an EXPLICIT protocol, not the module default: the Config
    # cell may have changed MIN_STEPS_*, and this test must check the formula
    # rather than track whatever the notebook was configured with.
    ref = dataclasses.replace(P.PROTOCOL, min_steps_stage1=300,
                              min_steps_stage2=200, stage1_epochs=15,
                              stage2_epochs=10, warmup_frac=0.10)
    # EuroSAT k=5: 50 images -> 2 batches -> the manuscript's 15 epochs would
    # be 30 Stage-1 steps. The floor lifts it to >= 300.
    p1 = P.plan_stage(50, 1, ref)
    assert p1["steps_per_epoch"] == 2
    assert p1["epochs"] == 150 and p1["total_steps"] == 300
    assert p1["floor_engaged"] is True
    p2 = P.plan_stage(50, 2, ref)
    assert p2["epochs"] == 100 and p2["total_steps"] == 200

    # Full data: the floor is inert, epochs stay at 15/10.
    big = P.plan_stage(19000, 1, ref)
    assert big["epochs"] == 15 and big["floor_engaged"] is False
    assert P.plan_stage(19000, 2, ref)["epochs"] == 10

    # k=100 on EuroSAT (1000 images) leaves Stage 2 at its base 10 epochs.
    assert P.plan_stage(1000, 2, ref)["epochs"] == 10

    # Whatever the notebook is actually configured with, the floor property
    # must hold: every stage reaches at least its configured minimum steps.
    for n in (50, 100, 225, 250, 450, 1000, 19000, 22050):
        for stage, floor in ((1, P.PROTOCOL.min_steps_stage1),
                             (2, P.PROTOCOL.min_steps_stage2)):
            pl = P.plan_stage(n, stage)
            assert pl["total_steps"] >= min(floor, pl["steps_per_epoch"] *
                                            pl["epochs"]), (n, stage)
            assert pl["total_steps"] >= floor or not pl["floor_engaged"]

    # RESISC45 k=5 is 225 images = 7*32 + 1: the singleton batch is dropped,
    # because a batch of 1 makes ResNet-50 BatchNorm raise in train() mode.
    assert P.drop_last_for(225, 32) is True
    assert P.steps_per_epoch(225, 32) == 7
    assert P.drop_last_for(224, 32) is False
    assert P.drop_last_for(50, 32) is False

    # Cosine spans the ACTUAL step count, not a stale T_max of 15/10.
    tot, wu = p1["total_steps"], p1["warmup_steps"]
    assert wu == round(300 * 0.10) == 30
    assert P.lr_scale(0, tot, wu) < 0.05                    # warmup starts low
    assert abs(P.lr_scale(wu, tot, wu) - 1.0) < 1e-9        # peak after warmup
    assert P.lr_scale(tot, tot, wu) < 1e-9                  # cosine reaches 0
    mid = [P.lr_scale(s, tot, wu) for s in range(wu, tot)]
    assert all(mid[i] >= mid[i + 1] - 1e-12 for i in range(len(mid) - 1))

def test_validation_point_count_is_fixed():
    """Model-selection opportunities never exceed 15 (Stage 1) / 10 (Stage 2).

    Swept over every training-set size the ladder can produce on either
    dataset, not a hand-picked few: the interesting failures are at awkward
    ratios like 19 epochs against a base of 15, which a sparse sample misses.
    """
    sizes = set()
    for n_cls, n_full in ((10, 19000), (45, 22050)):
        sizes.add(n_full)
        for k in (5, 10, 25, 50, 100):
            sizes.add(min(n_full, k * n_cls))
    sizes |= set(range(2, 400))            # dense sweep of the small end
    for n in sorted(sizes):
        for stage, base in ((1, 15), (2, 10)):
            pl = P.plan_stage(n, stage)
            npts = len(pl["val_epochs"])
            assert npts <= base, (
                f"n={n} stage={stage}: {npts} validation points against a base "
                f"of {base} -- the number of model-selection opportunities "
                "must not vary with the label budget")
            assert npts >= 1
            assert pl["epochs"] - 1 in pl["val_epochs"], "last epoch validates"
            assert all(0 <= e < pl["epochs"] for e in pl["val_epochs"])

def test_batch_size_frozen():
    assert P.BATCH_SIZE == 32 and P.PROTOCOL.batch_size == 32
    for n in (5, 50, 225, 19000):
        assert P.plan_stage(n, 1)["batch_size"] == 32
    try:
        dataclasses.replace(P.PROTOCOL, batch_size=16)
        raise AssertionError("batch size must be immutable at 32")
    except AssertionError as e:
        if "frozen at 32" not in str(e):
            raise
    for bad in ({"label_smoothing": 0.1}, {"use_tta": True},
                {"torch_compile": True}):
        try:
            dataclasses.replace(P.PROTOCOL, **bad)
            raise AssertionError(f"{bad} must be rejected")
        except AssertionError as e:
            if "must be rejected" in str(e):
                raise

# --------
# 6. Stage-1 cache equivalence + frozen-means-eval
# --------
class _TinyRaw(Dataset):
    def __init__(self, n=64, ch=9, size=8, k=4, seed=0):
        g = np.random.RandomState(seed)
        self.x = g.randint(0, 6000, size=(n, ch, size, size)).astype(np.int32)
        self.y = np.repeat(np.arange(k), n // k).astype(np.int64)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return {"image": torch.from_numpy(self.x[i]), "label": int(self.y[i])}

def _tiny_model(k=4, ch=9):
    torch.manual_seed(0)
    body = nn.Sequential(
        nn.Conv2d(ch, 16, 3, padding=1),
        nn.BatchNorm2d(16),          # so train() vs eval() actually differs
        nn.ReLU(),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
    )
    return MO.FMClassifier(body, 16, k, False, "resnet50", dropout=0.3,
                           hidden=8)

def _test_proto():
    return dataclasses.replace(P.PROTOCOL, stage1_epochs=3, min_steps_stage1=4,
                               stage2_epochs=2, min_steps_stage2=2,
                               use_amp=False, channels_last=False)

def test_stage1_cache_equivalence():
    dev = torch.device("cpu")
    proto = _test_proto()
    ds = _TinyRaw()
    val = _TinyRaw(n=32, seed=1)
    prep = D.GPUPreprocessor("eurosat", "ms9", True, dev, img_size=8)

    torch.manual_seed(0)
    m1 = _tiny_model()
    for mod in m1.modules():
        if isinstance(mod, torch.nn.Dropout):
            mod.p = 0.0
    ftr, ytr = extract_features(m1, ds, prep, dev, None, proto.batch_size, 0)
    fva, yva = extract_features(m1, val, prep, dev, None, proto.batch_size, 0)
    h_c, pl_c, best_c, _ = T.run_stage1(m1, ftr, ytr, fva, yva, dev, proto,
                                        seed=42)

    torch.manual_seed(0)
    m2 = _tiny_model()
    for mod in m2.modules():
        if isinstance(mod, torch.nn.Dropout):
            mod.p = 0.0
    h_u, pl_u, best_u, _ = T.run_stage1_uncached(m2, ds, val, prep, prep, dev,
                                                 proto, seed=42, amp_dtype=None,
                                                 num_workers=0)
    assert pl_c["epochs"] == pl_u["epochs"] == 3
    assert abs(best_c - best_u) < 1e-6, (best_c, best_u)
    for a, b in zip(h_c["val_accuracy"], h_u["val_accuracy"]):
        assert (np.isnan(a) and np.isnan(b)) or abs(a - b) < 1e-6
    for (na, pa), (nb, pb) in zip(m1.classifier.named_parameters(),
                                  m2.classifier.named_parameters()):
        assert torch.allclose(pa, pb, atol=1e-5), na

def test_frozen_backbone_is_eval_mode():
    """A 'frozen' backbone left in train() keeps mutating BatchNorm stats."""
    dev = torch.device("cpu")
    ds = _TinyRaw()
    prep = D.GPUPreprocessor("eurosat", "ms9", True, dev, img_size=8)
    m = _tiny_model()
    before = m.backbone[1].running_mean.clone()
    T.run_stage1_uncached(m, ds, _TinyRaw(n=32, seed=1), prep, prep, dev,
                          _test_proto(), seed=0, amp_dtype=None, num_workers=0)
    assert torch.allclose(before, m.backbone[1].running_mean), (
        "Stage-1 backbone BatchNorm statistics changed: the backbone was not "
        "actually frozen")
    for p in m.backbone.parameters():
        assert p.requires_grad is False

def test_run_determinism():
    dev = torch.device("cpu")
    proto = _test_proto()
    ds, val = _TinyRaw(), _TinyRaw(n=32, seed=1)
    prep = D.GPUPreprocessor("eurosat", "ms9", True, dev, img_size=8)
    outs = []
    for _ in range(2):
        from utils import set_seed
        set_seed(1234)
        m = _tiny_model()
        T.run_stage1_uncached(m, ds, val, prep, prep, dev, proto, seed=1234,
                              amp_dtype=None, num_workers=0)
        _, yp, _ = T.predict_images(m, val, prep, dev, 32, None, 0)
        outs.append(yp)
    assert np.array_equal(outs[0], outs[1]), "identical seeds must agree"

def test_probability_assertions():
    good = np.array([[0.2, 0.8], [0.5, 0.5]])
    T.assert_probs_valid(good)
    for bad in (np.array([[0.2, 0.5]]), np.array([[np.nan, 1.0]])):
        try:
            T.assert_probs_valid(bad)
            raise AssertionError("expected failure")
        except AssertionError as e:
            if "expected failure" in str(e):
                raise

# --------
# 7. EuroSAT array cache byte-identity
# --------
def test_eurosat_cache_byte_identical():
    try:
        import rasterio
        from rasterio.transform import from_origin
    except ImportError:
        print("       (skipped: rasterio unavailable)")
        return
    tmp = tempfile.mkdtemp()
    try:
        rng = np.random.RandomState(0)
        data = rng.randint(0, 12000, size=(13, 64, 64)).astype(np.uint16)
        tif = os.path.join(tmp, "AnnualCrop_1.tif")
        with rasterio.open(tif, "w", driver="GTiff", height=64, width=64,
                           count=13, dtype="uint16",
                           transform=from_origin(0, 0, 10, 10)) as dst:
            dst.write(data)
        with rasterio.open(tif) as src:
            direct = src.read()[D.EUROSAT_SATLAS_BANDS].astype(np.uint16)
        arr = np.lib.format.open_memmap(
            os.path.join(tmp, "c.npy"), mode="w+", dtype=np.uint16,
            shape=(1, 9, 64, 64))
        with rasterio.open(tif) as src:
            arr[0] = src.read()[D.EUROSAT_SATLAS_BANDS].astype(np.uint16)
        arr.flush()
        cached = np.load(os.path.join(tmp, "c.npy"), mmap_mode="r")[0]
        assert np.array_equal(np.asarray(cached), direct), "cache differs"
        assert cached.dtype == np.uint16
        # and the selection really is B4,B3,B2,... from the 13-band product
        assert np.array_equal(np.asarray(cached)[0], data[3])
        assert np.array_equal(np.asarray(cached)[1], data[2])
        assert np.array_equal(np.asarray(cached)[8], data[12])
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

# --------
# 8. Metrics against hand-computed values
# --------
def test_metrics_hand_computed():
    y_true = np.array([0, 1, 0, 1])
    y_prob = np.array([[0.9, 0.1], [0.8, 0.2], [0.6, 0.4], [0.55, 0.45]])
    y_pred = y_prob.argmax(1)                       # all zeros
    assert list(y_pred) == [0, 0, 0, 0]
    m = M.compute_metrics(y_true, y_pred, y_prob, 2)
    assert abs(m["test_acc"] - 0.5) < 1e-12
    # class 0: P=0.5 R=1.0 F1=2/3 ; class 1: F1=0 -> macro 1/3
    assert abs(m["macro_f1"] - 1.0 / 3.0) < 1e-9
    assert abs(m["weighted_f1"] - 1.0 / 3.0) < 1e-9
    assert abs(m["kappa"] - 0.0) < 1e-9
    # ECE over 15 equal-width bins, worked by hand:
    #   conf .90 alone : |1 - .90|   * 1/4 = .025
    #   conf .80 alone : |0 - .80|   * 1/4 = .200
    #   conf .60,.55   : |.5 - .575| * 2/4 = .0375
    assert abs(m["ece"] - 0.2625) < 1e-9, m["ece"]
    assert abs(M.expected_calibration_error(y_true, y_prob, 15) - 0.2625) < 1e-9

    f1s = M.per_class_f1(y_true, y_pred, 2)
    assert abs(f1s[0] - 2.0 / 3.0) < 1e-9 and abs(f1s[1]) < 1e-12
    cm = M.confusion(y_true, y_pred, 2)
    assert cm.tolist() == [[2, 0], [2, 0]]

def test_holm_and_seed_tests():
    p = [0.01, 0.04, 0.03]
    adj = M.holm_bonferroni(p)
    # sorted: .01(x3)=.03, .03(x2)=.06, .04(x1)=.04 -> monotone -> .06
    assert abs(adj[0] - 0.03) < 1e-12
    assert abs(adj[2] - 0.06) < 1e-12
    assert abs(adj[1] - 0.06) < 1e-12
    assert all(0 <= a <= 1 for a in adj)
    assert M.WILCOXON_MIN_P_5_SEEDS == 0.0625
    a = np.array([0.10, 0.11, 0.12, 0.13, 0.14])
    b = a - 0.02
    _, pw = M.wilcoxon_across_seeds(a, b)
    assert abs(pw - 0.0625) < 1e-9, pw            # the floor, exactly
    _, pt = M.paired_ttest_across_seeds(a, b)
    assert pt < 0.05, "the t-test is what can carry significance at n=5"

def test_mcnemar_discordant_rule():
    """the paper: exact when the NUMBER OF DISCORDANT PAIRS is below 25."""
    n = 200
    y = np.zeros(n, int)
    a = np.zeros(n, int); b = np.zeros(n, int)
    a[:2] = 1                     # a wrong on 2   -> c_only = 2
    b[2:32] = 1                   # b wrong on 30  -> b_only = 30
    p, ndisc, bo, co = M.mcnemar_test(y, a, b)
    assert ndisc == 32 and bo == 30 and co == 2
    # min(b,c) = 2 < 25 would have chosen the exact test; b+c = 32 must not.
    assert ndisc >= 25
    assert M.is_non_monotonic([0.1, 0.3, 0.2]) is True
    assert M.is_non_monotonic([0.1, 0.2, 0.3]) is False

# --------
# 9. Resume logic after a simulated interruption
# --------
def test_resume_and_sentinel():
    tmp = tempfile.mkdtemp()
    try:
        paths = ST.Paths(os.path.join(tmp, "drive"), os.path.join(tmp, "scratch"))
        paths.verify_writable()
        csv_path = paths.results_csv("eurosat")
        k1 = ST.run_key("eurosat", "resnet50_satlas-ms_ms9", 5, 42)
        ST.append_result(csv_path, {"run_key": k1, "dataset": "eurosat",
                                    "config_id": "resnet50_satlas-ms_ms9",
                                    "k_shot": 5, "seed": 42, "test_acc": 0.5})
        r = ST.Resume(paths)
        assert r.is_done("eurosat", "resnet50_satlas-ms_ms9", 5, 42)
        assert not r.is_done("eurosat", "resnet50_satlas-ms_ms9", 10, 42)

        # A run that started and never finished must be redone, even though a
        # later crash could have left artefacts behind.
        k2 = ST.run_key("eurosat", "swin_b_imagenet_rgb3", 5, 42)
        ST.begin_run(paths, k2, {})
        ST.append_result(csv_path, {"run_key": k2, "dataset": "eurosat",
                                    "config_id": "swin_b_imagenet_rgb3",
                                    "k_shot": 5, "seed": 42, "test_acc": 0.4})
        r2 = ST.Resume(paths)
        assert not r2.is_done("eurosat", "swin_b_imagenet_rgb3", 5, 42), (
            "an interrupted run must not be read as finished")
        ST.end_run(paths, k2)
        r3 = ST.Resume(paths)
        assert r3.is_done("eurosat", "swin_b_imagenet_rgb3", 5, 42)

        # duplicate rows from a redone run are collapsed on read
        ST.append_result(csv_path, {"run_key": k1, "dataset": "eurosat",
                                    "config_id": "resnet50_satlas-ms_ms9",
                                    "k_shot": 5, "seed": 42, "test_acc": 0.6})
        df = ST.load_results(csv_path)
        assert (df.run_key == k1).sum() == 1
        assert float(df[df.run_key == k1].test_acc.iloc[0]) == 0.6

        # nothing durable outside ARTIFACT_ROOT
        for sub in ("results", "preds", "splits", "checkpoints", "logs"):
            assert os.path.isdir(os.path.join(paths.root, sub))
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

# --------
# 10. analyze.py on a synthetic results CSV with known answers
# --------
def _synth_results(tmp):
    """A_S9=.90 A_S3=.80 A_I9=.70 A_I3=.50 at every budget, 5 seeds."""
    rows = []
    spec = [("resnet50_satlas-ms_ms9", "satlas", "ms9", 0.90),
            ("resnet50_satlas-rgb_rgb3", "satlas", "rgb3", 0.80),
            ("resnet50_imagenet_ms9", "imagenet", "ms9", 0.70),
            ("resnet50_imagenet_rgb3", "imagenet", "rgb3", 0.50)]
    for cid, src, ch, acc in spec:
        for k in ["5", "10", "25", "50", "100", "full"]:
            for i, seed in enumerate([42, 123, 2024, 7, 999]):
                a = acc if k == "full" else acc * {"5": 0.5, "10": 0.9,
                                                   "25": 0.99, "50": 1.0,
                                                   "100": 1.0}[k]
                rows.append({
                    "run_key": f"eurosat|{cid}|{k}|{seed}", "dataset": "eurosat",
                    "config_id": cid, "backbone": "resnet50", "pretraining": src,
                    "channel_mode": ch, "satlas_variant": "", "tier": "core",
                    "augment": False, "k_shot": k, "seed": seed,
                    "test_acc": a + (i - 2) * 1e-4, "macro_f1": a,
                    "frozen_acc": a - 0.05, "gpu_name": "Tesla T4"})
    p = os.path.join(tmp, "eurosat_results.csv")
    import pandas as pd
    pd.DataFrame(rows).to_csv(p, index=False)
    return p

def test_analyze_known_answers():
    import pandas as pd
    tmp = tempfile.mkdtemp()
    try:
        df = pd.read_csv(_synth_results(tmp))
        fac = AN.factorial_decomposition(df)
        full = fac[fac.k_shot == "full"].iloc[0]
        # E_source   = 1/2[(.9+.8)-(.7+.5)] = .25
        # E_spectral = 1/2[(.9+.7)-(.8+.5)] = .15
        # interaction= (.9-.8)-(.7-.5)      = -.10
        assert abs(full.effect_source - 0.25) < 1e-3, full.effect_source
        assert abs(full.effect_spectral - 0.15) < 1e-3, full.effect_spectral
        assert abs(full.interaction - (-0.10)) < 1e-3, full.interaction

        d = AN.compute_deltas(df, "eurosat")
        row = d[(d.channel_mode == "ms9") & (d.k_shot == "full")].iloc[0]
        assert abs(row.delta - 0.20) < 1e-3, row.delta      # .90 - .70
        assert bool(row.gpu_mismatch) is False

        le = AN.label_efficiency(df, "eurosat")
        r = le[le.config_id == "resnet50_satlas-ms_ms9"].iloc[0]
        # threshold .95*.90; curve is .45,.81,.891,.90,.90 -> bracket 10..25
        f = (0.855 - 0.81) / (0.891 - 0.81)
        exp = 10 ** (np.log10(10) + f * (np.log10(25) - np.log10(10)))
        assert abs(r.labels_per_class_to_95pct - exp) < 0.5, (
            r.labels_per_class_to_95pct, exp)
        assert r.method == "interpolated" and not r.non_monotonic

        fv = AN.frozen_vs_finetuned(df, "eurosat")
        assert abs(fv.finetuning_gain.iloc[0] - 0.05) < 1e-6

        st = AN.seed_tests(df, "eurosat")
        assert "wilcoxon_p_floor" in st.columns
        assert (st.wilcoxon_p_floor == 0.0625).all()
        assert "p_ttest" in st.columns and "p_ttest_holm" in st.columns
        # Holm is applied within each budget, not across the whole grid
        for k, g in st.groupby("k_shot"):
            m = g.p_wilcoxon.notna()
            if m.any():
                assert np.allclose(
                    g.loc[m, "p_wilcoxon_holm"].values,
                    M.holm_bonferroni(g.loc[m, "p_wilcoxon"].values),
                    equal_nan=True)

        # mixed augment values must be refused
        mixed = df.copy()
        mixed.loc[mixed.index[:10], "augment"] = True
        for fn in (AN.factorial_decomposition,):
            try:
                fn(mixed)
                raise AssertionError("expected MixedAugmentError")
            except AN.MixedAugmentError:
                pass
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

def test_config_grid_shape():
    from configs import configs_for, K_SHOTS, SEEDS, grid_size, cfg_id
    assert len(configs_for("eurosat", ["core"])) == 6
    assert len(configs_for("eurosat", ["factorial"])) == 4
    assert len(configs_for("resisc45", ["core"])) == 6
    assert len(configs_for("resisc45", ["matched"])) == 1
    assert len(configs_for("eurosat")) + len(configs_for("resisc45")) == 17
    assert grid_size(tiers=["core"]) == 360
    assert grid_size(tiers=["factorial"]) == 120
    assert grid_size(tiers=["matched"]) == 30
    assert grid_size() == 510
    assert grid_size(tiers=["aug"]) == 360
    assert len(K_SHOTS) == 6 and len(SEEDS) == 5
    # augment is off everywhere in the main tiers, and part of the id
    for ds in ("eurosat", "resisc45"):
        assert all(c.augment is False for c in configs_for(ds))
        assert all(c.augment is True for c in configs_for(ds, ["aug"]))
    a = configs_for("eurosat", ["core"])[0]
    b = configs_for("eurosat", ["aug"])[0]
    assert cfg_id(a) != cfg_id(b) and cfg_id(b).endswith("_aug")
    # Config ids must be unique WITHIN a dataset. They are deliberately not
    # unique across datasets (resnet50_imagenet_rgb3 exists on both); run_key
    # and every prediction filename are dataset-scoped, so that is safe.
    from configs import CONFIGS
    for ds in ("eurosat", "resisc45"):
        ids = [cfg_id(c) for c in CONFIGS[ds]]
        assert len(ids) == len(set(ids)), (ds, ids)
    from store import run_key
    assert run_key("eurosat", "x", 5, 42) != run_key("resisc45", "x", 5, 42)

def test_llrd_groups_uniform():
    """Both backbones must get the SAME multiplier set from the same gamma."""
    resnet_names = (["conv1.weight", "bn1.weight", "bn1.bias"] +
                    [f"layer{i}.0.conv1.weight" for i in (1, 2, 3, 4)])
    satlas_resnet = ["backbone.resnet." + n for n in resnet_names]
    swin_names = ([f"features.{i}.0.weight" for i in range(8)] + ["norm.weight"])
    satlas_swin = ["backbone.backbone." + n for n in swin_names]
    for names, fam in ((resnet_names, "resnet50"), (satlas_resnet, "resnet50"),
                       (swin_names, "swin_b"), (satlas_swin, "swin_b")):
        stages = {MO.stage_of(n, fam) for n in names}
        assert stages == {0, 1, 2, 3, 4}, (fam, sorted(stages))
    assert MO.resnet_stage_of("backbone.resnet.layer3.5.bn2.weight") == 3
    assert MO.swin_stage_of("backbone.backbone.features.7.1.norm1.weight") == 4
    assert MO.swin_stage_of("norm.bias") == 4

    m = _tiny_model()
    g = MO.llrd_param_groups(m, 1e-4, 0.75, 5, 10.0, 1e-5)
    head = [x for x in g if x["name"] == "head"][0]
    assert abs(head["lr"] - 1e-3) < 1e-12, "head gets 10x the backbone lr"
    for grp in g:
        if grp["name"].startswith("backbone_stage"):
            s = int(grp["name"][-1])
            assert abs(grp["lr"] - 1e-4 * 0.75 ** (4 - s)) < 1e-15

# --------
def run_all(verbose=True):
    global _RESULTS
    _RESULTS = []
    print("=" * 62)
    print("FAST TEST SUITE")
    print("=" * 62)
    suite = [
        ("normalization matches the protocol table", test_normalization_table_iii),
        ("band order is B4,B3,B2,B5,B6,B7,B8,B11,B12", test_band_order),
        ("clip happens before resize", test_clip_before_resize),
        ("channel inflation preserves pre-activation magnitude",
         test_channel_inflation_preserves_magnitude),
        ("final feature maps are 7x7 at 224", test_final_feature_maps_are_7x7),
        ("classification head identical across configs",
         test_shared_head_identical_everywhere),
        ("k-shot sampling correct and deterministic", test_kshot_sampling),
        ("validation/test never subsampled", test_val_test_never_subsampled),
        ("min-step formula and cosine T_max", test_min_step_formula_and_tmax),
        ("validation point count fixed at 15/10",
         test_validation_point_count_is_fixed),
        ("batch size frozen; forbidden options rejected",
         test_batch_size_frozen),
        ("Stage-1 cached == uncached", test_stage1_cache_equivalence),
        ("frozen backbone is genuinely frozen (eval mode)",
         test_frozen_backbone_is_eval_mode),
        ("run determinism under a fixed seed", test_run_determinism),
        ("probability sanity assertions fire", test_probability_assertions),
        ("EuroSAT array cache byte-identical", test_eurosat_cache_byte_identical),
        ("metrics match hand-computed values", test_metrics_hand_computed),
        ("Holm, Wilcoxon floor, paired t", test_holm_and_seed_tests),
        ("McNemar switches on discordant pairs", test_mcnemar_discordant_rule),
        ("resume + sentinel after interruption", test_resume_and_sentinel),
        ("analyze.py on synthetic CSV with known answers",
         test_analyze_known_answers),
        ("grid shape is 17 configs / 510 runs", test_config_grid_shape),
        ("LLRD groups uniform across backbones", test_llrd_groups_uniform),
    ]
    for name, fn in suite:
        _check(name, fn)
    n_fail = sum(1 for _, ok, _ in _RESULTS if not ok)
    print("=" * 62)
    print(f"{len(_RESULTS) - n_fail}/{len(_RESULTS)} passed")
    print("=" * 62)
    if n_fail:
        for name, ok, err in _RESULTS:
            if not ok:
                print(f"  FAILED: {name}\n    {err}")
        raise SystemExit(
            f"{n_fail} test(s) failed -- stopping before any GPU time is "
            "spent. Fix the failure above before continuing.")
    return True

Overwriting src/tests_fast.py


In [ ]:
# Import everything fresh (safe to re-run after editing any %%writefile cell).
import importlib, sys, dataclasses
for _m in ["protocol", "configs", "utils", "store", "kshot", "datasets",
           "models", "featcache", "train", "metrics", "staging",
           "run_experiment", "analyze", "preflight", "tests_fast"]:
    sys.modules.pop(_m, None)
sys.path.insert(0, "./src")

import protocol, configs, utils, store, kshot, datasets, models
import featcache, train, metrics, staging, run_experiment, analyze, preflight

# Apply the Config cell to the frozen protocol. One object, read by every run.
#models.SWIN_ARCH = SWIN_ARCH
PROTO = dataclasses.replace(
    protocol.PROTOCOL,
    min_steps_stage1=MIN_STEPS_STAGE1, min_steps_stage2=MIN_STEPS_STAGE2,
    use_amp=USE_AMP, warmup_frac=WARMUP_FRAC, grad_clip_norm=GRAD_CLIP_NORM,
    llrd_gamma=LLRD_GAMMA, head_lr_mult=HEAD_LR_MULT,
    channels_last=CHANNELS_LAST, use_ema=USE_EMA, ema_decay=EMA_DECAY,
    freeze_bn_stage2=FREEZE_BN_STAGE2, grad_checkpointing=GRAD_CHECKPOINTING)
protocol.PROTOCOL = PROTO

PATHS = store.Paths(ARTIFACT_ROOT, SCRATCH_ROOT)
PATHS.verify_writable()
RESUME = store.Resume(PATHS)

print(f"protocol fingerprint : {PROTO.fingerprint()}")
print(f"batch size           : {PROTO.batch_size} (frozen)")
print(f"swin arch            : {models.SWIN_ARCH}")
print(f"ARTIFACT_ROOT        : {PATHS.root}  ({PATHS.free_gb():.0f} GB free)")
print(f"scratch              : {PATHS.scratch}")
print(f"resume               : {RESUME.report()}")
print(f"grid selected        : {configs.grid_size(tiers=TIERS)} runs "
      f"across {TIERS}")

protocol fingerprint : 6e223e81e1ad
batch size           : 32 (frozen)
swin arch            : swin_v2_b
ARTIFACT_ROOT        : /content/drive/MyDrive/fm_transfer_study  (175 GB free)
scratch              : /content/fmts
resume               : 510 run(s) already complete
grid selected        : 510 runs across ['core', 'factorial', 'matched']


## 4 — Stage the data

Downloads or copies the archives, extracts them, checks integrity, and builds the decoded caches. **20–40 minutes the first time**, seconds on a restarted session — every step can be re-run safely.

EuroSAT is decoded once into a memory-mapped `uint16` array at native 64x64 in model band order. Reading a 13-band GeoTIFF per item through rasterio would otherwise dominate the cost of the whole study.

In [ ]:
import os, numpy as np

# -------- EuroSAT ----------
eur_root = os.path.join(PATHS.data, "eurosat")
if not os.path.isdir(eur_root) or not os.listdir(eur_root):
    if EUROSAT_SOURCE == "kaggle":
        got = staging.kaggle_download(KAGGLE_EUROSAT_SLUG, PATHS.archives)
        if os.path.isdir(got):
            os.makedirs(eur_root, exist_ok=True)
            import shutil
            for item in os.listdir(got):
                s, d = os.path.join(got, item), os.path.join(eur_root, item)
                if not os.path.exists(d):
                    (shutil.copytree if os.path.isdir(s) else shutil.copy2)(s, d)
        else:
            staging.extract_archive(got, eur_root)
    else:
        arc = staging.stage_archive(EUROSAT_ARCHIVE, PATHS.archives)
        staging.extract_archive(arc, eur_root)

EUR = staging.resolve_eurosat(eur_root)
EUR_CACHE = staging.build_eurosat_cache(EUR, PATHS.cache)

# The released partition is used verbatim; its CSVs are hashed onto every row.
eur_splits = {k: v for k, v in EUR["csvs"].items()}

# -------- RESISC45 ----------
res_root = os.path.join(PATHS.data, "resisc45")
if not os.path.isdir(res_root) or not os.listdir(res_root):
    if RESISC45_SOURCE == "kaggle" and KAGGLE_RESISC45_SLUG:
        got = staging.kaggle_download(KAGGLE_RESISC45_SLUG, PATHS.archives)
        if os.path.isdir(got):
            res_root = got
        else:
            staging.extract_archive(got, res_root)
    else:
        arc = staging.stage_archive(RESISC45_ARCHIVE, PATHS.archives)
        staging.extract_archive(arc, res_root)

# find the directory that actually holds the 45 class folders
def _find_class_root(root, n=45):
    for dirpath, dirnames, _ in os.walk(root):
        subs = [d for d in dirnames
                if os.path.isdir(os.path.join(dirpath, d))]
        if len(subs) == n:
            return dirpath
    raise RuntimeError(
        f"Could not find 45 class folders under {root}. RESISC45 must be the "
        "FULL NWPU-RESISC45 release (45 classes x 700 images); reduced "
        "mirrors are not interchangeable.")

RES_ROOT = _find_class_root(res_root)
# The split is generated once with seed 42, persisted to ARTIFACT_ROOT and
# reused verbatim by every configuration and every seed (see the paper).
RES_SPLITS = datasets.build_resisc45_splits(RES_ROOT, PATHS.splits, seed=42)
RES_CACHE = staging.build_resisc45_cache(RES_SPLITS, RES_ROOT, PATHS.cache,
                                         enabled=CACHE_RESISC45_NPY)

# -------- handles ----------
EUR_NAMES = {i: n for i, n in enumerate(datasets.EUROSAT_CLASSES)}
import csv as _csv
_rn = {}
with open(RES_SPLITS["train"], newline="") as fh:
    for r in _csv.DictReader(fh):
        _rn[int(r["label"])] = r["class"]
RES_NAMES = _rn

HANDLES = {}
if "eurosat" in RUN_DATASETS:
    HANDLES["eurosat"] = run_experiment.build_handles(
        "eurosat", EUR_CACHE, eur_splits, EUR_NAMES)
if "resisc45" in RUN_DATASETS:
    HANDLES["resisc45"] = run_experiment.build_handles(
        "resisc45", RES_CACHE, RES_SPLITS, RES_NAMES, data_root=RES_ROOT)

# -------- dataset_summary.csv : resolves the the paper editorial note ----
rows = []
for ds, h in HANDLES.items():
    names = EUR_NAMES if ds == "eurosat" else RES_NAMES
    for split, obj in (("train", h.train), ("val", h.val), ("test", h.test)):
        rows += staging.summarize_split(ds, split, obj.labels, names)
SUMMARY = staging.write_dataset_summary(
    rows, os.path.join(PATHS.results, "dataset_summary.csv"),
    os.path.join(PATHS.results, "dataset_summary.md"))

for ds, h in HANDLES.items():
    tot = len(h.train) + h.n_val + h.n_test
    print(f"{ds:9s}: train={len(h.train)} val={h.n_val} test={h.n_test} "
          f"TOTAL={tot}  classes={h.n_classes}")
    print(f"           split hashes: {h.hashes}")
print("\ndataset_summary.csv written. Use its EuroSAT total to replace the")
print("class counts above against the split CSVs.")

[stage] kagglehub download: apollo2506/eurosat-dataset
Using Colab cache for faster access to the 'eurosat-dataset' dataset.
[stage] EuroSAT: 3597 GeoTIFFs under /content/fmts/data/eurosat/EuroSATallBands
[stage] EuroSAT partition: train=18900, val=5400, test=2700 (total 27000)
[stage]   eurosat/train: 2500/18900
[stage]   eurosat/train: 5000/18900
[stage]   eurosat/train: 7500/18900
[stage]   eurosat/train: 10000/18900
[stage]   eurosat/train: 12500/18900
[stage]   eurosat/train: 15000/18900
[stage]   eurosat/train: 17500/18900
[stage] eurosat/train: cached 18900 patches
[stage]   eurosat/val: 2500/5400
[stage]   eurosat/val: 5000/5400
[stage] eurosat/val: cached 5400 patches
[stage]   eurosat/test: 2500/2700
[stage] eurosat/test: cached 2700 patches
[stage] copying /content/drive/MyDrive/datasets/NWPU-RESISC45.zip -> /content/fmts/archives/NWPU-RESISC45.zip (0.43 GB)
[stage] extracting NWPU-RESISC45.zip -> /content/fmts/data/resisc45
[stage]   resisc45/train: 5000/22050
[stage]   res

## 5 — Tests

A fast suite on tiny synthetic data, CPU only. **Under two minutes.**

It checks the commitments the paper makes, not merely that the code runs: the normalization contract, channel inflation preserving pre-activation magnitude, 7x7 final feature maps, k-shot sampling, the step-floor formula, Stage-1 cache equivalence, EuroSAT cache byte-identity, determinism, resume after interruption, metrics against hand-computed values, and the analysis script against a synthetic file with known answers.

**If anything fails, stop here — before spending any GPU time.**

In [ ]:
import importlib, tests_fast
importlib.reload(tests_fast)
tests_fast.run_all()
print("\nAll tests passed. Safe to spend GPU time.")

FAST TEST SUITE
  PASS  normalization matches Table III
  PASS  band order is B4,B3,B2,B5,B6,B7,B8,B11,B12
  PASS  clip happens before resize


/content/src/tests_fast.py:137: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  assert abs(float(inflated.weight.abs().sum() /


  PASS  channel inflation preserves pre-activation magnitude
  PASS  final feature maps are 7x7 at 224
  PASS  classification head identical across configs
  PASS  k-shot sampling correct and deterministic
  PASS  validation/test never subsampled
  PASS  min-step formula and cosine T_max
  PASS  validation point count fixed at 15/10
  PASS  batch size frozen; forbidden options rejected
  PASS  Stage-1 cached == uncached
  PASS  frozen backbone is genuinely frozen (eval mode)
  PASS  run determinism under a fixed seed
  PASS  probability sanity assertions fire
  PASS  EuroSAT array cache byte-identical
  PASS  metrics match hand-computed values
  PASS  Holm, Wilcoxon floor, paired t
  PASS  McNemar switches on discordant pairs


/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:423: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return hypotest_fun_in(*args, **kwds)
/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:423: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return hypotest_fun_in(*args, **kwds)


  PASS  resume + sentinel after interruption


/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:423: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return hypotest_fun_in(*args, **kwds)
/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:423: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return hypotest_fun_in(*args, **kwds)
/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:423: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return hypotest_fun_in(*args, **kwds)
/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:423: RuntimeWarning: Precision loss occurred in moment calculati

  PASS  analyze.py on synthetic CSV with known answers
  PASS  grid shape is 17 configs / 510 runs
  PASS  LLRD groups uniform across backbones
23/23 passed

All tests passed. Safe to spend GPU time.


/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:423: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return hypotest_fun_in(*args, **kwds)
/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:423: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return hypotest_fun_in(*args, **kwds)
/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:423: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return hypotest_fun_in(*args, **kwds)
/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:423: RuntimeWarning: Precision loss occurred in moment calculati

## 6 — Preflight

Reports the GPU, proves the heaviest configuration (Swin, nine channels, batch 32, 224x224) fits both forward and backward, times 20 steps per backbone, confirms the two Swin arms use the same architecture, and projects the total grid time. **One to three minutes.**

Batch size is not adjusted here; it stays at 32. If memory is tight, set `GRAD_CHECKPOINTING = True`, which is mathematically identical.

In [ ]:
SIZES = {ds: {"train": len(h.train), "val": h.n_val, "test": h.n_test}
         for ds, h in HANDLES.items()}
PRE = preflight.preflight(sizes=SIZES, tiers=TIERS,
                          datasets=list(HANDLES), proto=PROTO,
                          check_satlas=True)

GPU PREFLIGHT
  device              : cuda
  gpu                 : NVIDIA A100-SXM4-80GB
  vram_total_gb       : 85.09
  vram_free_gb        : 84.65
  cuda_capability     : 8.0
  bf16_supported      : True

-- memory: heaviest configuration --
  Swin/9-band, batch 32 @ 224: OK (peak 7.43 GB)
  ResNet-50/9-band, batch 32: OK

-- throughput (20 steps per backbone) --
  resnet50 : train  1066.5 img/s   infer  2800.7 img/s
  swin_b   : train   192.1 img/s   infer   343.8 img/s

-- determinism --
  identical predictions across two identical runs: True (max |logit diff| 0.00e+00)

-- architecture control --
  Swin arms aligned: ours=v2, Satlas=v2 (swin_v2_b)

-- projected grid time --
  eurosat  :  17.05 h
  resisc45 :  15.76 h
  TOTAL    :  32.82 h (tiers=['core', 'factorial', 'matched'])
  Note: a projection from synthetic-input timings. The pilot cell measures the real thing.


## 7 — Smoke test

Two configurations x two budgets x one seed, end to end, writing real result rows. **5–20 minutes.** These are genuine grid runs, so nothing is wasted — the full grid skips them on resume.

In [ ]:
import time
SMOKE_DS = "eurosat" if "eurosat" in HANDLES else list(HANDLES)[0]
_core = configs.configs_for(SMOKE_DS, ["core"])
SMOKE_CFGS = [_core[0], _core[3]]          # one Satlas arm, one ImageNet arm
FCACHE = featcache.FeatureCache(PATHS.feats)

t0 = time.time()
for cfg in SMOKE_CFGS:
    for k in [5, 10]:
        cid = configs.cfg_id(cfg)
        if RESUME.is_done(SMOKE_DS, cid, k, 42):
            print(f"[skip] {cid} k={k}"); continue
        row = run_experiment.run_one(SMOKE_DS, cfg, k, 42, HANDLES[SMOKE_DS],
                                     PATHS, FCACHE, utils.get_device(), PROTO,
                                     NUM_WORKERS, VERBOSE_EPOCHS, SAVE_CURVES)
        RESUME.mark(row["run_key"])
print(f"\nSmoke test finished in {(time.time()-t0)/60:.1f} min.")
print("Results CSV:", PATHS.results_csv(SMOKE_DS))

    s1 ep10/150 val=0.7144
    s1 ep20/150 val=0.6907
    s1 ep30/150 val=0.7028
    s1 ep40/150 val=0.6978
    s1 ep50/150 val=0.6948
    s1 ep60/150 val=0.6702
    s1 ep70/150 val=0.6765
    s1 ep80/150 val=0.6807
    s1 ep90/150 val=0.6844
    s1 ep100/150 val=0.6880
    s1 ep110/150 val=0.6902
    s1 ep120/150 val=0.6904
    s1 ep130/150 val=0.6904
    s1 ep140/150 val=0.6906
    s1 ep150/150 val=0.6907
    s2 ep10/100 val=0.7891
    s2 ep20/100 val=0.8230
    s2 ep30/100 val=0.7965
    s2 ep40/100 val=0.8189
    s2 ep50/100 val=0.7887
    s2 ep60/100 val=0.7793
    s2 ep70/100 val=0.7820
    s2 ep80/100 val=0.7935
    s2 ep90/100 val=0.7959
    s2 ep100/100 val=0.8002
[done] eurosat resnet50_satlas-ms_ms9 k=5 seed=42 acc=0.8285 frozen=0.7130 e1=150 e2=100 (0.8 min)
    s1 ep5/75 val=0.7426
    s1 ep10/75 val=0.7267
    s1 ep15/75 val=0.7426
    s1 ep20/75 val=0.7489
    s1 ep25/75 val=0.7467
    s1 ep30/75 val=0.7461
    s1 ep35/75 val=0.7374
    s1 ep40/75 val=0.7407
    s1 ep45/

100%|██████████| 336M/336M [00:01<00:00, 215MB/s]


    s1 ep10/150 val=0.7139
    s1 ep20/150 val=0.6661
    s1 ep30/150 val=0.6883
    s1 ep40/150 val=0.6926
    s1 ep50/150 val=0.6924
    s1 ep60/150 val=0.6922
    s1 ep70/150 val=0.6922
    s1 ep80/150 val=0.6933
    s1 ep90/150 val=0.6937
    s1 ep100/150 val=0.6941
    s1 ep110/150 val=0.6943
    s1 ep120/150 val=0.6944
    s1 ep130/150 val=0.6941
    s1 ep140/150 val=0.6950
    s1 ep150/150 val=0.6950
    s2 ep10/100 val=0.7959
    s2 ep20/100 val=0.8120
    s2 ep30/100 val=0.8289
    s2 ep40/100 val=0.8137
    s2 ep50/100 val=0.7613
    s2 ep60/100 val=0.7956
    s2 ep70/100 val=0.8020
    s2 ep80/100 val=0.8059
    s2 ep90/100 val=0.8026
    s2 ep100/100 val=0.8028
[done] eurosat swin_b_imagenet_rgb3 k=5 seed=42 acc=0.8215 frozen=0.7252 e1=150 e2=100 (2.9 min)
    s1 ep5/75 val=0.7483
    s1 ep10/75 val=0.8019
    s1 ep15/75 val=0.8063
    s1 ep20/75 val=0.7857
    s1 ep25/75 val=0.8069
    s1 ep30/75 val=0.8050
    s1 ep35/75 val=0.8041
    s1 ep40/75 val=0.8043
    s1 ep45/75

## 8 — Pilot, then stop

Runs the core tier only, at two budgets x two seeds, prints a summary, and stops so you can read it before committing to the full grid. The projected cost is printed first; trim `PILOT_BUDGETS` or `PILOT_SEEDS` in cell 1 if it is more than you want to spend.

Nothing after this cell runs on its own. Read the summary, then run cell 9.

In [ ]:
import numpy as np, pandas as pd

n_pilot = sum(len(configs.configs_for(ds, ["core"])) for ds in HANDLES) \
          * len(PILOT_BUDGETS) * len(PILOT_SEEDS)
print(f"Pilot: {n_pilot} runs (core tier, budgets={PILOT_BUDGETS}, "
      f"seeds={PILOT_SEEDS}).")
if PRE.get("projection"):
    frac = n_pilot / max(1, configs.grid_size(tiers=["core"]))
    print(f"Rough projection: ~{PRE['projection']['hours']*frac:.1f} h "
          "(cheapest budgets, so likely less).")

PILOT = run_experiment.run_grid(
    HANDLES, PATHS, RESUME, utils.get_device(), tiers=["core"],
    datasets=list(HANDLES), budgets=PILOT_BUDGETS, seeds=PILOT_SEEDS,
    max_hours=MAX_HOURS, proto=PROTO, num_workers=NUM_WORKERS,
    feat_cache=FCACHE, verbose=VERBOSE_EPOCHS, save_curves=SAVE_CURVES)

print("\n" + "=" * 70)
print("PILOT SUMMARY - read this before running the full grid")
print("=" * 70)
for ds in HANDLES:
    df = store.load_results(PATHS.results_csv(ds))
    if df is None:
        continue
    df = df[df.k_shot.astype(str).isin([str(k) for k in PILOT_BUDGETS])]
    if not len(df):
        continue
    piv = df.pivot_table(index=["config_id"], columns="k_shot",
                         values=["test_acc", "frozen_acc"], aggfunc="mean")
    print(f"\n--- {ds} : mean accuracy over {len(PILOT_SEEDS)} seeds ---")
    print(piv.round(4).to_string())
    print(f"\n  median run time : {df.total_seconds.median()/60:.1f} min")
    print(f"  epochs (s1/s2)  : "
          f"{sorted(set(zip(df.epochs_stage1, df.epochs_stage2)))}")
    print(f"  step floor hit  : s1={df.floor_engaged_s1.astype(str).eq('True').mean():.0%}"
          f"  s2={df.floor_engaged_s2.astype(str).eq('True').mean():.0%}")
    print(f"  GPUs seen       : {sorted(df.gpu_name.unique())}")
print("\n" + "=" * 70)
print("STOP. Check that accuracies are sane, that the frozen (Stage-1) and")
print("fine-tuned (Stage-2) numbers differ sensibly, and that run time x 510")
print("fits your compute. Then run cell 9.")
print("=" * 70)

Pilot: 24 runs (core tier, budgets=[5, 10], seeds=[42, 123]).
Rough projection: ~0.7 h (cheapest budgets, so likely less).
    s1 ep10/150 val=0.7157
    s1 ep20/150 val=0.6867
    s1 ep30/150 val=0.6956
    s1 ep40/150 val=0.7143
    s1 ep50/150 val=0.7137
    s1 ep60/150 val=0.7148
    s1 ep70/150 val=0.7133
    s1 ep80/150 val=0.7141
    s1 ep90/150 val=0.7139
    s1 ep100/150 val=0.7146
    s1 ep110/150 val=0.7143
    s1 ep120/150 val=0.7148
    s1 ep130/150 val=0.7152
    s1 ep140/150 val=0.7152
    s1 ep150/150 val=0.7152
    s2 ep10/100 val=0.7720
    s2 ep20/100 val=0.7991
    s2 ep30/100 val=0.7989
    s2 ep40/100 val=0.7989
    s2 ep50/100 val=0.7996
    s2 ep60/100 val=0.7770
    s2 ep70/100 val=0.7854
    s2 ep80/100 val=0.7872
    s2 ep90/100 val=0.7907
    s2 ep100/100 val=0.7911
[done] eurosat resnet50_satlas-ms_ms9 k=5 seed=123 acc=0.8096 frozen=0.7219 e1=150 e2=100 (0.6 min)
    s1 ep10/150 val=0.6376
    s1 ep20/150 val=0.7720
    s1 ep30/150 val=0.7757
    s1 ep40/15

100%|██████████| 97.8M/97.8M [00:00<00:00, 221MB/s]


    s1 ep10/150 val=0.7396
    s1 ep20/150 val=0.7793
    s1 ep30/150 val=0.7731
    s1 ep40/150 val=0.7746
    s1 ep50/150 val=0.7744
    s1 ep60/150 val=0.7754
    s1 ep70/150 val=0.7767
    s1 ep80/150 val=0.7772
    s1 ep90/150 val=0.7770
    s1 ep100/150 val=0.7767
    s1 ep110/150 val=0.7774
    s1 ep120/150 val=0.7770
    s1 ep130/150 val=0.7772
    s1 ep140/150 val=0.7772
    s1 ep150/150 val=0.7772
    s2 ep10/100 val=0.7059
    s2 ep20/100 val=0.7261
    s2 ep30/100 val=0.7406
    s2 ep40/100 val=0.7461
    s2 ep50/100 val=0.7444
    s2 ep60/100 val=0.7491
    s2 ep70/100 val=0.7550
    s2 ep80/100 val=0.7576
    s2 ep90/100 val=0.7604
    s2 ep100/100 val=0.7633
[done] eurosat resnet50_imagenet_rgb3 k=5 seed=42 acc=0.7667 frozen=0.7893 e1=150 e2=100 (0.8 min)
    s1 ep10/150 val=0.7985
    s1 ep20/150 val=0.8061
    s1 ep30/150 val=0.8056
    s1 ep40/150 val=0.8059
    s1 ep50/150 val=0.8078
    s1 ep60/150 val=0.8072
    s1 ep70/150 val=0.8085
    s1 ep80/150 val=0.8087
   

## 9 — Full grid

Resumable, honours `MAX_HOURS`, and writes `progress.md` after each block. Runs in tier order (core, factorial, matched), budget-major and ascending, so a grid that stops early still answers the primary question.

**Re-run this cell as often as you need.** Completed runs are skipped; an interrupted run is redone rather than counted as finished. Expect many hours. `progress.md` under `ARTIFACT_ROOT/logs/` carries the live projection.

In [ ]:
FCACHE = featcache.FeatureCache(PATHS.feats)

In [ ]:
RESUME = store.Resume(PATHS)          # re-read after any disconnect
print(RESUME.report())

GRID = run_experiment.run_grid(
    HANDLES, PATHS, RESUME, utils.get_device(), tiers=TIERS,
    datasets=list(HANDLES), max_hours=MAX_HOURS, proto=PROTO,
    num_workers=NUM_WORKERS, feat_cache=FCACHE, verbose=VERBOSE_EPOCHS,
    save_curves=SAVE_CURVES)

print(open(os.path.join(PATHS.logs, "progress.md")).read())

## 10 — Analysis

Runs every analysis the available results support and writes the outputs to `ARTIFACT_ROOT/results/`: pretraining deltas, the EuroSAT 2x2 decomposition, label efficiency, frozen versus fine-tuned, McNemar with Holm correction applied within each budget, seed-level Wilcoxon and paired t-tests, error overlap, per-class F1 and confusion matrices.

Everything is computed from the archived predictions — no model is retrained. **One to five minutes.**

In [ ]:
CLASS_NAMES = {"eurosat": EUR_NAMES, "resisc45": RES_NAMES}
made = analyze.main(PATHS, class_names=CLASS_NAMES, augment=False)
print("\n".join(str(m) for m in made))

In [ ]:
src = open("/content/src/analyze.py").read()
i = src.find("def figure_deltas")
print(src[i:i+2200])

def figure_deltas(deltas, out_png):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    if not len(deltas):
        return None
    dss = sorted(deltas.dataset.unique())
    fig, axes = plt.subplots(1, len(dss), figsize=(7 * len(dss), 4.5),
                             squeeze=False)
    for ax, ds in zip(axes[0], dss):
        sub = deltas[deltas.dataset == ds]
        for (bb, ch, sc), g in sub.groupby(["backbone", "channel_mode",
                                            "satlas_config"]):
            g = g.set_index("k_shot").reindex(K_ORDER).dropna(subset=["delta"])
            variant = sc.split("_")[1] if "_" in sc else sc
            ax.plot(range(len(g)), g.delta.values, marker="o",
                    label=f"{bb} / {ch} / {variant}")
            ax.set_xticks(range(len(g)))
            ax.set_xticklabels(g.index)
        ax.axhline(0, c="grey", lw=1, ls="--")
        ax.set_title(f"{ds}: SatlasPretrain - ImageNet")
        ax.set_xlabel(

---

## Optional A — where the step floor comes from

Sweeps the Stage-1 step floor on a single configuration and reports **validation** accuracy only. Nothing here touches the test split and nothing here changes the grid; it exists so that the chosen `MIN_STEPS_STAGE1` rests on a measurement rather than an assertion. Roughly 10–20 minutes.

In [ ]:
# Validation-only sweep on a single configuration at the smallest budget.
import dataclasses, numpy as np, pandas as pd
SWEEP_DS = "eurosat" if "eurosat" in HANDLES else list(HANDLES)[0]
SWEEP_CFG = configs.configs_for(SWEEP_DS, ["core"])[0]
SWEEP_K, SWEEP_SEED = 5, 42
SWEEP_VALUES = [30, 100, 200, 300, 600, 1200]

h = HANDLES[SWEEP_DS]
dev = utils.get_device()
rows = []
for ms in SWEEP_VALUES:
    proto = dataclasses.replace(PROTO, min_steps_stage1=ms)
    utils.set_seed(SWEEP_SEED)
    m = models.build_model(SWEEP_CFG.backbone, SWEEP_CFG.pretraining,
                           h.n_classes, channel_mode=SWEEP_CFG.channel_mode,
                           satlas_variant=SWEEP_CFG.satlas_variant or "ms",
                           device=dev).to(dev)
    m.set_backbone_trainable(False)
    prep = datasets.preprocessor_for(SWEEP_DS, SWEEP_CFG.channel_mode,
                                     SWEEP_CFG.pretraining, dev)
    bfp = utils.backbone_fingerprint(m.backbone)
    amp = utils.amp_dtype_for(dev, proto.use_amp)
    f = {s: FCACHE.get_or_compute(
            FCACHE.key(SWEEP_DS, configs.cfg_id(SWEEP_CFG), bfp, s),
            (lambda ds=getattr(h, s): featcache.extract_features(
                m, ds, prep, dev, amp, proto.batch_size, NUM_WORKERS)))
         for s in ("train", "val")}
    sub, idx = kshot.kshot_subset(h.train, h.train_labels, SWEEP_K, SWEEP_SEED,
                                  h.n_classes)
    ftr, ytr = f["train"][0][idx], f["train"][1][idx]
    _, plan, best_val, best_ep = train.run_stage1(
        m, ftr, ytr, f["val"][0], f["val"][1], dev, proto, SWEEP_SEED)
    rows.append({"min_steps": ms, "epochs": plan["epochs"],
                 "total_steps": plan["total_steps"],
                 "best_val_acc": best_val, "best_epoch": best_ep})
    print(f"  min_steps={ms:5d}  epochs={plan['epochs']:4d}  "
          f"steps={plan['total_steps']:5d}  val_acc={best_val:.4f}")

sw = pd.DataFrame(rows)
sw.to_csv(os.path.join(PATHS.results, "min_steps_sweep.csv"), index=False)
print("\n", sw.to_string(index=False))
print("\nPick the knee: the smallest floor after which validation accuracy")
print("stops improving. Validation only - the test split is untouched.")

  min_steps=   30  epochs=  15  steps=   30  val_acc=0.7004
  min_steps=  100  epochs=  50  steps=  100  val_acc=0.6987
  min_steps=  200  epochs= 100  steps=  200  val_acc=0.6974
  min_steps=  300  epochs= 150  steps=  300  val_acc=0.7144
  min_steps=  600  epochs= 300  steps=  600  val_acc=0.7022
  min_steps= 1200  epochs= 600  steps= 1200  val_acc=0.7022

  min_steps  epochs  total_steps  best_val_acc  best_epoch
        30      15           30      0.700370           3
       100      50          100      0.698704           7
       200     100          200      0.697407          34
       300     150          300      0.714444           9
       600     300          600      0.702222         179
      1200     600         1200      0.702222         519

Pick the knee: the smallest floor after which validation accuracy
stops improving. Validation only - the test split is untouched.


---

## Optional B — augmentation tier

The separately flagged `aug` tier mirrors the core tier with geometric augmentation on the training split only: random resized crop, horizontal and vertical flips, 90-degree rotations. **No colour jitter** — it is undefined on red-edge and SWIR bands.

Results carry `augment=True`, and the analysis script refuses to pool them with the main results.

In [ ]:
# Only run this if you want to answer "can augmentation substitute for
# geospatial pretraining when labels are scarce?" It is NOT part of the main
# results and must never be mixed with them.
AUG = run_experiment.run_grid(
    HANDLES, PATHS, store.Resume(PATHS), utils.get_device(), tiers=["aug"],
    datasets=list(HANDLES), max_hours=MAX_HOURS, proto=PROTO,
    num_workers=NUM_WORKERS, feat_cache=FCACHE, verbose=VERBOSE_EPOCHS)
analyze.main(PATHS, class_names=CLASS_NAMES, augment=True)

    s2 ep10/100 val=0.7839
    s2 ep20/100 val=0.8035
    s2 ep30/100 val=0.8244
    s2 ep40/100 val=0.8198
    s2 ep50/100 val=0.8193
    s2 ep60/100 val=0.8159
    s2 ep70/100 val=0.8278
    s2 ep80/100 val=0.8298
    s2 ep90/100 val=0.8330
    s2 ep100/100 val=0.8298
[done] eurosat resnet50_satlas-ms_ms9_aug k=5 seed=42 acc=0.8281 frozen=0.7426 e1=150 e2=100 (1.6 min)
    s2 ep10/100 val=0.7589
    s2 ep20/100 val=0.7854
    s2 ep30/100 val=0.7828
    s2 ep40/100 val=0.7763
    s2 ep50/100 val=0.7744
    s2 ep60/100 val=0.8104
    s2 ep70/100 val=0.8048
    s2 ep80/100 val=0.7981
    s2 ep90/100 val=0.8041
    s2 ep100/100 val=0.7993
[done] eurosat resnet50_satlas-ms_ms9_aug k=5 seed=123 acc=0.8278 frozen=0.7204 e1=150 e2=100 (1.6 min)
    s2 ep10/100 val=0.7450
    s2 ep20/100 val=0.7837
    s2 ep30/100 val=0.7448
    s2 ep40/100 val=0.7476
    s2 ep50/100 val=0.7528
    s2 ep60/100 val=0.7781
    s2 ep70/100 val=0.7628
    s2 ep80/100 val=0.7730
    s2 ep90/100 val=0.7719
    s2 e

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b6ad6225620>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1673, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.12/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 1136, in wait
    ready = selector.select(timeout)
            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/selectors.py", line 415, in select
    fd_event_list = self._selector.poll(timeout)
    

KeyboardInterrupt: 

In [ ]:
# Reads only — writes nothing, changes nothing.
#  still needs. Paste the output back.
import os, glob, json
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 400)

ROOT = "/content/drive/MyDrive/fm_transfer_study"

print("=" * 70)
print("FILES PRESENT")
print("=" * 70)
for f in sorted(glob.glob(f"{ROOT}/**/*.*", recursive=True)):
    if os.path.isfile(f) and not f.endswith(".npz"):
        print(f"{os.path.getsize(f)/1024:9.1f} KB  {f[len(ROOT)+1:]}")
npz = glob.glob(f"{ROOT}/**/*.npz", recursive=True)
print(f"          ... plus {len(npz)} .npz prediction files")

# --------
print("\n" + "=" * 70)
print("MAIN RESULTS — one row per run")
print("=" * 70)
frames = []
for f in glob.glob(f"{ROOT}/**/*_results.csv", recursive=True):
    d = pd.read_csv(f)
    d["_src"] = os.path.basename(f)
    frames.append(d)
    print(f"\n{os.path.basename(f)}: {len(d)} rows")
    print("  columns:", d.columns.tolist())

if frames:
    R = pd.concat(frames, ignore_index=True)
    print(f"\nTOTAL ROWS: {len(R)}")
    if "augment" in R.columns:
        print("by augment flag:", R.augment.value_counts().to_dict())
        R = R[R.augment == False]
        print(f"main-results rows (augment=False): {len(R)}")

    # every metric column, mean +/- std over seeds
    keys = [c for c in ("dataset", "config_id", "k_shot") if c in R.columns]
    metrics = [c for c in R.columns if any(
        m in c.lower() for m in
        ("acc", "f1", "kappa", "auc", "ece", "calib", "frozen", "loss"))]
    metrics = [c for c in metrics if pd.api.types.is_numeric_dtype(R[c])]
    print("\nmetric columns found:", metrics)

    print("\n" + "-" * 70)
    print("PER-CONFIG SUMMARY  (mean +/- std over seeds)")
    print("-" * 70)
    g = R.groupby(keys)[metrics].agg(["mean", "std", "count"])
    print(g.round(4).to_string())

# --------
print("\n" + "=" * 70)
print("ALL OTHER CSVs IN FULL")
print("=" * 70)
for f in sorted(glob.glob(f"{ROOT}/**/*.csv", recursive=True)):
    b = os.path.basename(f)
    if b.endswith("_results.csv") or b.startswith("resisc45_t") \
       or b in ("resisc45_train.csv", "resisc45_val.csv", "resisc45_test.csv"):
        continue
    d = pd.read_csv(f)
    print(f"\n----- {b}  ({len(d)} rows) -----")
    print(d.round(5).to_string())

# --------
print("\n" + "=" * 70)
print("MARKDOWN / TEXT ARTIFACTS")
print("=" * 70)
for f in sorted(glob.glob(f"{ROOT}/**/*.md", recursive=True)):
    print(f"\n----- {os.path.basename(f)} -----")
    t = open(f).read()
    print(t[:4000] + ("\n...[truncated]" if len(t) > 4000 else ""))

print("\n" + "=" * 70)
print("DONE — paste everything above")
print("=" * 70)

In [ ]:
#  FıNAL DATA EXTRACTION AND DIAGNOSIS — every remaining analysis, in one cell.
#  Reads only. Writes CSVs to ARTIFACT_ROOT. No GPU, no retraining.
#  Output is bounded so it pastes back without truncating.
#
#  Run in a fresh Colab cell after Cell 1 (needs PATHS, or edit ROOT).
import os, glob, itertools, warnings
import numpy as np, pandas as pd
from scipy import stats
warnings.filterwarnings("ignore")

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_columns", 40)

def hdr(n, t): print("\n" + "=" * 74); print(f"{n}. {t}"); print("=" * 74)

# ── locate ARTIFACT_ROOT robustly ────────────────────────────────────
try:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        print("mounting Drive..."); drive.mount("/content/drive")
except Exception:
    pass

def _has_results(d):
    return bool(glob.glob(os.path.join(d, "**", "*_results.csv"), recursive=True))

_cands = []
if "PATHS" in dir():
    try: _cands.append(PATHS.root)
    except Exception: pass
if "ARTIFACT_ROOT" in dir():
    _cands.append(ARTIFACT_ROOT)
_cands += ["/content/drive/MyDrive/fm_transfer_study",
           "/content/drive/MyDrive/Colab Notebooks/fm_transfer_study",
           "/content/fmts"]

ROOT = next((c for c in _cands if c and _has_results(c)), None)

if ROOT is None:
    print("Searching Drive for *_results.csv ...")
    found = glob.glob("/content/drive/MyDrive/**/*_results.csv", recursive=True)
    if found:
        ROOT = os.path.dirname(found[0])
        print("  found under:", ROOT)
    else:
        print("\n" + "!" * 74)
        print("Could not locate *_results.csv anywhere under /content/drive/MyDrive.")
        print("Checked:")
        for c in _cands:
            print(f"   {c}  ->  exists={os.path.isdir(c) if c else False}")
        print("\nIf Drive is mounted under a different account, or the folder was")
        print("renamed, set ROOT manually at the top of this cell and re-run.")
        print("!" * 74)
        raise SystemExit(1)

print(f"ARTIFACT_ROOT = {ROOT}")

def holm(p):
    p = np.asarray(p, float); m = len(p); o = np.argsort(p)
    adj = np.empty(m); run = 0.0
    for r, i in enumerate(o):
        run = max(run, (m - r) * p[i]); adj[i] = min(run, 1.0)
    return adj

# ══════════════════════════════════════════════ 0. DISCOVER
hdr(0, "STRUCTURE DISCOVERY")
csvs = sorted(glob.glob(f"{ROOT}/**/*.csv", recursive=True))
npzs = sorted(glob.glob(f"{ROOT}/**/*.npz", recursive=True))
print(f"{len(csvs)} csv, {len(npzs)} npz under {ROOT}")
for c in csvs:
    b = os.path.basename(c)
    if b.startswith("resisc45_t") or b.endswith(("_train.csv", "_val.csv", "_test.csv")):
        continue
    try: print(f"   {b:34s} {len(pd.read_csv(c)):>7,} rows")
    except Exception as e: print(f"   {b:34s} READ FAIL {e}")

if npzs:
    print(f"\nexample npz: {os.path.basename(npzs[0])}")
    _d = np.load(npzs[0], allow_pickle=True)
    for k in _d.files:
        a = _d[k]
        print(f"   key '{k}': shape {getattr(a,'shape','-')} dtype {getattr(a,'dtype','-')}")

# ══════════════════════════════════════════════ 1. SIX-METRIC TABLE
hdr(1, "SIX-METRIC TABLE  (mean over 5 seeds, then std)")
_rf = glob.glob(f"{ROOT}/**/*_results.csv", recursive=True)
print("results files:", [os.path.basename(f) for f in _rf])
R = pd.concat([pd.read_csv(f) for f in _rf], ignore_index=True)
AUG = R[R.augment == True] if "augment" in R.columns else R.iloc[0:0]
if "augment" in R.columns:
    R = R[R.augment == False]
print(f"main rows {len(R)}   augmentation rows {len(AUG)}")
print("columns:", ", ".join(R.columns))

MET = [c for c in R.columns if pd.api.types.is_numeric_dtype(R[c]) and any(
    m in c.lower() for m in ("acc", "f1", "kappa", "auc", "ece", "calib"))
    and "epoch" not in c.lower()]
print("metrics found:", MET)

KEY = [c for c in ("dataset", "config_id", "k_shot") if c in R.columns]
MU = R.groupby(KEY)[MET].mean().round(4)
SD = R.groupby(KEY)[MET].std().round(4)
MU.to_csv(f"{ROOT}/six_metric_mean.csv"); SD.to_csv(f"{ROOT}/six_metric_std.csv")
print("\n--- MEANS ---"); print(MU.to_string())
print("\n--- STD OVER 5 SEEDS ---"); print(SD.to_string())

# ══════════════════════════════════════════════ 2. STAGE 1 vs STAGE 2
hdr(2, "FROZEN (Stage 1) vs FINE-TUNED (Stage 2)")
s2 = next((c for c in R.columns if c.lower() in ("test_acc", "acc", "accuracy")), None)
s1 = next((c for c in R.columns if "frozen" in c.lower()
           and pd.api.types.is_numeric_dtype(R[c])), None)
print(f"stage2 column = {s2}   stage1 column = {s1}")
if s1 and s2:
    G = R.groupby(KEY)[[s1, s2]].mean().round(4)
    G["gap"] = (G[s2] - G[s1]).round(4)
    G.to_csv(f"{ROOT}/stage1_stage2.csv")
    print(G.to_string())

# ══════════════════════════════════════════════ 3. PREDICTIONS
hdr(3, "LOADING PER-IMAGE PREDICTIONS (full labels, seed 42)")
def read_npz(p):
    d = np.load(p, allow_pickle=True)
    g = lambda *n: next((np.asarray(d[k]).ravel() for k in n if k in d.files), None)
    return g("y_true","ytrue","labels","y","targets"), g("y_pred","ypred","pred","preds","yhat")

P = {}
for p in npzs:
    b = os.path.basename(p)
    if "full" not in b or "42" not in b or "stage1" in b.lower():
        continue
    try: yt, yp = read_npz(p)
    except Exception: continue
    if yt is None or yp is None or len(yt) != len(yp):
        continue
    ds = "eurosat" if "eurosat" in b else "resisc45"
    cfg = (b.replace(".npz","").replace("eurosat_","").replace("resisc45_","")
             .replace("_full","").replace("_42","").replace("_stage2",""))
    P[(ds, cfg)] = (yt, yp)
print(f"loaded {len(P)} prediction sets")
for k in sorted(P):
    yt, yp = P[k]; print(f"   {k[0]:9s} {k[1]:34s} n={len(yt):>6,}  acc={np.mean(yt==yp):.4f}")
if not P:
    print("!! No predictions matched the pattern. Paste these filenames back:")
    for p in npzs[:10]: print("   ", os.path.basename(p))

# ══════════════════════════════════════════════ 4. McNEMAR
hdr(4, "McNEMAR  (exact if discordant < 25, else chi-square; Holm within dataset)")
rows = []
for ds in sorted({k[0] for k in P}):
    cf = sorted(c for d, c in P if d == ds)
    for a, b in itertools.combinations(cf, 2):
        ya, pa = P[(ds,a)]; yb, pb = P[(ds,b)]
        n = min(len(ya), len(yb))
        ca, cb = pa[:n] == ya[:n], pb[:n] == yb[:n]
        n01, n10 = int((ca & ~cb).sum()), int((~ca & cb).sum()); dsc = n01 + n10
        if dsc == 0: pv, mth = 1.0, "identical"
        elif dsc < 25: pv, mth = float(stats.binomtest(n01, dsc, .5).pvalue), "exact"
        else: pv, mth = float(stats.chi2.sf((abs(n01-n10)-1)**2/dsc, 1)), "chi2"
        rows.append(dict(dataset=ds, config_a=a, config_b=b, acc_a=round(ca.mean(),4),
                         acc_b=round(cb.mean(),4), n01=n01, n10=n10, discordant=dsc,
                         method=mth, p_raw=pv))
MC = pd.DataFrame(rows)
if len(MC):
    MC["p_holm"] = np.concatenate([holm(g.p_raw.values) for _, g in MC.groupby("dataset")])
    MC[["p_raw","p_holm"]] = MC[["p_raw","p_holm"]].round(6)
    MC.to_csv(f"{ROOT}/mcnemar.csv", index=False)
    print(f"{len(MC)} pairs; {(MC.p_holm < .05).sum()} significant after Holm\n")
    print(MC.sort_values(["dataset","p_holm"]).to_string(index=False))

# ══════════════════════════════════════════════ 5. JACCARD
hdr(5, "JACCARD ERROR OVERLAP")
rows = []
for ds in sorted({k[0] for k in P}):
    cf = sorted(c for d, c in P if d == ds)
    for a, b in itertools.combinations(cf, 2):
        ya, pa = P[(ds,a)]; yb, pb = P[(ds,b)]
        n = min(len(ya), len(yb))
        ea = set(np.flatnonzero(pa[:n] != ya[:n])); eb = set(np.flatnonzero(pb[:n] != yb[:n]))
        u = len(ea | eb)
        rows.append(dict(dataset=ds, config_a=a, config_b=b, err_a=len(ea), err_b=len(eb),
                         shared=len(ea & eb), jaccard=round(len(ea & eb)/u, 4) if u else 1.0))
JA = pd.DataFrame(rows)
if len(JA):
    JA.to_csv(f"{ROOT}/error_overlap.csv", index=False)
    print(JA.sort_values(["dataset","jaccard"], ascending=[True,False]).to_string(index=False))

# ══════════════════════════════════════════════ 6. EUROSAT PER-CLASS F1
hdr(6, "EuroSAT PER-CLASS F1  (full labels, mean over seeds, both stages)")
pcf = [f for f in csvs if "per_class" in os.path.basename(f).lower()
                       or "perclass" in os.path.basename(f).lower()]
print("per-class files:", [os.path.basename(f) for f in pcf])
for f in pcf:
    d = pd.read_csv(f)
    if "dataset" not in d.columns or "k_shot" not in d.columns: continue
    e = d[(d.dataset == "eurosat") & (d.k_shot.astype(str) == "full")]
    if not len(e): continue
    fc = next(c for c in e.columns if "f1" in c.lower())
    piv = e.pivot_table(index="class_name", columns=["config_id","stage"],
                        values=fc, aggfunc="mean").round(4)
    piv.to_csv(f"{ROOT}/eurosat_per_class_f1.csv")
    print(piv.to_string())

# ══════════════════════════════════════════════ 7. CONFUSION
hdr(7, "CONFUSION — top off-diagonal pairs (best config per dataset)")
for ds in sorted({k[0] for k in P}):
    cf = sorted(c for d, c in P if d == ds)
    best = max(cf, key=lambda c: np.mean(P[(ds,c)][0] == P[(ds,c)][1]))
    yt, yp = P[(ds,best)]
    K = int(max(yt.max(), yp.max())) + 1
    C = np.zeros((K, K), int)
    for t, q in zip(yt, yp): C[t, q] += 1
    np.savetxt(f"{ROOT}/confusion_{ds}.csv", C, fmt="%d", delimiter=",")
    off = sorted(((C[i,j], i, j) for i in range(K) for j in range(K)
                  if i != j and C[i,j] > 0), reverse=True)
    print(f"\n{ds}  (best: {best})   true -> pred : count")
    for c, i, j in off[:12]: print(f"   {i:>2} -> {j:>2} : {c}")

# ══════════════════════════════════════════════ 8. COMPUTE + AUG TIER
hdr(8, "TOTAL COMPUTE AND AUGMENTATION TIER")
tcol = next((c for c in R.columns if any(t in c.lower() for t in
            ("minutes","runtime","elapsed","seconds","duration","time"))
            and pd.api.types.is_numeric_dtype(R[c])), None)
print("time column:", tcol)
if tcol:
    main = R[tcol].sum(); aug = AUG[tcol].sum() if len(AUG) else 0.0
    unit = "min" if "min" in tcol.lower() else ("s" if "sec" in tcol.lower() else "?")
    div = 60 if unit == "min" else (3600 if unit == "s" else 1)
    print(f"main grid : {main:>12,.1f} {unit}  = {main/div:>8,.2f} h")
    print(f"aug tier  : {aug:>12,.1f} {unit}  = {aug/div:>8,.2f} h")
    print(f"TOTAL     : {main+aug:>12,.1f} {unit}  = {(main+aug)/div:>8,.2f} h")
if len(AUG):
    print(f"\naugmentation runs completed: {len(AUG)} of 360")
    g = AUG.groupby(["dataset","k_shot"]).size().rename("runs")
    for (d, k), n in g.items():
        exp = 30 if d == "eurosat" else 35
        print(f"   {d:9s} k={str(k):<5} {n:>3} runs  {'COMPLETE' if n >= exp else 'partial'}")
    ok = [f"{d} k={k}" for (d,k), n in g.items()
          if n >= (30 if d == "eurosat" else 35)]
    print("\nreportable budget levels:", ", ".join(ok) if ok else "none complete")
    if s2:
        print("\naug tier mean accuracy by config and budget:")
        print(AUG.groupby(["dataset","config_id","k_shot"])[s2].agg(["mean","count"]).round(4).to_string())

print("\n" + "=" * 74)
print("Wrote to ARTIFACT_ROOT: six_metric_mean.csv, six_metric_std.csv,")
print("  stage1_stage2.csv, mcnemar.csv, error_overlap.csv,")
print("  eurosat_per_class_f1.csv, confusion_<dataset>.csv")
print("=" * 74)